In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
SPINE-GPE v7 — FASE 0: DATA, IDENTIFIABILITY & REPRODUCIBILITY LOCK
===================================================================

Objetivo
--------
Criar a fundação auditável e fail-closed da tese "A Cidade Algorítmica":

1. inventariar e versionar as fontes;
2. registrar hashes e proveniência;
3. baixar dados públicos apenas de domínios autorizados;
4. auditar os dois arquivos locais PNADc usados como base direta;
5. criar dicionário semântico, contratos, schema registry e golden tests;
6. auditar cobertura temporal/espacial;
7. registrar observado/proxy/imputado/simulado;
8. congelar DAGs, estimandos e limites de identificação;
9. impedir fallbacks silenciosos e leakage;
10. produzir um relatório de viabilidade para TODAS as fases da SPINE-GPE v7.

Execução recomendada
--------------------
Windows / Colab Local Runtime (acessa D:\\):
    python SPINE_GPEv7_FASE0.py --mode core
    python SPINE_GPEv7_FASE0.py --mode full --allow-official-ftp --download-osm

Google Colab hospedado (não acessa D:\\):
    O script monta/usa Google Drive quando disponível e grava em:
    /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7

Modos
-----
inventory : cria registry/contratos/DAGs, audita arquivos locais e descobre metadados.
core      : baixa fontes essenciais e de tamanho moderado.
full      : tenta baixar todo o acervo configurado; RAIS/CAGED exigem opt-in de FTP.

Segurança
---------
- HTTPS é obrigatório por padrão.
- O FTP oficial do MTE é texto claro; só é usado com --allow-official-ftp.
- Todos os arquivos recebem SHA-256 e registro de cabeçalhos/proveniência.
- Respostas HTML disfarçadas de dados são rejeitadas.
- Fallback de identificação direta para proxy é PROIBIDO.

Este script é a Fase 0. Ele NÃO executa ainda as estimativas finais das Fases 1–6.
"""

from __future__ import annotations

import argparse
import csv
import dataclasses
import datetime as dt
import ftplib
import hashlib
import importlib.metadata
import io
import json
import logging
import mimetypes
import os
import platform
import re
import shutil
import socket
import subprocess
import sys
import tempfile
import textwrap
import time
import traceback
import urllib.parse
import zipfile
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterable, Iterator, Mapping, Sequence

# Dependências externas são carregadas depois do bootstrap.

SCRIPT_VERSION = "7.0.0-phase0"
SCHEMA_VERSION = "spine-gpe-v7-schema-1.0.0"
DEFAULT_WINDOWS_ROOT = Path(r"D:\aCidadeAlgoritmica\SPINE-GPEv7")
DEFAULT_COLAB_ROOT = Path("/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7")
DEFAULT_COLAB_EPHEMERAL_ROOT = Path("/content/SPINE-GPEv7")

LOCAL_PNADC_PATHS = {
    "pnadc_2022q4_direct": Path(r"D:\aCidadeAlgoritmica\PNADC_042022_20250815\PNADC_042022.txt"),
    "pnadc_2024q3_direct": Path(r"D:\aCidadeAlgoritmica\PNADC_032024_20250815\PNADC_032024.txt"),
}

OFFICIAL_HTTPS_HOSTS = {
    "ftp.ibge.gov.br",
    "www.ibge.gov.br",
    "servicodados.ibge.gov.br",
    "apisidra.ibge.gov.br",
    "dados.recife.pe.gov.br",
    "portal.inmet.gov.br",
    "www.gov.br",
    "api.bcb.gov.br",
    "download.geofabrik.de",
    "overpass-api.de",
    "nominatim.openstreetmap.org",
}

CORE_YEARS = list(range(2017, dt.datetime.now().year + 1))
PNADC_REQUIRED_DIRECT = {
    "UF", "UPA", "Estrato", "V1028", "V2007", "V2009", "V2010",
    "V4010", "V4012", "V4013", "V4039", "S140093", "SD14001",
}
PNADC_REQUIRED_CORE = {"UF", "UPA", "Estrato", "V1028", "V4010", "V4012", "V4013", "V4039"}
PNAD_COVID_REQUIRED = {"UF", "V1012", "V1032", "C001", "C007", "C007C", "C009", "C010"}

# -----------------------------------------------------------------------------
# Modelos de metadados
# -----------------------------------------------------------------------------

@dataclass(slots=True)
class SourceSpec:
    source_id: str
    phases: list[str]
    agency: str
    dataset: str
    source_class: str
    measurement_status: str
    strategy: str
    discovery_url: str | None = None
    local_path: str | None = None
    package_slug: str | None = None
    years: list[int] = field(default_factory=list)
    required: bool = True
    expected_formats: list[str] = field(default_factory=list)
    spatial_scope: str = ""
    temporal_scope: str = ""
    license: str = ""
    security: str = "https"
    claim_ceiling: str = "descriptive"
    notes: str = ""


@dataclass(slots=True)
class ArtifactRecord:
    run_id: str
    source_id: str
    status: str
    phase: str
    agency: str
    dataset: str
    measurement_status: str
    source_url: str | None = None
    local_path: str | None = None
    file_name: str | None = None
    media_type: str | None = None
    bytes: int | None = None
    sha256: str | None = None
    etag: str | None = None
    last_modified: str | None = None
    downloaded_at_utc: str | None = None
    discovered_at_utc: str | None = None
    security: str | None = None
    license: str | None = None
    error: str | None = None
    notes: str | None = None


@dataclass(slots=True)
class GateResult:
    gate_id: str
    status: str  # PASS/WARN/FAIL/BLOCKED/SKIP
    severity: str
    source_id: str
    message: str
    evidence: dict[str, Any] = field(default_factory=dict)


@dataclass(slots=True)
class AuditResult:
    source_id: str
    path: str
    kind: str
    size_bytes: int
    sha256: str
    encoding: str | None = None
    delimiter: str | None = None
    sample_rows: int | None = None
    estimated_rows: int | None = None
    columns: list[str] = field(default_factory=list)
    line_length_min: int | None = None
    line_length_max: int | None = None
    line_length_mode: int | None = None
    date_min: str | None = None
    date_max: str | None = None
    spatial_bounds: list[float] | None = None
    warnings: list[str] = field(default_factory=list)


# -----------------------------------------------------------------------------
# Logging e utilidades
# -----------------------------------------------------------------------------

def utc_now() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def json_default(obj: Any) -> Any:
    if isinstance(obj, Path):
        return str(obj)
    if dataclasses.is_dataclass(obj):
        return asdict(obj)
    if isinstance(obj, set):
        return sorted(obj)
    raise TypeError(f"Tipo não serializável: {type(obj)!r}")


def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(data, ensure_ascii=False, indent=2, default=json_default), encoding="utf-8")
    tmp.replace(path)


def append_jsonl(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(data, ensure_ascii=False, default=json_default) + "\n")


def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def estimate_line_count(path: Path, sample_bytes: int = 64 * 1024 * 1024) -> int | None:
    size = path.stat().st_size
    if size == 0:
        return 0
    read_bytes = min(size, sample_bytes)
    with path.open("rb") as f:
        data = f.read(read_bytes)
    n = data.count(b"\n")
    if n == 0:
        return None
    return int(round(n * size / read_bytes))


def sanitize_filename(name: str) -> str:
    name = urllib.parse.unquote(name).strip().replace("\\", "_").replace("/", "_")
    name = re.sub(r"[^0-9A-Za-zÀ-ÿ._()\- ]+", "_", name)
    name = re.sub(r"\s+", "_", name)
    return name[:220] or "download.bin"


def setup_logger(log_path: Path) -> logging.Logger:
    logger = logging.getLogger("spine_phase0")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(sh)
    logger.addHandler(fh)
    return logger


def is_colab() -> bool:
    return "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))


def resolve_root(cli_root: str | None) -> tuple[Path, str]:
    if cli_root:
        return Path(cli_root).expanduser(), "cli"
    env_root = os.environ.get("SPINE_GPE_ROOT")
    if env_root:
        return Path(env_root).expanduser(), "env"
    if os.name == "nt":
        return DEFAULT_WINDOWS_ROOT, "windows_default"
    if is_colab():
        if DEFAULT_COLAB_ROOT.parent.exists():
            return DEFAULT_COLAB_ROOT, "colab_drive_fallback"
        return DEFAULT_COLAB_EPHEMERAL_ROOT, "colab_ephemeral_fallback"
    return Path.cwd() / "SPINE-GPEv7", "portable_fallback"


def resolve_local_pnadc_paths() -> dict[str, Path]:
    """Permite override por variáveis de ambiente no Colab hospedado."""
    out = dict(LOCAL_PNADC_PATHS)
    overrides = {
        "pnadc_2022q4_direct": os.environ.get("PNADC_2022Q4_PATH"),
        "pnadc_2024q3_direct": os.environ.get("PNADC_2024Q3_PATH"),
    }
    for key, value in overrides.items():
        if value:
            out[key] = Path(value)
    if os.name != "nt" and is_colab():
        # Fallback esperado no Drive, preservando o mesmo layout lógico.
        drive = Path("/content/drive/MyDrive/aCidadeAlgoritmica")
        candidates = {
            "pnadc_2022q4_direct": drive / "PNADC_042022_20250815/PNADC_042022.txt",
            "pnadc_2024q3_direct": drive / "PNADC_032024_20250815/PNADC_032024.txt",
        }
        for key, candidate in candidates.items():
            if not out[key].exists() and candidate.exists():
                out[key] = candidate
    return out


def create_project_tree(root: Path) -> dict[str, Path]:
    tree = {
        "root": root,
        "admin": root / "00_admin",
        "registry": root / "00_admin/registry",
        "contracts": root / "00_admin/contracts",
        "manifests": root / "00_admin/manifests",
        "logs": root / "00_admin/logs",
        "reports": root / "00_admin/reports",
        "dags": root / "00_admin/dags",
        "decisions": root / "00_admin/decisions",
        "raw": root / "01_raw",
        "raw_local_refs": root / "01_raw/00_local_references",
        "raw_ibge": root / "01_raw/10_ibge",
        "raw_mte": root / "01_raw/20_mte",
        "raw_recife": root / "01_raw/30_recife_ckan",
        "raw_inmet": root / "01_raw/40_inmet",
        "raw_anp": root / "01_raw/50_anp",
        "raw_bcb": root / "01_raw/60_bcb",
        "raw_osm": root / "01_raw/70_osm",
        "interim": root / "02_interim",
        "processed": root / "03_processed",
        "models": root / "04_models",
        "outputs": root / "05_outputs",
        "cache": root / "99_cache",
    }
    for p in tree.values():
        p.mkdir(parents=True, exist_ok=True)
    return tree


def bootstrap_packages(logger: logging.Logger) -> None:
    packages = [
        "requests>=2.31", "beautifulsoup4>=4.12", "lxml>=5.0",
        "pandas>=2.1", "pyarrow>=15", "polars>=0.20",
        "charset-normalizer>=3.3", "py7zr>=0.21", "openpyxl>=3.1",
        "pyyaml>=6.0", "tqdm>=4.66", "rich>=13.7",
        "osmnx>=1.9", "geopandas>=0.14",
    ]
    logger.info("Bootstrap de dependências Python...")
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    subprocess.run(cmd, check=True)


# -----------------------------------------------------------------------------
# Registry completo de fontes
# -----------------------------------------------------------------------------

def build_source_registry(local_paths: Mapping[str, Path]) -> list[SourceSpec]:
    current_year = dt.datetime.now().year
    specs: list[SourceSpec] = [
        SourceSpec(
            source_id="pnadc_2022q4_direct_local",
            phases=["0", "1", "2"], agency="IBGE", dataset="PNADc 2022T4 — microdado local",
            source_class="survey_microdata", measurement_status="observado_direto",
            strategy="local_reference", local_path=str(local_paths["pnadc_2022q4_direct"]),
            spatial_scope="Brasil/UF/RM conforme desenho", temporal_scope="2022T4",
            claim_ceiling="inferência survey; comparação ajustada; não causal sem hipóteses adicionais",
            notes="Base direta de plataforma; S140093 é obrigatório; fallback para proxy é proibido."
        ),
        SourceSpec(
            source_id="pnadc_2024q3_direct_local",
            phases=["0", "1", "2"], agency="IBGE", dataset="PNADc 2024T3 — microdado local",
            source_class="survey_microdata", measurement_status="observado_direto",
            strategy="local_reference", local_path=str(local_paths["pnadc_2024q3_direct"]),
            spatial_scope="Brasil/UF/RM conforme desenho", temporal_scope="2024T3",
            claim_ceiling="inferência survey; comparação ajustada; não causal sem hipóteses adicionais",
            notes="Base direta de plataforma; S140093 é obrigatório; comparação 2022–2024 sofre sazonalidade."
        ),
        SourceSpec(
            source_id="pnadc_official_2022q4_archive", phases=["0", "1", "2"], agency="IBGE",
            dataset="PNADc 2022T4 — arquivo oficial", source_class="survey_microdata",
            measurement_status="observado_direto", strategy="ibge_directory_regex",
            discovery_url="https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2022/",
            years=[2022], expected_formats=["zip"], temporal_scope="2022T4", spatial_scope="Brasil",
            claim_ceiling="proveniência e reprodução", notes=r"Regex: PNADC_042022.*\.zip"
        ),
        SourceSpec(
            source_id="pnadc_official_2024q3_archive", phases=["0", "1", "2"], agency="IBGE",
            dataset="PNADc 2024T3 — arquivo oficial", source_class="survey_microdata",
            measurement_status="observado_direto", strategy="ibge_directory_regex",
            discovery_url="https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2024/",
            years=[2024], expected_formats=["zip"], temporal_scope="2024T3", spatial_scope="Brasil",
            claim_ceiling="proveniência e reprodução", notes=r"Regex: PNADC_032024.*\.zip"
        ),
        SourceSpec(
            source_id="pnadc_documentation", phases=["0", "1", "2"], agency="IBGE",
            dataset="PNADc trimestral — documentação, inputs, dicionários e deflatores",
            source_class="official_metadata", measurement_status="observado_metadado",
            strategy="ibge_directory_all",
            discovery_url="https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/Documentacao/",
            expected_formats=["zip", "xls", "xlsx", "pdf", "txt", "sas"], spatial_scope="Brasil",
            temporal_scope="versão vigente", claim_ceiling="definição semântica oficial"
        ),
        SourceSpec(
            source_id="pnadc_regular_historical", phases=["0", "1", "2", "4"], agency="IBGE",
            dataset="PNADc trimestral regular — série histórica", source_class="survey_microdata",
            measurement_status="observado_survey_proxy_calibravel", strategy="pnadc_historical_quarters",
            discovery_url="https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/",
            years=list(range(2017, 2025)), expected_formats=["zip"], spatial_scope="Brasil/UF/RM conforme desenho",
            temporal_scope="2017T1–2024T4", claim_ceiling="reconstrução probabilística histórica; plataforma não direta fora dos módulos",
            notes="Mode=full baixa os quatro trimestres por ano; core apenas descobre o inventário remoto."
        ),
        SourceSpec(
            source_id="pnadc_platform_official_tables", phases=["0", "1"], agency="IBGE/SIDRA",
            dataset="Tabelas oficiais — trabalho por plataformas digitais 2024", source_class="official_benchmark",
            measurement_status="observado_agregado_oficial", strategy="ibge_product_sidra_links",
            discovery_url="https://www.ibge.gov.br/estatisticas/sociais/trabalho/17270-pnad-%20continua.html/17270-pnad-continua.html?edicao=44741",
            expected_formats=["html", "json"], spatial_scope="Brasil/Grandes Regiões/UF/RM conforme tabela",
            temporal_scope="2022T4 e 2024T3", claim_ceiling="golden tests e reprodução de totais oficiais"
        ),
        SourceSpec(
            source_id="pnad_covid_2020", phases=["0", "1", "2"], agency="IBGE",
            dataset="PNAD COVID19 — microdados mensais", source_class="survey_microdata",
            measurement_status="observado_ocupacional", strategy="ibge_pnad_covid",
            discovery_url="https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_PNAD_COVID19/Microdados/",
            years=[2020], expected_formats=["zip", "xls", "xlsx", "pdf"], spatial_scope="Brasil/UF",
            temporal_scope="2020-05 a 2020-11", claim_ceiling="ponte pandêmica ocupacional; não plataforma direta"
        ),
        SourceSpec(
            source_id="rais_nordeste_2017_latest", phases=["0", "1", "2", "4"], agency="MTE/PDET",
            dataset="RAIS vínculos públicos — Nordeste", source_class="administrative_microdata",
            measurement_status="observado_administrativo", strategy="mte_ftp_rais_nordeste",
            discovery_url="ftp://ftp.mtps.gov.br/pdet/microdados/RAIS/",
            years=list(range(2017, current_year)), expected_formats=["7z", "txt"], spatial_scope="Nordeste/município",
            temporal_scope=f"2017–{current_year-1}", security="official_plaintext_ftp_opt_in",
            claim_ceiling="baseline formal; registro administrativo; não representa informalidade"
        ),
        SourceSpec(
            source_id="novo_caged_2020_latest", phases=["0", "1", "2"], agency="MTE/PDET",
            dataset="Novo CAGED — movimentações", source_class="administrative_microdata",
            measurement_status="observado_administrativo", strategy="mte_ftp_novo_caged",
            discovery_url="ftp://ftp.mtps.gov.br/pdet/microdados/NOVO%20CAGED/",
            years=list(range(2020, current_year + 1)), expected_formats=["7z", "txt"], spatial_scope="Brasil/município",
            temporal_scope=f"2020–{current_year}", security="official_plaintext_ftp_opt_in",
            claim_ceiling="dinâmica mensal formal; não plataforma direta"
        ),
        SourceSpec(
            source_id="censo2022_pe_setores", phases=["0", "3A", "4", "5"], agency="IBGE",
            dataset="Censo 2022 — setores com atributos PE", source_class="census_geodata",
            measurement_status="observado_censitario", strategy="direct_https",
            discovery_url="https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/malha_com_atributos/setores/gpkg/UF/PE/PE_setores_CD2022.gpkg",
            expected_formats=["gpkg"], spatial_scope="Pernambuco/setor", temporal_scope="2022",
            claim_ceiling="estrutura territorial censitária; SAE/MRP model-based"
        ),
        SourceSpec(
            source_id="recife_ckan_bundle", phases=["0", "3A", "3B", "4", "5"], agency="Prefeitura do Recife",
            dataset="Pacote CKAN urbano Recife", source_class="municipal_open_data",
            measurement_status="observado_municipal", strategy="recife_ckan_bundle",
            discovery_url="https://dados.recife.pe.gov.br/api/3/action/", spatial_scope="Recife",
            temporal_scope="conforme recurso", license="ODbL/licença indicada no pacote",
            claim_ceiling="dinâmica urbana observada ou potencial locacional conforme conjunto"
        ),
        SourceSpec(
            source_id="inmet_recife", phases=["0", "3B", "4", "5"], agency="INMET",
            dataset="Dados meteorológicos históricos — estação Recife A301", source_class="weather_timeseries",
            measurement_status="observado_instrumental", strategy="inmet_annual_station",
            discovery_url="https://portal.inmet.gov.br/dadoshistoricos", years=CORE_YEARS,
            expected_formats=["zip", "csv"], spatial_scope="Recife/estação A301", temporal_scope=f"2017–{current_year}",
            claim_ceiling="controle meteorológico e choques observados"
        ),
        SourceSpec(
            source_id="anp_combustiveis", phases=["0", "2", "4", "5"], agency="ANP",
            dataset="Série histórica de preços de combustíveis", source_class="price_microdata",
            measurement_status="observado_mercado", strategy="anp_page_links",
            discovery_url="https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/serie-historica-de-precos-de-combustiveis",
            years=CORE_YEARS, expected_formats=["csv", "pdf"], spatial_scope="Município/posto",
            temporal_scope=f"2017–{current_year}", claim_ceiling="choques de custo e pass-through; não custo individual"
        ),
        SourceSpec(
            source_id="bcb_ipca", phases=["0", "1", "2", "4", "5"], agency="Banco Central do Brasil/IBGE",
            dataset="SGS 433 — IPCA mensal", source_class="price_index", measurement_status="observado_indice",
            strategy="bcb_api", discovery_url="https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json",
            expected_formats=["json"], spatial_scope="Brasil", temporal_scope="1980–atual",
            claim_ceiling="harmonização monetária"
        ),
        SourceSpec(
            source_id="osm_recife", phases=["0", "3A", "3B", "4", "5"], agency="OpenStreetMap",
            dataset="Rede viária multimodal Recife", source_class="volunteered_geodata",
            measurement_status="observado_cartografico", strategy="osmnx_place",
            discovery_url="https://www.openstreetmap.org/", expected_formats=["graphml", "gpkg"],
            spatial_scope="Recife", temporal_scope="snapshot da execução", license="ODbL",
            claim_ceiling="estrutura de rede; qualidade depende da cobertura OSM"
        ),
    ]
    return specs


RECIFE_CKAN_PACKAGES: dict[str, dict[str, Any]] = {
    "velocidade_2016": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2016", "required": False},
    "velocidade_2017": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2017", "required": False},
    "velocidade_2018": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media", "required": False},
    "velocidade_2019": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2019", "required": False},
    "velocidade_2020": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2020", "required": False},
    "velocidade_2021": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2021", "required": False},
    "velocidade_2022": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2022", "required": True},
    "velocidade_2023": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2023", "required": True},
    "velocidade_2024": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2024", "required": True},
    "equipamentos_transito": {"slug": "equipamentos-de-monitoramento-e-fiscalizacao-de-transito", "required": True},
    "sinistros_transito": {"slug": "acidentes-de-transito-com-e-sem-vitimas", "required": True},
    "empresas": {"slug": "empresas-da-cidade-do-recife", "required": True},
    "bares_restaurantes": {"slug": "bares-e-restaurantes", "required": True},
    "itbi": {"slug": "imposto-sobre-transmissao-de-bens-imoveis-itbi", "required": True},
    # Os itens abaixo usam busca CKAN se o slug mudar.
    "iluminacao": {"query": "iluminação pública", "required": False},
    "ciclovias": {"query": "malha cicloviária", "required": False},
    "salva_bike": {"query": "Salva Bike", "required": False},
    "mercados_publicos": {"query": "mercados públicos", "required": False},
    "parques_pracas": {"query": "parques praças", "required": False},
    "equipamentos_publicos": {"query": "equipamentos públicos", "required": False},
    "terminais": {"query": "terminais transporte", "required": False},
    "malha_viaria": {"query": "malha viária logradouros eixos viários", "required": False},
}


# -----------------------------------------------------------------------------
# Dicionário semântico, contratos, estimandos e DAGs
# -----------------------------------------------------------------------------

def semantic_dictionary() -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []

    def add(source: str, var: str, concept: str, role: str, status: str,
            allowed_use: str, forbidden_use: str = "", notes: str = "") -> None:
        rows.append({
            "schema_version": SCHEMA_VERSION, "source": source, "variable": var,
            "concept": concept, "role": role, "measurement_status": status,
            "allowed_use": allowed_use, "forbidden_use": forbidden_use, "notes": notes,
        })

    # PNADc direta e regular
    add("PNADc", "S140093", "uso de plataforma de entrega", "treatment/direct identifier",
        "observado_direto", "identificação direta no módulo especial",
        "não substituir silenciosamente por CBO/CNAE quando ausente")
    add("PNADc", "SD14001", "trabalho por plataforma em ao menos um tipo", "direct identifier",
        "observado_direto", "prevalência geral de plataforma", "não equivale especificamente a entrega")
    for v, c in [("S140091", "táxi"), ("S140092", "transporte particular de passageiros"),
                 ("S140094", "serviços gerais/profissionais")]:
        add("PNADc", v, f"uso de plataforma — {c}", "direct identifier", "observado_direto",
            "tipologia de plataforma", "não usar como entrega")
    add("PNADc", "V4010", "ocupação no trabalho principal", "covariate/occupation",
        "observado_survey", "ocupação e proxy histórica calibrada", "não é plataforma por si só")
    add("PNADc", "V4012", "posição na ocupação", "covariate/employment position",
        "observado_survey", "comparadores e formalidade", "código 4 não deve ser tratado como conta própria")
    add("PNADc", "V4013", "atividade do empreendimento", "covariate/activity",
        "observado_survey", "ocupação × atividade e proxy histórica", "não confundir CNAE classe 53202 com código domiciliar 53002")
    add("PNADc", "V4039", "horas habitualmente trabalhadas", "outcome/exposure",
        "observado_survey", "jornada", "não combinar mecanicamente com renda-hora no mesmo lado da regressão")
    add("PNADc", "VD4016", "rendimento habitual mensal do trabalho principal", "outcome",
        "observado_survey", "renda bruta principal", "não subtrair custos apenas do tratado antes da estimação")
    add("PNADc", "VD4020", "rendimento efetivo mensal de todos os trabalhos", "outcome",
        "observado_survey", "renda efetiva, após confirmação no dicionário", "não tratar como ocupação")
    add("PNADc", "V1028", "peso final", "survey design", "observado_survey",
        "estimação survey", "não usar como variável substantiva")
    add("PNADc", "UPA", "unidade primária de amostragem", "survey design", "observado_survey",
        "variância survey", "não ignorar em inferência")
    add("PNADc", "Estrato", "estrato amostral", "survey design", "observado_survey",
        "variância survey", "não ignorar em inferência")

    # PNAD COVID
    add("PNAD_COVID", "C007C", "ocupação em categorias especiais", "occupation identifier",
        "observado_ocupacional", "16=motoboy; 17=entrega de mercadorias",
        "não chamar diretamente de uso de aplicativo")
    add("PNAD_COVID", "C007", "posição na ocupação", "covariate", "observado_survey",
        "formalidade/conta própria", "não equiparar todos os demais códigos a formal")
    add("PNAD_COVID", "C001", "trabalhou na semana", "universe filter", "observado_survey",
        "sensibilidade trabalhador ativo", "não confundir com todo o universo ocupado")
    add("PNAD_COVID", "C014", "contribuição ao INSS", "social protection", "observado_survey",
        "proteção previdenciária", "não confundir com local de trabalho")

    # RAIS/CAGED
    add("RAIS", "CBO 2002", "ocupação formal", "occupation identifier", "observado_administrativo",
        "baseline formal", "não representa entregadores informais")
    add("RAIS", "Vl Remun Média Nom", "remuneração média nominal", "outcome", "observado_administrativo",
        "renda formal", "não comparar nominalmente entre anos sem deflator")
    add("RAIS", "Qtd Hora Contr", "horas contratuais", "outcome/exposure", "observado_administrativo",
        "jornada contratual", "não equivale a horas efetivas")

    # Espacial/temporal
    add("SINTAXE", "NAIN_r", "integração angular normalizada por raio", "spatial prior",
        "derivado_observado", "estrutura configuracional", "não chamar de demanda observada")
    add("SINTAXE", "NACH_r", "choice angular normalizada por raio", "spatial prior",
        "derivado_observado", "potencial de intermediação", "não chamar de fluxo real")
    add("CTTU", "fluxo_15min", "contagem veicular por equipamento e intervalo", "dynamic outcome",
        "observado_instrumental", "treino/validação do grafo dinâmico", "não equivale a fluxo de entregadores")
    add("SIM", "pedidos_potenciais", "pedidos gerados pelo emulador", "latent simulation",
        "simulado", "cenários e identificação de conjunto", "não reportar como pedidos observados")
    add("SIM", "TFD_contrafactual", "carga de custos sob política simulada", "counterfactual outcome",
        "simulado_parcialmente_identificado", "comparação de regimes", "não reportar como valor individual observado")
    return rows


def data_contracts() -> dict[str, Any]:
    return {
        "schema_version": SCHEMA_VERSION,
        "global": {
            "timezone": "America/Recife",
            "crs_geographic": "EPSG:4674",
            "crs_projected_recife": "EPSG:31985",
            "currency_nominal": "BRL",
            "currency_real_base": "IPCA, base configurável no pipeline",
            "missing_policy": "nunca converter missing em zero sem regra explícita",
            "fallback_policy": "fail_closed",
        },
        "pnadc_direct": {
            "required_variables": sorted(PNADC_REQUIRED_DIRECT),
            "treatment": "S140093",
            "survey_design": ["V1028", "UPA", "Estrato"],
            "forbidden_fallbacks": ["V4010 sozinho", "VD4019/VD4020 como ocupação", "proxy quando S140093 ausente"],
            "gates": [
                "S140093 presente no layout oficial",
                "tratados e controles positivos",
                "totais ponderados replicam tabelas oficiais dentro da tolerância",
                "amostra de outcome separada do universo identificado",
            ],
        },
        "pnad_covid": {
            "required_variables": sorted(PNAD_COVID_REQUIRED),
            "narrow_delivery": "C007C == 17",
            "broad_logistics": "C007C in {16,17}",
            "platform_direct": False,
            "claim_ceiling": "ocupação de entrega no choque pandêmico",
        },
        "rais": {
            "delimiter": ";",
            "expected_encoding_candidates": ["latin1", "cp1252", "utf-8"],
            "occupation_aliases": ["CBO Ocupação 2002", "CBO 2002 Ocupação", "CBO Ocupação 2002"],
            "income_aliases": ["Vl Remun Média Nom", "Vl Remun Média (SM)", "Vl Remun Dezembro Nom"],
            "hours_aliases": ["Qtd Hora Contr"],
            "geography_aliases": ["Mun Trab", "Município"],
            "claim_ceiling": "formalidade registrada",
        },
        "cttu_speed": {
            "time_resolution_expected_minutes": 15,
            "minimum_stable_sensors": 20,
            "minimum_continuous_months": 6,
            "maximum_missing_fraction": 0.30,
            "minimum_map_match_rate": 0.80,
            "claim_ceiling": "tráfego/velocidade observados; não demanda de plataforma",
        },
        "spatial": {
            "required_modes": ["drive", "bike", "walk"],
            "syntax_radii_m": [400, 800, 1200, 2000, 5000, "global"],
            "preserve_tags": ["bridge", "tunnel", "layer", "oneway", "access", "highway"],
            "hexagons_m": [250, 500],
            "aggregation": "length_weighted",
        },
    }


def estimand_registry() -> list[dict[str, Any]]:
    return [
        {
            "estimand_id": "E1_PLATFORM_GAP",
            "phase": "2",
            "question": "Diferença ajustada entre entregadores de plataforma e comparadores não-plataforma",
            "population": "suporte comum dentro da PNADc especial do mesmo trimestre",
            "treatment": "S140093",
            "outcomes": "renda bruta, jornada, renda-hora, informalidade, previdência",
            "identification": "seleção em observáveis + desenho survey",
            "claim_ceiling": "diferença ajustada; não efeito causal forte",
        },
        {
            "estimand_id": "E2_ICA_ASSOC",
            "phase": "2",
            "question": "Associação entre intensidade de controle algorítmico e resultados laborais",
            "population": "trabalhadores de plataforma",
            "treatment": "Índice de Controle Algorítmico",
            "outcomes": "renda, horas, informalidade, TFD parcialmente identificado",
            "identification": "associação ajustada",
            "claim_ceiling": "intensificação compatível; causalidade condicionada",
        },
        {
            "estimand_id": "E3_TFD_BOUNDS",
            "phase": "2/4",
            "question": "Limites da carga de custos necessários e não compensados",
            "population": "entregadores sob cenários de custo plausíveis",
            "treatment": "não aplicável",
            "outcomes": "[TFD_L, TFD_U] e carga sobre renda",
            "identification": "partial identification + Monte Carlo",
            "claim_ceiling": "intervalo populacional/cenário; não contabilidade individual",
        },
        {
            "estimand_id": "E4_COST_PASS_THROUGH",
            "phase": "2",
            "question": "Quanto choques de combustível são compensados na remuneração",
            "population": "trabalhadores motorizados e comparadores",
            "treatment": "choque exógeno/semiexógeno no preço do combustível",
            "outcomes": "renda nominal/real e custo esperado",
            "identification": "painel agregado/repeated cross-section com controles e placebos",
            "claim_ceiling": "pass-through incompleto; não fórmula tarifária interna",
        },
        {
            "estimand_id": "E5_SYNTAX_DYNAMIC_VALUE",
            "phase": "3B",
            "question": "Contribuição incremental da Sintaxe Espacial à previsão urbana fora da amostra",
            "population": "sensores/corredores/períodos observados",
            "treatment": "ablação de NAIN/NACH e grafo angular",
            "outcomes": "erro e calibração de fluxo/velocidade/fricção",
            "identification": "experimento computacional preditivo com holdout espaço-temporal",
            "claim_ceiling": "valor informacional da configuração; não causalidade social",
        },
        {
            "estimand_id": "E6_ALGO_SET",
            "phase": "4",
            "question": "Conjunto de regimes de despacho compatíveis com momentos observados",
            "population": "população sintética calibrada",
            "treatment": "família de políticas de despacho",
            "outcomes": "momentos laborais, territoriais e distribuição do TFD",
            "identification": "SMM/indirect inference/ABC; set identification",
            "claim_ceiling": "propriedades compatíveis; não recuperação do algoritmo real",
        },
        {
            "estimand_id": "E7_POLICY_EX_ANTE",
            "phase": "5",
            "question": "Ganho de cobertura e equidade de uma rede de suporte",
            "population": "superfícies estimadas/simuladas por cenário",
            "treatment": "localização-alocação contrafactual",
            "outcomes": "distância, cobertura, P90, desigualdade e TFD modelado",
            "identification": "otimização ex ante",
            "claim_ceiling": "impacto modelado; não efeito realizado",
        },
    ]


def dag_files() -> dict[str, str]:
    return {
        "dag_platform_gap.dot": r'''digraph G {
  rankdir=LR;
  Platform [shape=box]; Outcome [shape=box];
  Occupation; Activity; Position; Region; Period; Age; Sex; Race; Education; Assets;
  Occupation -> Platform; Activity -> Platform; Position -> Platform; Region -> Platform;
  Age -> Platform; Sex -> Platform; Race -> Platform; Education -> Platform; Assets -> Platform;
  Occupation -> Outcome; Activity -> Outcome; Position -> Outcome; Region -> Outcome; Period -> Outcome;
  Age -> Outcome; Sex -> Outcome; Race -> Outcome; Education -> Outcome; Assets -> Outcome;
  Platform -> Outcome;
}''',
        "dag_tfd_mechanism.dot": r'''digraph G {
  rankdir=LR;
  StructuralInequality -> ResidentialLocation;
  StructuralInequality -> LaborPosition;
  ResidentialLocation -> SpatialAccess;
  UrbanConfiguration -> SpatialAccess;
  PlatformRegime -> AlgorithmicControl;
  PlatformRegime -> Dispatch;
  SpatialAccess -> Dispatch;
  Dispatch -> UnpaidTime;
  Dispatch -> EmptyDistance;
  Dispatch -> Risk;
  FuelPrice -> MonetaryCost;
  EmptyDistance -> MonetaryCost;
  UnpaidTime -> TFD;
  MonetaryCost -> TFD;
  Risk -> TFD;
  Compensation -> TFD [label="reduz"];
  AlgorithmicControl -> Dispatch;
}''',
        "dag_policy.dot": r'''digraph G {
  rankdir=LR;
  EstimatedExposure -> CandidateWeights;
  StructuralFriction -> CandidateWeights;
  Vulnerability -> EquityConstraint;
  Budget -> FacilityChoice;
  CandidateWeights -> FacilityChoice;
  EquityConstraint -> FacilityChoice;
  FacilityChoice -> AccessDistance;
  AccessDistance -> ModeledTFD;
  UrbanDynamics -> AccessDistance;
}''',
    }


# -----------------------------------------------------------------------------
# Cliente HTTP seguro e download versionado
# -----------------------------------------------------------------------------

class SafeHTTP:
    def __init__(self, logger: logging.Logger, timeout: int = 90, retries: int = 4):
        import requests
        from requests.adapters import HTTPAdapter
        from urllib3.util.retry import Retry

        self.logger = logger
        self.timeout = timeout
        self.session = requests.Session()
        retry = Retry(
            total=retries, connect=retries, read=retries,
            backoff_factor=1.5,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=frozenset(["HEAD", "GET"]),
            respect_retry_after_header=True,
        )
        self.session.mount("https://", HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=10))
        self.session.headers.update({
            "User-Agent": f"SPINE-GPEv7-Phase0/{SCRIPT_VERSION} academic-research",
            "Accept-Encoding": "gzip, deflate",
        })

    @staticmethod
    def validate_url(url: str) -> None:
        parsed = urllib.parse.urlparse(url)
        if parsed.scheme != "https":
            raise ValueError(f"Somente HTTPS é permitido: {url}")
        if parsed.hostname not in OFFICIAL_HTTPS_HOSTS:
            raise ValueError(f"Host fora da allowlist: {parsed.hostname}")

    def get(self, url: str, **kwargs: Any):
        self.validate_url(url)
        return self.session.get(url, timeout=kwargs.pop("timeout", self.timeout), **kwargs)

    def head(self, url: str, **kwargs: Any):
        self.validate_url(url)
        return self.session.head(url, timeout=kwargs.pop("timeout", self.timeout), allow_redirects=True, **kwargs)

    def download(self, url: str, destination: Path, source: SourceSpec,
                 run_id: str, phase: str = "0", overwrite: bool = False,
                 max_bytes: int | None = None) -> ArtifactRecord:
        self.validate_url(url)
        destination.parent.mkdir(parents=True, exist_ok=True)
        discovered = utc_now()
        try:
            with self.get(url, stream=True) as r:
                r.raise_for_status()
                content_type = (r.headers.get("Content-Type") or "").lower()
                content_length = int(r.headers.get("Content-Length") or 0)
                if max_bytes and content_length and content_length > max_bytes:
                    return ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="SKIPPED_SIZE_LIMIT",
                        phase=phase, agency=source.agency, dataset=source.dataset,
                        measurement_status=source.measurement_status, source_url=url,
                        file_name=destination.name, bytes=content_length, security="https",
                        license=source.license, discovered_at_utc=discovered,
                        notes=f"Limite: {max_bytes} bytes"
                    )
                if "text/html" in content_type and destination.suffix.lower() not in {".html", ".htm"}:
                    raise ValueError(f"Resposta HTML inesperada para {destination.name}")
                if destination.exists() and not overwrite:
                    return ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="EXISTS",
                        phase=phase, agency=source.agency, dataset=source.dataset,
                        measurement_status=source.measurement_status, source_url=url,
                        local_path=str(destination), file_name=destination.name,
                        bytes=destination.stat().st_size, sha256=sha256_file(destination),
                        etag=r.headers.get("ETag"), last_modified=r.headers.get("Last-Modified"),
                        security="https", license=source.license, discovered_at_utc=discovered,
                    )
                tmp = destination.with_suffix(destination.suffix + ".part")
                h = hashlib.sha256()
                total = 0
                with tmp.open("wb") as f:
                    for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                        if not chunk:
                            continue
                        total += len(chunk)
                        if max_bytes and total > max_bytes:
                            f.close()
                            tmp.unlink(missing_ok=True)
                            return ArtifactRecord(
                                run_id=run_id, source_id=source.source_id, status="SKIPPED_SIZE_LIMIT",
                                phase=phase, agency=source.agency, dataset=source.dataset,
                                measurement_status=source.measurement_status, source_url=url,
                                file_name=destination.name, bytes=total, security="https",
                                license=source.license, discovered_at_utc=discovered,
                            )
                        h.update(chunk)
                        f.write(chunk)
                digest = h.hexdigest()
                if tmp.stat().st_size < 32:
                    tmp.unlink(missing_ok=True)
                    raise ValueError("Arquivo baixado é pequeno demais; possível resposta de erro")
                tmp.replace(destination)
                return ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DOWNLOADED",
                    phase=phase, agency=source.agency, dataset=source.dataset,
                    measurement_status=source.measurement_status, source_url=url,
                    local_path=str(destination), file_name=destination.name,
                    media_type=content_type, bytes=total, sha256=digest,
                    etag=r.headers.get("ETag"), last_modified=r.headers.get("Last-Modified"),
                    downloaded_at_utc=utc_now(), discovered_at_utc=discovered,
                    security="https", license=source.license,
                )
        except Exception as exc:
            self.logger.error("Falha no download %s: %s", url, exc)
            return ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="ERROR",
                phase=phase, agency=source.agency, dataset=source.dataset,
                measurement_status=source.measurement_status, source_url=url,
                local_path=str(destination), file_name=destination.name,
                security="https", license=source.license, discovered_at_utc=discovered,
                error=f"{type(exc).__name__}: {exc}",
            )


# -----------------------------------------------------------------------------
# Descoberta HTTP/CKAN
# -----------------------------------------------------------------------------

def html_links(http: SafeHTTP, url: str) -> list[tuple[str, str]]:
    from bs4 import BeautifulSoup

    r = http.get(url)
    r.raise_for_status()
    soup = BeautifulSoup(r.content, "lxml")
    out: list[tuple[str, str]] = []
    for a in soup.find_all("a", href=True):
        href = urllib.parse.urljoin(url, a.get("href"))
        text = " ".join(a.get_text(" ", strip=True).split())
        if urllib.parse.urlparse(href).scheme == "https":
            out.append((text, href))
    return out


def select_links(links: Sequence[tuple[str, str]], pattern: str,
                 extensions: Sequence[str] | None = None) -> list[tuple[str, str]]:
    rx = re.compile(pattern, flags=re.I)
    out = []
    for text, href in links:
        blob = f"{text} {href}"
        path = urllib.parse.urlparse(href).path.lower()
        if rx.search(blob) and (not extensions or any(path.endswith("." + e.lower()) or f".{e.lower()}/" in path for e in extensions)):
            out.append((text, href))
    return out


def ckan_action(http: SafeHTTP, action: str, params: Mapping[str, Any]) -> dict[str, Any]:
    url = f"https://dados.recife.pe.gov.br/api/3/action/{action}"
    r = http.get(url, params=params)
    r.raise_for_status()
    data = r.json()
    if not data.get("success"):
        raise RuntimeError(f"CKAN retornou success=false em {action}")
    return data["result"]


def resolve_ckan_package(http: SafeHTTP, item: Mapping[str, Any]) -> dict[str, Any] | None:
    if item.get("slug"):
        try:
            return ckan_action(http, "package_show", {"id": item["slug"]})
        except Exception:
            pass
    query = item.get("query") or item.get("slug")
    if not query:
        return None
    result = ckan_action(http, "package_search", {"q": query, "rows": 5})
    candidates = result.get("results", [])
    return candidates[0] if candidates else None


def choose_ckan_resources(package: Mapping[str, Any], mode: str) -> list[dict[str, Any]]:
    allowed = {"CSV", "JSON", "GEOJSON", "ZIP", "SHP", "GPKG", "XLS", "XLSX", "PDF", "KML"}
    resources = []
    for res in package.get("resources", []):
        fmt = str(res.get("format") or "").upper().strip()
        url = str(res.get("url") or "")
        name = str(res.get("name") or "")
        if fmt in allowed and url.startswith("https://"):
            if mode == "core":
                # No core: dicionários, geometrias, recursos recentes e anos 2022–2024.
                if any(token in name.lower() for token in ["dicion", "2022", "2023", "2024", "equipamento", "ativa", "bares"]):
                    resources.append(dict(res))
                elif len(package.get("resources", [])) <= 5:
                    resources.append(dict(res))
            else:
                resources.append(dict(res))
    # Deduplicação por URL.
    seen = set()
    dedup = []
    for r in resources:
        if r["url"] not in seen:
            seen.add(r["url"])
            dedup.append(r)
    return dedup


# -----------------------------------------------------------------------------
# FTP oficial MTE — somente opt-in
# -----------------------------------------------------------------------------

def ftp_connect() -> ftplib.FTP:
    ftp = ftplib.FTP("ftp.mtps.gov.br", timeout=90)
    ftp.login()
    ftp.set_pasv(True)
    return ftp


def ftp_nlst_safe(ftp: ftplib.FTP, path: str) -> list[str]:
    try:
        return ftp.nlst(path)
    except ftplib.error_perm:
        return []


def ftp_download_file(ftp: ftplib.FTP, remote_path: str, destination: Path,
                      source: SourceSpec, run_id: str, logger: logging.Logger) -> ArtifactRecord:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        return ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="EXISTS", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=f"ftp://ftp.mtps.gov.br{remote_path}", local_path=str(destination),
            file_name=destination.name, bytes=destination.stat().st_size,
            sha256=sha256_file(destination), security="official_plaintext_ftp_opt_in",
            discovered_at_utc=utc_now(), notes="Transporte sem criptografia; integridade local por SHA-256"
        )
    tmp = destination.with_suffix(destination.suffix + ".part")
    h = hashlib.sha256()
    total = 0
    try:
        with tmp.open("wb") as f:
            def callback(chunk: bytes) -> None:
                nonlocal total
                total += len(chunk)
                h.update(chunk)
                f.write(chunk)
            ftp.retrbinary(f"RETR {remote_path}", callback, blocksize=8 * 1024 * 1024)
        tmp.replace(destination)
        return ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="DOWNLOADED", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=f"ftp://ftp.mtps.gov.br{remote_path}", local_path=str(destination),
            file_name=destination.name, bytes=total, sha256=h.hexdigest(),
            security="official_plaintext_ftp_opt_in", downloaded_at_utc=utc_now(),
            discovered_at_utc=utc_now(), notes="FTP oficial MTE sem criptografia; SHA-256 calculado após download"
        )
    except Exception as exc:
        tmp.unlink(missing_ok=True)
        logger.error("FTP falhou em %s: %s", remote_path, exc)
        return ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=f"ftp://ftp.mtps.gov.br{remote_path}", local_path=str(destination),
            file_name=destination.name, security="official_plaintext_ftp_opt_in",
            error=f"{type(exc).__name__}: {exc}", discovered_at_utc=utc_now(),
        )


def download_mte_rais(source: SourceSpec, dest: Path, run_id: str,
                      logger: logging.Logger, years: Sequence[int]) -> list[ArtifactRecord]:
    records: list[ArtifactRecord] = []
    ftp = ftp_connect()
    try:
        for year in years:
            base = f"/pdet/microdados/RAIS/{year}"
            names = ftp_nlst_safe(ftp, base)
            matches = [n for n in names if re.search(r"NORDESTE.*\.(7z|zip)$", n, re.I)]
            if not matches:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                    agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                    source_url=f"ftp://ftp.mtps.gov.br{base}/", security=source.security,
                    discovered_at_utc=utc_now(), notes="Nenhum arquivo Nordeste encontrado"
                ))
                continue
            for remote in matches:
                name = sanitize_filename(Path(remote).name)
                records.append(ftp_download_file(ftp, remote, dest / str(year) / name, source, run_id, logger))
    finally:
        try:
            ftp.quit()
        except Exception:
            ftp.close()
    return records


def download_mte_novo_caged(source: SourceSpec, dest: Path, run_id: str,
                            logger: logging.Logger, years: Sequence[int]) -> list[ArtifactRecord]:
    """Baixa somente arquivos CAGEDMOV, evitando FOR/EXC salvo mudança explícita."""
    records: list[ArtifactRecord] = []
    ftp = ftp_connect()
    try:
        root_candidates = ["/pdet/microdados/NOVO CAGED", "/pdet/microdados/NOVO%20CAGED"]
        root = None
        for candidate in root_candidates:
            if ftp_nlst_safe(ftp, candidate):
                root = candidate
                break
        if root is None:
            return [ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security=source.security,
                discovered_at_utc=utc_now(), error="Diretório NOVO CAGED não encontrado"
            )]
        for year in years:
            year_dir = f"{root}/{year}"
            month_dirs = ftp_nlst_safe(ftp, year_dir)
            for month_dir in month_dirs:
                names = ftp_nlst_safe(ftp, month_dir)
                matches = [n for n in names if re.search(r"CAGEDMOV.*\.(7z|zip)$", n, re.I)]
                for remote in matches:
                    month = re.search(r"20\d{4}", remote)
                    sub = month.group(0) if month else str(year)
                    records.append(ftp_download_file(
                        ftp, remote, dest / str(year) / sub / sanitize_filename(Path(remote).name),
                        source, run_id, logger
                    ))
    finally:
        try:
            ftp.quit()
        except Exception:
            ftp.close()
    return records


# -----------------------------------------------------------------------------
# Extração, layout PNAD e auditoria de arquivos
# -----------------------------------------------------------------------------

def extract_archive(path: Path, destination: Path, logger: logging.Logger,
                    member_pattern: str | None = None) -> list[Path]:
    destination.mkdir(parents=True, exist_ok=True)
    extracted: list[Path] = []
    rx = re.compile(member_pattern, re.I) if member_pattern else None
    suffix = path.suffix.lower()
    try:
        if suffix == ".zip":
            with zipfile.ZipFile(path) as zf:
                for info in zf.infolist():
                    if info.is_dir():
                        continue
                    if rx and not rx.search(info.filename):
                        continue
                    target = destination / sanitize_filename(Path(info.filename).name)
                    with zf.open(info) as src, target.open("wb") as dst:
                        shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
                    extracted.append(target)
        elif suffix == ".7z":
            import py7zr
            with py7zr.SevenZipFile(path, mode="r") as archive:
                names = archive.getnames()
                selected = [n for n in names if not rx or rx.search(n)]
                archive.extract(path=destination, targets=selected or None)
            extracted = [p for p in destination.rglob("*") if p.is_file()]
        else:
            logger.warning("Formato de arquivo não extraível automaticamente: %s", path)
    except Exception as exc:
        logger.error("Falha ao extrair %s: %s", path, exc)
    return extracted


def detect_encoding(path: Path, sample_bytes: int = 2_000_000) -> str:
    from charset_normalizer import from_bytes
    with path.open("rb") as f:
        raw = f.read(sample_bytes)
    match = from_bytes(raw).best()
    return match.encoding if match and match.encoding else "utf-8"


def audit_fixed_width(path: Path, source_id: str, max_lines: int = 20_000) -> AuditResult:
    lengths: Counter[int] = Counter()
    sample = 0
    with path.open("rb") as f:
        for line in f:
            lengths[len(line.rstrip(b"\r\n"))] += 1
            sample += 1
            if sample >= max_lines:
                break
    mode = lengths.most_common(1)[0][0] if lengths else None
    result = AuditResult(
        source_id=source_id, path=str(path), kind="fixed_width_text",
        size_bytes=path.stat().st_size, sha256=sha256_file(path),
        sample_rows=sample, estimated_rows=estimate_line_count(path),
        line_length_min=min(lengths) if lengths else None,
        line_length_max=max(lengths) if lengths else None,
        line_length_mode=mode,
    )
    if len(lengths) > 5:
        result.warnings.append(f"Muitos comprimentos de linha no sample: {dict(lengths.most_common(10))}")
    return result


def sniff_delimiter(sample: str) -> str | None:
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=";,\t|")
        return dialect.delimiter
    except csv.Error:
        return None


def audit_delimited(path: Path, source_id: str, sample_rows: int = 50_000) -> AuditResult:
    import pandas as pd

    encoding = detect_encoding(path)
    with path.open("r", encoding=encoding, errors="replace") as f:
        sample_text = f.read(100_000)
    delim = sniff_delimiter(sample_text) or ";"
    warnings: list[str] = []
    try:
        df = pd.read_csv(path, sep=delim, encoding=encoding, nrows=sample_rows, low_memory=False)
        columns = [str(c) for c in df.columns]
    except Exception as exc:
        columns = []
        warnings.append(f"Falha ao ler amostra: {type(exc).__name__}: {exc}")
    return AuditResult(
        source_id=source_id, path=str(path), kind="delimited_text",
        size_bytes=path.stat().st_size, sha256=sha256_file(path), encoding=encoding,
        delimiter=delim, sample_rows=sample_rows, estimated_rows=estimate_line_count(path),
        columns=columns, warnings=warnings,
    )


def audit_generic(path: Path, source_id: str) -> AuditResult:
    ext = path.suffix.lower()
    if ext == ".txt":
        # PNADc é fixed-width; demais TXT tendem a ser delimitados.
        if "PNADC_" in path.name.upper() or "PNAD_COVID" in path.name.upper():
            return audit_fixed_width(path, source_id)
        return audit_delimited(path, source_id)
    if ext in {".csv", ".tsv"}:
        return audit_delimited(path, source_id)
    return AuditResult(
        source_id=source_id, path=str(path), kind=ext.lstrip(".") or "binary",
        size_bytes=path.stat().st_size, sha256=sha256_file(path),
    )


def parse_sas_input_layout(text: str) -> list[dict[str, Any]]:
    """Extrai layout de instruções SAS do tipo @posição variável $largura. ou largura."""
    rows = []
    rx = re.compile(
        r"@(?P<start>\d+)\s+(?P<name>[A-Za-z_][A-Za-z0-9_]*)\s+(?P<char>\$)?(?P<width>\d+)(?:\.\d+)?",
        flags=re.I,
    )
    for line in text.splitlines():
        m = rx.search(line)
        if not m:
            continue
        start = int(m.group("start"))
        width = int(m.group("width"))
        rows.append({
            "variable": m.group("name"), "start_1based": start,
            "end_1based": start + width - 1, "width": width,
            "type": "string" if m.group("char") else "numeric",
        })
    return rows


def discover_layout_files(root: Path) -> list[Path]:
    patterns = ["*.sas", "*input*.txt", "*INPUT*.txt", "*layout*.txt", "*dicion*.xls*", "*Dicion*.xls*"]
    out: list[Path] = []
    for pattern in patterns:
        out.extend(root.rglob(pattern))
    return sorted(set(out))


def extract_variable_names_from_docs(paths: Sequence[Path]) -> set[str]:
    names: set[str] = set()
    for path in paths:
        try:
            if path.suffix.lower() in {".txt", ".sas"}:
                enc = detect_encoding(path)
                text = path.read_text(encoding=enc, errors="replace")
                for row in parse_sas_input_layout(text):
                    names.add(row["variable"])
                names.update(re.findall(r"\b(?:S14\d{4}|SD14\d{3}|V\d{4,5}|VD\d{4}|C\d{3}[A-Z]?)\b", text))
            elif path.suffix.lower() in {".xls", ".xlsx"}:
                import pandas as pd
                book = pd.ExcelFile(path)
                for sheet in book.sheet_names:
                    df = pd.read_excel(path, sheet_name=sheet, header=None, nrows=5000)
                    for value in df.astype(str).to_numpy().ravel():
                        names.update(re.findall(r"\b(?:S14\d{4}|SD14\d{3}|V\d{4,5}|VD\d{4}|C\d{3}[A-Z]?)\b", value))
        except Exception:
            continue
    return names


# -----------------------------------------------------------------------------
# Golden tests e fail-closed gates
# -----------------------------------------------------------------------------

def run_local_pnadc_gates(local_paths: Mapping[str, Path], audits: list[AuditResult]) -> list[GateResult]:
    gates: list[GateResult] = []
    for source_id, path in local_paths.items():
        if not path.exists():
            gates.append(GateResult(
                gate_id=f"{source_id}.exists", status="FAIL", severity="critical",
                source_id=source_id,
                message="Arquivo local PNADc obrigatório não encontrado.",
                evidence={"expected_path": str(path)},
            ))
            continue
        size = path.stat().st_size
        gates.append(GateResult(
            gate_id=f"{source_id}.exists", status="PASS", severity="critical",
            source_id=source_id, message="Arquivo local encontrado.",
            evidence={"path": str(path), "bytes": size},
        ))
        gates.append(GateResult(
            gate_id=f"{source_id}.size", status="PASS" if size > 10_000_000 else "FAIL",
            severity="critical", source_id=source_id,
            message="Tamanho plausível para microdado PNADc." if size > 10_000_000 else "Arquivo pequeno demais.",
            evidence={"bytes": size},
        ))
        audit = next((a for a in audits if a.source_id == source_id), None)
        if audit:
            stable = audit.line_length_mode is not None and audit.line_length_min == audit.line_length_max
            gates.append(GateResult(
                gate_id=f"{source_id}.fixed_width_stability",
                status="PASS" if stable else "WARN", severity="high", source_id=source_id,
                message="Comprimento fixed-width estável na amostra." if stable else "Variação de comprimento detectada; investigar CR/LF ou corrupção.",
                evidence={
                    "min": audit.line_length_min, "max": audit.line_length_max,
                    "mode": audit.line_length_mode, "sample_rows": audit.sample_rows,
                },
            ))
    return gates


def run_semantic_gates(variable_names: set[str], docs_found: bool) -> list[GateResult]:
    gates: list[GateResult] = []
    if not docs_found:
        gates.append(GateResult(
            gate_id="pnadc.docs.available", status="BLOCKED", severity="critical",
            source_id="pnadc_documentation",
            message="Documentação/layout oficial ainda não foi materializado; modelos ficam bloqueados.",
        ))
        return gates
    missing_direct = sorted(PNADC_REQUIRED_DIRECT - variable_names)
    missing_core = sorted(PNADC_REQUIRED_CORE - variable_names)
    gates.append(GateResult(
        gate_id="pnadc.layout.core_variables",
        status="PASS" if not missing_core else "FAIL", severity="critical",
        source_id="pnadc_documentation",
        message="Variáveis centrais presentes no layout." if not missing_core else "Variáveis centrais ausentes no layout detectado.",
        evidence={"missing": missing_core, "detected_count": len(variable_names)},
    ))
    gates.append(GateResult(
        gate_id="pnadc.layout.direct_platform",
        status="PASS" if "S140093" in variable_names else "FAIL", severity="critical",
        source_id="pnadc_documentation",
        message="S140093 validada no layout oficial." if "S140093" in variable_names else "S140093 não encontrada: PROIBIDO usar proxy como substituição silenciosa.",
        evidence={"missing_direct": missing_direct},
    ))
    gates.append(GateResult(
        gate_id="pnadc.no_silent_proxy_fallback", status="PASS", severity="critical",
        source_id="pnadc_direct",
        message="Política fail-closed registrada: ausência de S140093 bloqueia comparações diretas.",
    ))
    return gates


def global_fail_closed_gates() -> list[GateResult]:
    return [
        GateResult("global.no_synthetic_geography_as_observed", "PASS", "critical", "global",
                   "Geografia sintética nunca será tratada como localização observada."),
        GateResult("global.no_treatment_specific_outcome", "PASS", "critical", "global",
                   "É proibido subtrair custos somente do grupo tratado antes da comparação."),
        GateResult("global.no_random_labels_for_stgnn", "PASS", "critical", "global",
                   "STGNN só pode usar targets temporais observados; séries pseudoaleatórias são bloqueadas."),
        GateResult("global.no_causal_without_overlap", "PASS", "critical", "global",
                   "Estimadores causais/duplamente robustos exigem tratamento direto, controles e suporte."),
        GateResult("global.observed_estimated_simulated_separation", "PASS", "critical", "global",
                   "Todo artefato deve registrar observado/proxy/imputado/simulado."),
        GateResult("global.outcome_universe_separation", "PASS", "high", "global",
                   "Universo identificado e amostra válida de outcome serão preservados separadamente."),
    ]


# -----------------------------------------------------------------------------
# Downloaders por fonte
# -----------------------------------------------------------------------------

def download_ibge_directory(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                            mode: str, logger: logging.Logger) -> list[ArtifactRecord]:
    records: list[ArtifactRecord] = []
    assert source.discovery_url
    try:
        links = html_links(http, source.discovery_url)
        if source.source_id == "pnadc_official_2022q4_archive":
            chosen = select_links(links, r"PNADC_042022.*\.zip", ["zip"])
        elif source.source_id == "pnadc_official_2024q3_archive":
            chosen = select_links(links, r"PNADC_032024.*\.zip", ["zip"])
        else:
            chosen = [x for x in links if re.search(r"\.(zip|xlsx?|pdf|txt|sas)$", urllib.parse.urlparse(x[1]).path, re.I)]
        if mode == "inventory":
            for text, url in chosen:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                    agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                    source_url=url, file_name=sanitize_filename(Path(urllib.parse.urlparse(url).path).name or text),
                    security="https", license=source.license, discovered_at_utc=utc_now(),
                ))
            return records
        # Arquivos PNADc brutos são enormes e já existem localmente: core baixa docs, full baixa arquivos oficiais.
        if source.source_id.startswith("pnadc_official_") and mode != "full":
            for text, url in chosen:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DISCOVERED_NOT_DOWNLOADED_LOCAL_BASE_EXISTS",
                    phase="0", agency=source.agency, dataset=source.dataset,
                    measurement_status=source.measurement_status, source_url=url,
                    file_name=sanitize_filename(Path(urllib.parse.urlparse(url).path).name or text),
                    security="https", discovered_at_utc=utc_now(),
                    notes="Microdado local é a base; archive oficial será baixado apenas em mode=full."
                ))
            return records
        for text, url in chosen:
            name = sanitize_filename(Path(urllib.parse.urlparse(url).path).name or text)
            records.append(http.download(url, dest / name, source, run_id))
        if not chosen:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                error="Nenhum link compatível descoberto"
            ))
    except Exception as exc:
        logger.error("Descoberta IBGE falhou para %s: %s", source.source_id, exc)
        records.append(ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
            error=f"{type(exc).__name__}: {exc}"
        ))
    return records


def download_pnadc_historical(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                              mode: str, logger: logging.Logger) -> list[ArtifactRecord]:
    """Descobre/baixa os quatro trimestres PNADc por ano sem hardcode de sufixo de revisão."""
    records: list[ArtifactRecord] = []
    for year in source.years:
        year_url = urllib.parse.urljoin(source.discovery_url.rstrip("/") + "/", f"{year}/")
        try:
            links = html_links(http, year_url)
            chosen = select_links(links, rf"PNADC_0[1-4]{year}.*\.zip", ["zip"])
            for text, url in chosen:
                name = sanitize_filename(Path(urllib.parse.urlparse(url).path).name or text)
                if mode == "inventory" or mode == "core":
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                        agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                        source_url=url, file_name=name, security="https", discovered_at_utc=utc_now(),
                        notes="Download efetivo reservado a mode=full para controlar armazenamento."
                    ))
                else:
                    records.append(http.download(url, dest / str(year) / name, source, run_id))
            if not chosen:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                    agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                    source_url=year_url, security="https", discovered_at_utc=utc_now(),
                    notes=f"Nenhum trimestre localizado para {year}"
                ))
        except Exception as exc:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=year_url, security="https", discovered_at_utc=utc_now(),
                error=f"{type(exc).__name__}: {exc}"
            ))
    return records


def discover_ibge_sidra_tables(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                               mode: str, logger: logging.Logger) -> tuple[list[ArtifactRecord], dict[str, Any]]:
    """Registra IDs de tabelas SIDRA linkadas pela página oficial sem fazer consulta irrestrita gigante."""
    records: list[ArtifactRecord] = []
    meta: dict[str, Any] = {"source_page": source.discovery_url, "discovered_at_utc": utc_now(), "tables": []}
    try:
        r = http.get(source.discovery_url)
        r.raise_for_status()
        html = r.text
        ids = sorted(set(re.findall(r"sidra\.ibge\.gov\.br/(?:tabela|Tabela)/(\d+)", html, flags=re.I)))
        # Alguns links são redirecionados/embutidos com apenas /tabela/ID.
        ids += [x for x in sorted(set(re.findall(r"(?:/tabela/|Tabela/)(\d+)", html, flags=re.I))) if x not in ids]
        meta["tables"] = [{"table_id": x, "url": f"https://sidra.ibge.gov.br/tabela/{x}"} for x in ids]
        dest.mkdir(parents=True, exist_ok=True)
        html_path = dest / "pnadc_platform_2024_official_page.html"
        if mode != "inventory":
            html_path.write_text(html, encoding="utf-8")
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="DOWNLOADED", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, local_path=str(html_path), file_name=html_path.name,
                bytes=html_path.stat().st_size, sha256=sha256_file(html_path), security="https",
                downloaded_at_utc=utc_now(), discovered_at_utc=utc_now(),
                notes=f"IDs SIDRA descobertos: {len(ids)}"
            ))
        else:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                notes=f"IDs SIDRA descobertos: {len(ids)}"
            ))
        if not ids:
            logger.warning("Nenhum ID SIDRA foi extraído da página oficial; a página pode carregar links por JavaScript.")
    except Exception as exc:
        records.append(ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
            error=f"{type(exc).__name__}: {exc}"
        ))
    return records, meta


def download_pnad_covid(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                        mode: str, logger: logging.Logger) -> list[ArtifactRecord]:
    records: list[ArtifactRecord] = []
    base = source.discovery_url.rstrip("/") + "/"
    dirs = {
        "dados": urllib.parse.urljoin(base, "Dados/"),
        "docs": urllib.parse.urljoin(base, "Documentacao/"),
    }
    for kind, url in dirs.items():
        try:
            links = html_links(http, url)
            if kind == "dados":
                pattern = r"PNAD_COVID_(092020|052020|062020|072020|082020|102020|112020)\.zip"
                chosen = select_links(links, pattern, ["zip"])
                if mode == "core":
                    chosen = [x for x in chosen if "092020" in x[1]]
            else:
                chosen = [x for x in links if re.search(r"\.(zip|xlsx?|pdf|txt)$", urllib.parse.urlparse(x[1]).path, re.I)]
            for text, href in chosen:
                name = sanitize_filename(Path(urllib.parse.urlparse(href).path).name or text)
                if mode == "inventory":
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                        agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                        source_url=href, file_name=name, security="https", discovered_at_utc=utc_now()
                    ))
                else:
                    records.append(http.download(href, dest / kind / name, source, run_id))
        except Exception as exc:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=url, security="https", discovered_at_utc=utc_now(), error=f"{type(exc).__name__}: {exc}"
            ))
    return records


def download_recife_ckan(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                         mode: str, logger: logging.Logger, core_max_mb: int) -> tuple[list[ArtifactRecord], list[dict[str, Any]]]:
    records: list[ArtifactRecord] = []
    packages_meta: list[dict[str, Any]] = []
    for alias, item in RECIFE_CKAN_PACKAGES.items():
        try:
            pkg = resolve_ckan_package(http, item)
            if not pkg:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=f"recife_{alias}", status="NOT_FOUND", phase="0",
                    agency=source.agency, dataset=alias, measurement_status=source.measurement_status,
                    source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                    notes=f"required={item.get('required', False)}"
                ))
                continue
            packages_meta.append({
                "alias": alias, "id": pkg.get("id"), "name": pkg.get("name"), "title": pkg.get("title"),
                "metadata_modified": pkg.get("metadata_modified"), "license_title": pkg.get("license_title"),
                "num_resources": len(pkg.get("resources", [])), "url": pkg.get("url"),
            })
            resources = choose_ckan_resources(pkg, mode)
            if mode == "inventory":
                resources = [dict(r) for r in pkg.get("resources", [])]
            for res in resources:
                url = str(res.get("url") or "")
                if not url.startswith("https://"):
                    continue
                raw_name = res.get("name") or Path(urllib.parse.urlparse(url).path).name or res.get("id")
                ext = str(res.get("format") or "bin").lower()
                name = sanitize_filename(str(raw_name))
                if "." not in Path(name).name:
                    name += f".{ext}"
                subdir = dest / alias
                if mode == "inventory":
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=f"recife_{alias}", status="DISCOVERED", phase="0",
                        agency=source.agency, dataset=str(pkg.get("title") or alias),
                        measurement_status=source.measurement_status, source_url=url,
                        file_name=name, media_type=str(res.get("mimetype") or ext), security="https",
                        license=str(pkg.get("license_title") or source.license), discovered_at_utc=utc_now(),
                    ))
                else:
                    max_bytes = core_max_mb * 1024 * 1024 if mode == "core" else None
                    spec = dataclasses.replace(
                        source, source_id=f"recife_{alias}", dataset=str(pkg.get("title") or alias),
                        license=str(pkg.get("license_title") or source.license)
                    )
                    records.append(http.download(url, subdir / name, spec, run_id, max_bytes=max_bytes))
        except Exception as exc:
            logger.error("CKAN %s falhou: %s", alias, exc)
            records.append(ArtifactRecord(
                run_id=run_id, source_id=f"recife_{alias}", status="ERROR", phase="0",
                agency=source.agency, dataset=alias, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                error=f"{type(exc).__name__}: {exc}", notes=f"required={item.get('required', False)}"
            ))
    return records, packages_meta


def download_inmet(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                    mode: str, logger: logging.Logger) -> list[ArtifactRecord]:
    records: list[ArtifactRecord] = []
    try:
        links = html_links(http, source.discovery_url)
        year_rx = re.compile(r"(20\d{2})")
        candidates = []
        for text, url in links:
            match = year_rx.search(text + " " + url)
            if match and (url.lower().endswith(".zip") or ".zip" in url.lower()):
                year = int(match.group(1))
                if year in source.years:
                    candidates.append((year, text, url))
        if not candidates:
            # Padrão público histórico do portal INMET; o download ainda passa por validação HTTPS/allowlist.
            candidates = [
                (year, f"INMET {year}", f"https://portal.inmet.gov.br/uploads/dadoshistoricos/{year}.zip")
                for year in source.years
            ]
        if mode == "core":
            candidates = [x for x in candidates if x[0] >= 2020]
        for year, text, url in sorted(set(candidates)):
            name = sanitize_filename(Path(urllib.parse.urlparse(url).path).name or f"INMET_{year}.zip")
            if mode == "inventory":
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                    agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                    source_url=url, file_name=name, security="https", discovered_at_utc=utc_now(),
                ))
            else:
                rec = http.download(url, dest / "archives" / name, source, run_id)
                records.append(rec)
                if rec.status in {"DOWNLOADED", "EXISTS"} and rec.local_path:
                    extracted = extract_archive(Path(rec.local_path), dest / "A301", logger, r"(A301|RECIFE)")
                    for file in extracted:
                        records.append(ArtifactRecord(
                            run_id=run_id, source_id=source.source_id, status="EXTRACTED", phase="0",
                            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                            source_url=url, local_path=str(file), file_name=file.name,
                            bytes=file.stat().st_size, sha256=sha256_file(file), security="https",
                            downloaded_at_utc=utc_now(), discovered_at_utc=utc_now(),
                            notes="Subconjunto estação A301/Recife extraído do arquivo anual"
                        ))
        if not candidates:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                error="Links ZIP anuais não detectados; revisar HTML do portal INMET"
            ))
    except Exception as exc:
        records.append(ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
            error=f"{type(exc).__name__}: {exc}"
        ))
    return records


def download_anp(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                 mode: str, logger: logging.Logger) -> list[ArtifactRecord]:
    records: list[ArtifactRecord] = []
    try:
        links = html_links(http, source.discovery_url)
        candidates: list[tuple[int, str, str]] = []
        for text, url in links:
            blob = text + " " + url
            year_m = re.search(r"(20\d{2})", blob)
            if not year_m:
                continue
            year = int(year_m.group(1))
            if year not in source.years:
                continue
            if re.search(r"(csv|combust|automotiv|semestre)", blob, re.I):
                candidates.append((year, text, url))
        # Em core, limitar aos anos diretamente usados no choque e módulos.
        if mode == "core":
            candidates = [x for x in candidates if x[0] >= 2020]
        seen = set()
        for year, text, url in candidates:
            if url in seen:
                continue
            seen.add(url)
            name = sanitize_filename(Path(urllib.parse.urlparse(url).path).name or f"ANP_{year}.csv")
            if mode == "inventory":
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                    agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                    source_url=url, file_name=name, security="https", discovered_at_utc=utc_now(),
                ))
            else:
                records.append(http.download(url, dest / str(year) / name, source, run_id))
        if not candidates:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                error="Links de dados não detectados automaticamente; metadado permanece registrado"
            ))
    except Exception as exc:
        records.append(ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
            error=f"{type(exc).__name__}: {exc}"
        ))
    return records


def download_bcb_ipca(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                      mode: str) -> list[ArtifactRecord]:
    if mode == "inventory":
        return [ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, file_name="bcb_sgs_433_ipca.json",
            security="https", discovered_at_utc=utc_now(),
        )]
    return [http.download(source.discovery_url, dest / "bcb_sgs_433_ipca.json", source, run_id)]


def download_osm(source: SourceSpec, dest: Path, run_id: str,
                 logger: logging.Logger) -> list[ArtifactRecord]:
    records: list[ArtifactRecord] = []
    try:
        import osmnx as ox
        ox.settings.use_cache = True
        ox.settings.cache_folder = str(dest / "cache")
        ox.settings.requests_timeout = 180
        for network_type in ["drive", "bike", "walk"]:
            logger.info("Baixando OSM Recife (%s)...", network_type)
            graph = ox.graph_from_place("Recife, Pernambuco, Brazil", network_type=network_type, simplify=False)
            graphml = dest / f"recife_{network_type}_unsimplified.graphml"
            gpkg = dest / f"recife_{network_type}_unsimplified.gpkg"
            ox.save_graphml(graph, graphml)
            ox.save_graph_geopackage(graph, gpkg, directed=True)
            for path in [graphml, gpkg]:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DOWNLOADED", phase="0",
                    agency=source.agency, dataset=f"{source.dataset} — {network_type}",
                    measurement_status=source.measurement_status, source_url=source.discovery_url,
                    local_path=str(path), file_name=path.name, bytes=path.stat().st_size,
                    sha256=sha256_file(path), security="https via OSMnx/Nominatim/Overpass",
                    license=source.license, downloaded_at_utc=utc_now(), discovered_at_utc=utc_now(),
                ))
    except Exception as exc:
        records.append(ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
            error=f"{type(exc).__name__}: {exc}",
        ))
    return records


# -----------------------------------------------------------------------------
# Inventário local, manifests e relatórios
# -----------------------------------------------------------------------------

def register_local_reference(source: SourceSpec, run_id: str, refs_dir: Path) -> tuple[ArtifactRecord, AuditResult | None]:
    path = Path(source.local_path or "")
    if not path.exists():
        rec = ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="MISSING_LOCAL_REQUIRED", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            local_path=str(path), file_name=path.name, security="local", discovered_at_utc=utc_now(),
            error="Arquivo local não encontrado"
        )
        return rec, None
    audit = audit_fixed_width(path, source.source_id)
    pointer = {
        "source_id": source.source_id,
        "absolute_path": str(path.resolve()),
        "bytes": path.stat().st_size,
        "sha256": audit.sha256,
        "estimated_rows": audit.estimated_rows,
        "registered_at_utc": utc_now(),
        "copy_policy": "reference_only_no_duplication",
    }
    write_json(refs_dir / f"{source.source_id}.link.json", pointer)
    rec = ArtifactRecord(
        run_id=run_id, source_id=source.source_id, status="REGISTERED_LOCAL", phase="0",
        agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
        local_path=str(path.resolve()), file_name=path.name, bytes=path.stat().st_size,
        sha256=audit.sha256, security="local", discovered_at_utc=utc_now(),
        notes="Arquivo referenciado sem duplicação para preservar espaço"
    )
    return rec, audit


def manifest_to_csv(records: Sequence[ArtifactRecord], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = [f.name for f in dataclasses.fields(ArtifactRecord)]
    with path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for record in records:
            writer.writerow(asdict(record))


def gates_to_csv(gates: Sequence[GateResult], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=["gate_id", "status", "severity", "source_id", "message", "evidence"])
        writer.writeheader()
        for gate in gates:
            row = asdict(gate)
            row["evidence"] = json.dumps(row["evidence"], ensure_ascii=False)
            writer.writerow(row)


def write_environment_lock(path: Path, run_id: str, root: Path, root_mode: str) -> None:
    packages = {}
    for pkg in ["requests", "beautifulsoup4", "lxml", "pandas", "pyarrow", "polars", "py7zr", "openpyxl", "pyyaml", "osmnx", "geopandas"]:
        try:
            packages[pkg] = importlib.metadata.version(pkg)
        except importlib.metadata.PackageNotFoundError:
            packages[pkg] = None
    data = {
        "run_id": run_id,
        "script_version": SCRIPT_VERSION,
        "schema_version": SCHEMA_VERSION,
        "created_at_utc": utc_now(),
        "root": str(root),
        "root_resolution": root_mode,
        "python": sys.version,
        "platform": platform.platform(),
        "hostname": socket.gethostname(),
        "packages": packages,
        "git_commit": _git_commit(),
    }
    write_json(path, data)


def _git_commit() -> str | None:
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL, text=True).strip()
    except Exception:
        return None


def initial_decisions(run_id: str) -> list[dict[str, Any]]:
    return [
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D001", "decision": "S140093 é o identificador direto primário de entrega por plataforma.", "rationale": "Evita proxy silenciosa.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D002", "decision": "PNAD COVID C007C=17 é ocupação de entrega, não uso direto de plataforma.", "rationale": "Limite semântico oficial.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D003", "decision": "RAIS/PNAD COVID/PNADc não serão raw-pooled como painel causal.", "rationale": "Universos e desenhos incompatíveis.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D004", "decision": "Custos do TFD serão outcomes/cenários separados e nunca subtraídos somente do tratado antes da estimação.", "rationale": "Evita inserir mecanicamente a penalidade.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D005", "decision": "Pedidos, despacho, espera e retorno vazio sem logs serão sempre rotulados como simulados.", "rationale": "Separação observado/estimado/simulado.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D006", "decision": "STGNN usará targets urbanos observados e validação espaço-temporal; random labels são proibidos.", "rationale": "Validade preditiva.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D007", "decision": "Fase 4 identifica conjunto de regimes compatíveis; não recupera o algoritmo proprietário real.", "rationale": "Identificação parcial e equifinalidade.", "status": "locked"},
    ]


def generate_report(root: Path, run_id: str, mode: str, records: Sequence[ArtifactRecord],
                    audits: Sequence[AuditResult], gates: Sequence[GateResult], root_mode: str) -> str:
    status_counts = Counter(r.status for r in records)
    gate_counts = Counter(g.status for g in gates)
    required_failures = [g for g in gates if g.status in {"FAIL", "BLOCKED"} and g.severity == "critical"]
    lines = [
        "# SPINE-GPE v7 — Relatório da Fase 0",
        "",
        f"- **Run ID:** `{run_id}`",
        f"- **Versão:** `{SCRIPT_VERSION}`",
        f"- **Modo:** `{mode}`",
        f"- **Diretório:** `{root}`",
        f"- **Resolução do diretório:** `{root_mode}`",
        f"- **Gerado em:** `{utc_now()}`",
        "",
        "## Resultado executivo",
        "",
    ]
    if required_failures:
        lines.append("**LOCK: BLOQUEADO.** Há gates críticos não atendidos; nenhuma fase inferencial deve prosseguir.")
    else:
        lines.append("**LOCK: LIBERADO PARA A PRÓXIMA ETAPA**, condicionado aos limites registrados no Claim/Evidence Book.")
    lines += [
        "",
        "## Artefatos por status",
        "",
        "| Status | N |",
        "|---|---:|",
    ]
    for status, n in sorted(status_counts.items()):
        lines.append(f"| {status} | {n} |")
    lines += ["", "## Gates", "", "| Status | N |", "|---|---:|"]
    for status, n in sorted(gate_counts.items()):
        lines.append(f"| {status} | {n} |")
    lines += ["", "## Gates críticos não atendidos", ""]
    if not required_failures:
        lines.append("Nenhum.")
    else:
        for g in required_failures:
            lines.append(f"- **{g.gate_id}** — {g.message}")
    lines += [
        "",
        "## Arquivos locais auditados",
        "",
        "| Fonte | Caminho | GiB | SHA-256 | Linhas estimadas | Comprimento modal |",
        "|---|---|---:|---|---:|---:|",
    ]
    for a in audits:
        gib = a.size_bytes / 1024**3
        lines.append(f"| {a.source_id} | `{a.path}` | {gib:.3f} | `{a.sha256[:16]}…` | {a.estimated_rows or ''} | {a.line_length_mode or ''} |")
    lines += [
        "",
        "## Regras epistemológicas congeladas",
        "",
        "1. Observado, proxy, imputado e simulado permanecem separados em todos os manifests.",
        "2. Ausência de `S140093` bloqueia identificação direta; não ativa proxy automaticamente.",
        "3. PNAD COVID mede ocupação de entrega no choque pandêmico, não plataforma confirmada.",
        "4. RAIS mede formalidade registrada e não representa o universo informal.",
        "5. CTTU mede tráfego veicular geral, não demanda observada de entregadores.",
        "6. NAIN/NACH representam configuração; não são fluxo ou demanda observados.",
        "7. Outputs da Fase 4 são contrafactuais dentro do modelo e do conjunto compatível.",
        "",
        "## Próximo artefato esperado",
        "",
        "Executar o parser dos layouts oficiais PNADc e produzir a primeira `certified_table` com golden tests de prevalência, desenho survey e identificação direta.",
    ]
    return "\n".join(lines) + "\n"


# -----------------------------------------------------------------------------
# Orquestrador
# -----------------------------------------------------------------------------

def parse_args(argv: Sequence[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="SPINE-GPE v7 — Fase 0")
    parser.add_argument("--root", help="Diretório raiz; padrão D:\\aCidadeAlgoritmica\\SPINE-GPEv7 no Windows")
    parser.add_argument("--mode", choices=["inventory", "core", "full"], default="core")
    parser.add_argument("--bootstrap", action="store_true", help="Instala dependências mínimas")
    parser.add_argument("--allow-official-ftp", action="store_true", help="Autoriza FTP oficial MTE sem criptografia")
    parser.add_argument("--download-osm", action="store_true", help="Baixa redes OSM Recife via OSMnx")
    parser.add_argument("--core-max-resource-mb", type=int, default=600, help="Limite por recurso CKAN em mode=core")
    parser.add_argument("--overwrite", action="store_true", help="Reservado para futuras versões; downloads existentes são preservados")
    parser.add_argument("--strict", action="store_true", help="Retorna exit code 2 quando gate crítico falha")
    return parser.parse_args(argv)


def main(argv: Sequence[str] | None = None) -> int:
    args = parse_args(argv)
    root, root_mode = resolve_root(args.root)
    tree = create_project_tree(root)
    run_id = dt.datetime.now().strftime("%Y%m%dT%H%M%S") + "Z"
    logger = setup_logger(tree["logs"] / f"phase0_{run_id}.log")
    logger.info("SPINE-GPE v7 Fase 0 | run=%s | mode=%s | root=%s", run_id, args.mode, root)

    if args.bootstrap:
        bootstrap_packages(logger)

    # Imports externos validados após bootstrap.
    try:
        import requests  # noqa: F401
        import pandas  # noqa: F401
        import bs4  # noqa: F401
    except ImportError as exc:
        logger.error("Dependência ausente: %s. Execute com --bootstrap.", exc)
        return 1

    write_environment_lock(tree["admin"] / f"environment_lock_{run_id}.json", run_id, root, root_mode)
    local_paths = resolve_local_pnadc_paths()
    sources = build_source_registry(local_paths)
    write_json(tree["registry"] / "source_registry.json", [asdict(s) for s in sources])
    write_json(tree["registry"] / "semantic_dictionary.json", semantic_dictionary())
    write_json(tree["contracts"] / "data_contracts.json", data_contracts())
    write_json(tree["registry"] / "estimand_registry.json", estimand_registry())
    for name, content in dag_files().items():
        (tree["dags"] / name).write_text(content + "\n", encoding="utf-8")
    for decision in initial_decisions(run_id):
        append_jsonl(tree["decisions"] / "analysis_decisions.jsonl", decision)
    # Inicializa registros mutáveis sem inventar decisões futuras.
    for name in ["exclusions.jsonl", "data_losses.jsonl", "protocol_deviations.jsonl"]:
        (tree["decisions"] / name).touch(exist_ok=True)

    records: list[ArtifactRecord] = []
    audits: list[AuditResult] = []
    gates: list[GateResult] = global_fail_closed_gates()

    # 1) Arquivos locais fundamentais.
    for source in sources:
        if source.strategy == "local_reference":
            rec, audit = register_local_reference(source, run_id, tree["raw_local_refs"])
            records.append(rec)
            if audit:
                audits.append(audit)
    gates.extend(run_local_pnadc_gates(local_paths, audits))

    # 2) Descoberta/download público.
    http = SafeHTTP(logger)
    packages_meta: list[dict[str, Any]] = []
    for source in sources:
        try:
            logger.info("Fonte: %s | estratégia=%s", source.source_id, source.strategy)
            if source.strategy == "local_reference":
                continue
            if source.strategy in {"ibge_directory_regex", "ibge_directory_all"}:
                records.extend(download_ibge_directory(
                    http, source, tree["raw_ibge"] / source.source_id, run_id, args.mode, logger
                ))
            elif source.strategy == "pnadc_historical_quarters":
                records.extend(download_pnadc_historical(
                    http, source, tree["raw_ibge"] / "pnadc_regular_historical", run_id, args.mode, logger
                ))
            elif source.strategy == "ibge_product_sidra_links":
                recs, sidra_meta = discover_ibge_sidra_tables(
                    http, source, tree["raw_ibge"] / "official_benchmarks", run_id, args.mode, logger
                )
                records.extend(recs)
                write_json(tree["registry"] / "sidra_platform_tables.json", sidra_meta)
            elif source.strategy == "ibge_pnad_covid":
                records.extend(download_pnad_covid(
                    http, source, tree["raw_ibge"] / "pnad_covid", run_id, args.mode, logger
                ))
            elif source.strategy == "direct_https":
                if args.mode == "inventory":
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                        agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                        source_url=source.discovery_url, file_name=Path(urllib.parse.urlparse(source.discovery_url).path).name,
                        security="https", discovered_at_utc=utc_now(),
                    ))
                else:
                    name = sanitize_filename(Path(urllib.parse.urlparse(source.discovery_url).path).name)
                    records.append(http.download(source.discovery_url, tree["raw_ibge"] / "censo2022" / name, source, run_id))
            elif source.strategy == "recife_ckan_bundle":
                recs, meta = download_recife_ckan(
                    http, source, tree["raw_recife"], run_id, args.mode, logger, args.core_max_resource_mb
                )
                records.extend(recs)
                packages_meta.extend(meta)
            elif source.strategy == "inmet_annual_station":
                records.extend(download_inmet(http, source, tree["raw_inmet"], run_id, args.mode, logger))
            elif source.strategy == "anp_page_links":
                records.extend(download_anp(http, source, tree["raw_anp"], run_id, args.mode, logger))
            elif source.strategy == "bcb_api":
                records.extend(download_bcb_ipca(http, source, tree["raw_bcb"], run_id, args.mode))
            elif source.strategy in {"mte_ftp_rais_nordeste", "mte_ftp_novo_caged"}:
                if not args.allow_official_ftp:
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="BLOCKED_SECURITY_OPT_IN", phase="0",
                        agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                        source_url=source.discovery_url, security=source.security, discovered_at_utc=utc_now(),
                        notes="Use --allow-official-ftp para autorizar o FTP oficial MTE sem criptografia."
                    ))
                    gates.append(GateResult(
                        gate_id=f"{source.source_id}.transport_opt_in", status="BLOCKED", severity="high",
                        source_id=source.source_id,
                        message="Download automático bloqueado porque a fonte oficial oferece FTP sem criptografia.",
                    ))
                elif args.mode == "full":
                    if source.strategy == "mte_ftp_rais_nordeste":
                        records.extend(download_mte_rais(source, tree["raw_mte"] / "RAIS", run_id, logger, source.years))
                    else:
                        records.extend(download_mte_novo_caged(source, tree["raw_mte"] / "NOVO_CAGED", run_id, logger, source.years))
                else:
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="DISCOVERED_NOT_DOWNLOADED_MODE_FULL_REQUIRED",
                        phase="0", agency=source.agency, dataset=source.dataset,
                        measurement_status=source.measurement_status, source_url=source.discovery_url,
                        security=source.security, discovered_at_utc=utc_now(),
                        notes="Use --mode full --allow-official-ftp."
                    ))
            elif source.strategy == "osmnx_place":
                if args.download_osm and args.mode == "full":
                    records.extend(download_osm(source, tree["raw_osm"], run_id, logger))
                else:
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="DISCOVERED_NOT_DOWNLOADED",
                        phase="0", agency=source.agency, dataset=source.dataset,
                        measurement_status=source.measurement_status, source_url=source.discovery_url,
                        security="https", license=source.license, discovered_at_utc=utc_now(),
                        notes="Use --mode full --download-osm."
                    ))
        except Exception as exc:
            logger.error("Erro não tratado em %s: %s", source.source_id, exc)
            logger.debug(traceback.format_exc())
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, local_path=source.local_path,
                security=source.security, discovered_at_utc=utc_now(),
                error=f"{type(exc).__name__}: {exc}",
            ))

    write_json(tree["registry"] / "recife_ckan_packages_resolved.json", packages_meta)

    # 3) Extração de documentação baixada e resolução de nomes de variáveis.
    doc_dir = tree["raw_ibge"] / "pnadc_documentation"
    for archive in doc_dir.rglob("*.zip"):
        extract_archive(archive, tree["interim"] / "pnadc_documentation_extracted" / archive.stem, logger)
    doc_paths = discover_layout_files(tree["raw_ibge"]) + discover_layout_files(tree["interim"])
    variable_names = extract_variable_names_from_docs(doc_paths)
    write_json(tree["registry"] / "pnadc_detected_variables.json", {
        "schema_version": SCHEMA_VERSION,
        "detected_at_utc": utc_now(),
        "documentation_files": [str(p) for p in doc_paths],
        "variables": sorted(variable_names),
    })
    gates.extend(run_semantic_gates(variable_names, docs_found=bool(doc_paths)))

    # 4) Auditoria leve de arquivos tabulares baixados (evita reprocessar arquivos gigantes).
    for rec in records:
        if rec.status not in {"DOWNLOADED", "EXISTS", "EXTRACTED"} or not rec.local_path:
            continue
        path = Path(rec.local_path)
        if not path.exists() or path.stat().st_size > 3 * 1024**3:
            continue
        if path.suffix.lower() in {".csv", ".tsv", ".txt"}:
            try:
                audits.append(audit_generic(path, rec.source_id))
            except Exception as exc:
                logger.warning("Auditoria tabular falhou em %s: %s", path, exc)

    # 5) Gates de disponibilidade CKAN essenciais.
    required_aliases = [k for k, v in RECIFE_CKAN_PACKAGES.items() if v.get("required")]
    for alias in required_aliases:
        matched = [r for r in records if r.source_id == f"recife_{alias}" and r.status not in {"ERROR", "NOT_FOUND"}]
        gates.append(GateResult(
            gate_id=f"recife.{alias}.available",
            status="PASS" if matched else "FAIL",
            severity="critical" if alias in {"velocidade_2022", "velocidade_2023", "velocidade_2024", "equipamentos_transito"} else "high",
            source_id=f"recife_{alias}",
            message="Pacote/recurso CKAN localizado." if matched else "Pacote/recurso CKAN obrigatório não localizado.",
            evidence={"records": len(matched)},
        ))

    # 6) Manifest, auditorias, gates e relatório.
    write_json(tree["manifests"] / f"artifacts_{run_id}.json", [asdict(r) for r in records])
    manifest_to_csv(records, tree["manifests"] / f"artifacts_{run_id}.csv")
    write_json(tree["reports"] / f"audits_{run_id}.json", [asdict(a) for a in audits])
    write_json(tree["reports"] / f"gates_{run_id}.json", [asdict(g) for g in gates])
    gates_to_csv(gates, tree["reports"] / f"gates_{run_id}.csv")

    report = generate_report(root, run_id, args.mode, records, audits, gates, root_mode)
    report_path = tree["reports"] / f"PHASE0_REPORT_{run_id}.md"
    report_path.write_text(report, encoding="utf-8")
    (tree["reports"] / "PHASE0_REPORT_LATEST.md").write_text(report, encoding="utf-8")

    # Lock final: qualquer FAIL/BLOCKED critical impede avanço.
    critical_fail = [g for g in gates if g.severity == "critical" and g.status in {"FAIL", "BLOCKED"}]
    lock = {
        "run_id": run_id,
        "script_version": SCRIPT_VERSION,
        "schema_version": SCHEMA_VERSION,
        "status": "BLOCKED" if critical_fail else "RELEASED",
        "critical_failures": [asdict(g) for g in critical_fail],
        "report": str(report_path),
        "created_at_utc": utc_now(),
    }
    write_json(tree["admin"] / "PHASE0_LOCK.json", lock)

    logger.info("Fase 0 concluída. LOCK=%s | relatório=%s", lock["status"], report_path)
    if critical_fail:
        for gate in critical_fail:
            logger.error("GATE CRÍTICO: %s — %s", gate.gate_id, gate.message)
    return 2 if args.strict and critical_fail else 0


if __name__ == "__main__":
    raise SystemExit(main())

usage: colab_kernel_launcher.py [-h] [--root ROOT]
                                [--mode {inventory,core,full}] [--bootstrap]
                                [--allow-official-ftp] [--download-osm]
                                [--core-max-resource-mb CORE_MAX_RESOURCE_MB]
                                [--overwrite] [--strict]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-523b4d14-b074-4c1e-9ce2-f664160859ba.json


SystemExit: 2

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
SPINE-GPE v7 — FASE 0: DATA, IDENTIFIABILITY & REPRODUCIBILITY LOCK
===================================================================

Objetivo
--------
Criar a fundação auditável e fail-closed da tese "A Cidade Algorítmica":

1. inventariar e versionar as fontes;
2. registrar hashes e proveniência;
3. baixar dados públicos apenas de domínios autorizados;
4. auditar os dois arquivos locais PNADc usados como base direta;
5. criar dicionário semântico, contratos, schema registry e golden tests;
6. auditar cobertura temporal/espacial;
7. registrar observado/proxy/imputado/simulado;
8. congelar DAGs, estimandos e limites de identificação;
9. impedir fallbacks silenciosos e leakage;
10. produzir um relatório de viabilidade para TODAS as fases da SPINE-GPE v7.

Execução recomendada
--------------------
Windows / Colab Local Runtime (acessa D:\\):
    python SPINE_GPEv7_FASE0.py --mode core
    python SPINE_GPEv7_FASE0.py --mode full --allow-official-ftp --download-osm

Google Colab hospedado (não acessa D:\\):
    O script monta/usa Google Drive quando disponível e grava em:
    /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7

Modos
-----
inventory : cria registry/contratos/DAGs, audita arquivos locais e descobre metadados.
core      : baixa fontes essenciais e de tamanho moderado.
full      : tenta baixar todo o acervo configurado; RAIS/CAGED exigem opt-in de FTP.

Segurança
---------
- HTTPS é obrigatório por padrão.
- O FTP oficial do MTE é texto claro; só é usado com --allow-official-ftp.
- Todos os arquivos recebem SHA-256 e registro de cabeçalhos/proveniência.
- Respostas HTML disfarçadas de dados são rejeitadas.
- Fallback de identificação direta para proxy é PROIBIDO.

Este script é a Fase 0. Ele NÃO executa ainda as estimativas finais das Fases 1–6.
"""

from __future__ import annotations

import argparse
import csv
import dataclasses
import datetime as dt
import ftplib
import hashlib
import importlib.metadata
import io
import json
import logging
import mimetypes
import os
import platform
import re
import shutil
import socket
import subprocess
import sys
import tempfile
import textwrap
import time
import traceback
import urllib.parse
import zipfile
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterable, Iterator, Mapping, Sequence

# Dependências externas são carregadas depois do bootstrap.

SCRIPT_VERSION = "7.0.1-phase0-colab-argv-fix"
SCHEMA_VERSION = "spine-gpe-v7-schema-1.0.0"
DEFAULT_WINDOWS_ROOT = Path(r"D:\aCidadeAlgoritmica\SPINE-GPEv7")
DEFAULT_COLAB_ROOT = Path("/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7")
DEFAULT_COLAB_EPHEMERAL_ROOT = Path("/content/SPINE-GPEv7")

LOCAL_PNADC_PATHS = {
    "pnadc_2022q4_direct": Path(r"D:\aCidadeAlgoritmica\PNADC_042022_20250815\PNADC_042022.txt"),
    "pnadc_2024q3_direct": Path(r"D:\aCidadeAlgoritmica\PNADC_032024_20250815\PNADC_032024.txt"),
}

OFFICIAL_HTTPS_HOSTS = {
    "ftp.ibge.gov.br",
    "www.ibge.gov.br",
    "servicodados.ibge.gov.br",
    "apisidra.ibge.gov.br",
    "dados.recife.pe.gov.br",
    "portal.inmet.gov.br",
    "www.gov.br",
    "api.bcb.gov.br",
    "download.geofabrik.de",
    "overpass-api.de",
    "nominatim.openstreetmap.org",
}

CORE_YEARS = list(range(2017, dt.datetime.now().year + 1))
PNADC_REQUIRED_DIRECT = {
    "UF", "UPA", "Estrato", "V1028", "V2007", "V2009", "V2010",
    "V4010", "V4012", "V4013", "V4039", "S140093", "SD14001",
}
PNADC_REQUIRED_CORE = {"UF", "UPA", "Estrato", "V1028", "V4010", "V4012", "V4013", "V4039"}
PNAD_COVID_REQUIRED = {"UF", "V1012", "V1032", "C001", "C007", "C007C", "C009", "C010"}

# -----------------------------------------------------------------------------
# Modelos de metadados
# -----------------------------------------------------------------------------

@dataclass(slots=True)
class SourceSpec:
    source_id: str
    phases: list[str]
    agency: str
    dataset: str
    source_class: str
    measurement_status: str
    strategy: str
    discovery_url: str | None = None
    local_path: str | None = None
    package_slug: str | None = None
    years: list[int] = field(default_factory=list)
    required: bool = True
    expected_formats: list[str] = field(default_factory=list)
    spatial_scope: str = ""
    temporal_scope: str = ""
    license: str = ""
    security: str = "https"
    claim_ceiling: str = "descriptive"
    notes: str = ""


@dataclass(slots=True)
class ArtifactRecord:
    run_id: str
    source_id: str
    status: str
    phase: str
    agency: str
    dataset: str
    measurement_status: str
    source_url: str | None = None
    local_path: str | None = None
    file_name: str | None = None
    media_type: str | None = None
    bytes: int | None = None
    sha256: str | None = None
    etag: str | None = None
    last_modified: str | None = None
    downloaded_at_utc: str | None = None
    discovered_at_utc: str | None = None
    security: str | None = None
    license: str | None = None
    error: str | None = None
    notes: str | None = None


@dataclass(slots=True)
class GateResult:
    gate_id: str
    status: str  # PASS/WARN/FAIL/BLOCKED/SKIP
    severity: str
    source_id: str
    message: str
    evidence: dict[str, Any] = field(default_factory=dict)


@dataclass(slots=True)
class AuditResult:
    source_id: str
    path: str
    kind: str
    size_bytes: int
    sha256: str
    encoding: str | None = None
    delimiter: str | None = None
    sample_rows: int | None = None
    estimated_rows: int | None = None
    columns: list[str] = field(default_factory=list)
    line_length_min: int | None = None
    line_length_max: int | None = None
    line_length_mode: int | None = None
    date_min: str | None = None
    date_max: str | None = None
    spatial_bounds: list[float] | None = None
    warnings: list[str] = field(default_factory=list)


# -----------------------------------------------------------------------------
# Logging e utilidades
# -----------------------------------------------------------------------------

def utc_now() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def json_default(obj: Any) -> Any:
    if isinstance(obj, Path):
        return str(obj)
    if dataclasses.is_dataclass(obj):
        return asdict(obj)
    if isinstance(obj, set):
        return sorted(obj)
    raise TypeError(f"Tipo não serializável: {type(obj)!r}")


def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(data, ensure_ascii=False, indent=2, default=json_default), encoding="utf-8")
    tmp.replace(path)


def append_jsonl(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(data, ensure_ascii=False, default=json_default) + "\n")


def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def estimate_line_count(path: Path, sample_bytes: int = 64 * 1024 * 1024) -> int | None:
    size = path.stat().st_size
    if size == 0:
        return 0
    read_bytes = min(size, sample_bytes)
    with path.open("rb") as f:
        data = f.read(read_bytes)
    n = data.count(b"\n")
    if n == 0:
        return None
    return int(round(n * size / read_bytes))


def sanitize_filename(name: str) -> str:
    name = urllib.parse.unquote(name).strip().replace("\\", "_").replace("/", "_")
    name = re.sub(r"[^0-9A-Za-zÀ-ÿ._()\- ]+", "_", name)
    name = re.sub(r"\s+", "_", name)
    return name[:220] or "download.bin"


def setup_logger(log_path: Path) -> logging.Logger:
    logger = logging.getLogger("spine_phase0")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(sh)
    logger.addHandler(fh)
    return logger


def is_colab() -> bool:
    return "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))


def resolve_root(cli_root: str | None) -> tuple[Path, str]:
    if cli_root:
        return Path(cli_root).expanduser(), "cli"
    env_root = os.environ.get("SPINE_GPE_ROOT")
    if env_root:
        return Path(env_root).expanduser(), "env"
    if os.name == "nt":
        return DEFAULT_WINDOWS_ROOT, "windows_default"
    if is_colab():
        if DEFAULT_COLAB_ROOT.parent.exists():
            return DEFAULT_COLAB_ROOT, "colab_drive_fallback"
        return DEFAULT_COLAB_EPHEMERAL_ROOT, "colab_ephemeral_fallback"
    return Path.cwd() / "SPINE-GPEv7", "portable_fallback"


def resolve_local_pnadc_paths() -> dict[str, Path]:
    """Permite override por variáveis de ambiente no Colab hospedado."""
    out = dict(LOCAL_PNADC_PATHS)
    overrides = {
        "pnadc_2022q4_direct": os.environ.get("PNADC_2022Q4_PATH"),
        "pnadc_2024q3_direct": os.environ.get("PNADC_2024Q3_PATH"),
    }
    for key, value in overrides.items():
        if value:
            out[key] = Path(value)
    if os.name != "nt" and is_colab():
        # Fallback esperado no Drive, preservando o mesmo layout lógico.
        drive = Path("/content/drive/MyDrive/aCidadeAlgoritmica")
        candidates = {
            "pnadc_2022q4_direct": drive / "PNADC_042022_20250815/PNADC_042022.txt",
            "pnadc_2024q3_direct": drive / "PNADC_032024_20250815/PNADC_032024.txt",
        }
        for key, candidate in candidates.items():
            if not out[key].exists() and candidate.exists():
                out[key] = candidate
    return out


def create_project_tree(root: Path) -> dict[str, Path]:
    tree = {
        "root": root,
        "admin": root / "00_admin",
        "registry": root / "00_admin/registry",
        "contracts": root / "00_admin/contracts",
        "manifests": root / "00_admin/manifests",
        "logs": root / "00_admin/logs",
        "reports": root / "00_admin/reports",
        "dags": root / "00_admin/dags",
        "decisions": root / "00_admin/decisions",
        "raw": root / "01_raw",
        "raw_local_refs": root / "01_raw/00_local_references",
        "raw_ibge": root / "01_raw/10_ibge",
        "raw_mte": root / "01_raw/20_mte",
        "raw_recife": root / "01_raw/30_recife_ckan",
        "raw_inmet": root / "01_raw/40_inmet",
        "raw_anp": root / "01_raw/50_anp",
        "raw_bcb": root / "01_raw/60_bcb",
        "raw_osm": root / "01_raw/70_osm",
        "interim": root / "02_interim",
        "processed": root / "03_processed",
        "models": root / "04_models",
        "outputs": root / "05_outputs",
        "cache": root / "99_cache",
    }
    for p in tree.values():
        p.mkdir(parents=True, exist_ok=True)
    return tree


def bootstrap_packages(logger: logging.Logger) -> None:
    packages = [
        "requests>=2.31", "beautifulsoup4>=4.12", "lxml>=5.0",
        "pandas>=2.1", "pyarrow>=15", "polars>=0.20",
        "charset-normalizer>=3.3", "py7zr>=0.21", "openpyxl>=3.1",
        "pyyaml>=6.0", "tqdm>=4.66", "rich>=13.7",
        "osmnx>=1.9", "geopandas>=0.14",
    ]
    logger.info("Bootstrap de dependências Python...")
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    subprocess.run(cmd, check=True)


# -----------------------------------------------------------------------------
# Registry completo de fontes
# -----------------------------------------------------------------------------

def build_source_registry(local_paths: Mapping[str, Path]) -> list[SourceSpec]:
    current_year = dt.datetime.now().year
    specs: list[SourceSpec] = [
        SourceSpec(
            source_id="pnadc_2022q4_direct_local",
            phases=["0", "1", "2"], agency="IBGE", dataset="PNADc 2022T4 — microdado local",
            source_class="survey_microdata", measurement_status="observado_direto",
            strategy="local_reference", local_path=str(local_paths["pnadc_2022q4_direct"]),
            spatial_scope="Brasil/UF/RM conforme desenho", temporal_scope="2022T4",
            claim_ceiling="inferência survey; comparação ajustada; não causal sem hipóteses adicionais",
            notes="Base direta de plataforma; S140093 é obrigatório; fallback para proxy é proibido."
        ),
        SourceSpec(
            source_id="pnadc_2024q3_direct_local",
            phases=["0", "1", "2"], agency="IBGE", dataset="PNADc 2024T3 — microdado local",
            source_class="survey_microdata", measurement_status="observado_direto",
            strategy="local_reference", local_path=str(local_paths["pnadc_2024q3_direct"]),
            spatial_scope="Brasil/UF/RM conforme desenho", temporal_scope="2024T3",
            claim_ceiling="inferência survey; comparação ajustada; não causal sem hipóteses adicionais",
            notes="Base direta de plataforma; S140093 é obrigatório; comparação 2022–2024 sofre sazonalidade."
        ),
        SourceSpec(
            source_id="pnadc_official_2022q4_archive", phases=["0", "1", "2"], agency="IBGE",
            dataset="PNADc 2022T4 — arquivo oficial", source_class="survey_microdata",
            measurement_status="observado_direto", strategy="ibge_directory_regex",
            discovery_url="https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2022/",
            years=[2022], expected_formats=["zip"], temporal_scope="2022T4", spatial_scope="Brasil",
            claim_ceiling="proveniência e reprodução", notes=r"Regex: PNADC_042022.*\.zip"
        ),
        SourceSpec(
            source_id="pnadc_official_2024q3_archive", phases=["0", "1", "2"], agency="IBGE",
            dataset="PNADc 2024T3 — arquivo oficial", source_class="survey_microdata",
            measurement_status="observado_direto", strategy="ibge_directory_regex",
            discovery_url="https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2024/",
            years=[2024], expected_formats=["zip"], temporal_scope="2024T3", spatial_scope="Brasil",
            claim_ceiling="proveniência e reprodução", notes=r"Regex: PNADC_032024.*\.zip"
        ),
        SourceSpec(
            source_id="pnadc_documentation", phases=["0", "1", "2"], agency="IBGE",
            dataset="PNADc trimestral — documentação, inputs, dicionários e deflatores",
            source_class="official_metadata", measurement_status="observado_metadado",
            strategy="ibge_directory_all",
            discovery_url="https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/Documentacao/",
            expected_formats=["zip", "xls", "xlsx", "pdf", "txt", "sas"], spatial_scope="Brasil",
            temporal_scope="versão vigente", claim_ceiling="definição semântica oficial"
        ),
        SourceSpec(
            source_id="pnadc_regular_historical", phases=["0", "1", "2", "4"], agency="IBGE",
            dataset="PNADc trimestral regular — série histórica", source_class="survey_microdata",
            measurement_status="observado_survey_proxy_calibravel", strategy="pnadc_historical_quarters",
            discovery_url="https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/",
            years=list(range(2017, 2025)), expected_formats=["zip"], spatial_scope="Brasil/UF/RM conforme desenho",
            temporal_scope="2017T1–2024T4", claim_ceiling="reconstrução probabilística histórica; plataforma não direta fora dos módulos",
            notes="Mode=full baixa os quatro trimestres por ano; core apenas descobre o inventário remoto."
        ),
        SourceSpec(
            source_id="pnadc_platform_official_tables", phases=["0", "1"], agency="IBGE/SIDRA",
            dataset="Tabelas oficiais — trabalho por plataformas digitais 2024", source_class="official_benchmark",
            measurement_status="observado_agregado_oficial", strategy="ibge_product_sidra_links",
            discovery_url="https://www.ibge.gov.br/estatisticas/sociais/trabalho/17270-pnad-%20continua.html/17270-pnad-continua.html?edicao=44741",
            expected_formats=["html", "json"], spatial_scope="Brasil/Grandes Regiões/UF/RM conforme tabela",
            temporal_scope="2022T4 e 2024T3", claim_ceiling="golden tests e reprodução de totais oficiais"
        ),
        SourceSpec(
            source_id="pnad_covid_2020", phases=["0", "1", "2"], agency="IBGE",
            dataset="PNAD COVID19 — microdados mensais", source_class="survey_microdata",
            measurement_status="observado_ocupacional", strategy="ibge_pnad_covid",
            discovery_url="https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_PNAD_COVID19/Microdados/",
            years=[2020], expected_formats=["zip", "xls", "xlsx", "pdf"], spatial_scope="Brasil/UF",
            temporal_scope="2020-05 a 2020-11", claim_ceiling="ponte pandêmica ocupacional; não plataforma direta"
        ),
        SourceSpec(
            source_id="rais_nordeste_2017_latest", phases=["0", "1", "2", "4"], agency="MTE/PDET",
            dataset="RAIS vínculos públicos — Nordeste", source_class="administrative_microdata",
            measurement_status="observado_administrativo", strategy="mte_ftp_rais_nordeste",
            discovery_url="ftp://ftp.mtps.gov.br/pdet/microdados/RAIS/",
            years=list(range(2017, current_year)), expected_formats=["7z", "txt"], spatial_scope="Nordeste/município",
            temporal_scope=f"2017–{current_year-1}", security="official_plaintext_ftp_opt_in",
            claim_ceiling="baseline formal; registro administrativo; não representa informalidade"
        ),
        SourceSpec(
            source_id="novo_caged_2020_latest", phases=["0", "1", "2"], agency="MTE/PDET",
            dataset="Novo CAGED — movimentações", source_class="administrative_microdata",
            measurement_status="observado_administrativo", strategy="mte_ftp_novo_caged",
            discovery_url="ftp://ftp.mtps.gov.br/pdet/microdados/NOVO%20CAGED/",
            years=list(range(2020, current_year + 1)), expected_formats=["7z", "txt"], spatial_scope="Brasil/município",
            temporal_scope=f"2020–{current_year}", security="official_plaintext_ftp_opt_in",
            claim_ceiling="dinâmica mensal formal; não plataforma direta"
        ),
        SourceSpec(
            source_id="censo2022_pe_setores", phases=["0", "3A", "4", "5"], agency="IBGE",
            dataset="Censo 2022 — setores com atributos PE", source_class="census_geodata",
            measurement_status="observado_censitario", strategy="direct_https",
            discovery_url="https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/malha_com_atributos/setores/gpkg/UF/PE/PE_setores_CD2022.gpkg",
            expected_formats=["gpkg"], spatial_scope="Pernambuco/setor", temporal_scope="2022",
            claim_ceiling="estrutura territorial censitária; SAE/MRP model-based"
        ),
        SourceSpec(
            source_id="recife_ckan_bundle", phases=["0", "3A", "3B", "4", "5"], agency="Prefeitura do Recife",
            dataset="Pacote CKAN urbano Recife", source_class="municipal_open_data",
            measurement_status="observado_municipal", strategy="recife_ckan_bundle",
            discovery_url="https://dados.recife.pe.gov.br/api/3/action/", spatial_scope="Recife",
            temporal_scope="conforme recurso", license="ODbL/licença indicada no pacote",
            claim_ceiling="dinâmica urbana observada ou potencial locacional conforme conjunto"
        ),
        SourceSpec(
            source_id="inmet_recife", phases=["0", "3B", "4", "5"], agency="INMET",
            dataset="Dados meteorológicos históricos — estação Recife A301", source_class="weather_timeseries",
            measurement_status="observado_instrumental", strategy="inmet_annual_station",
            discovery_url="https://portal.inmet.gov.br/dadoshistoricos", years=CORE_YEARS,
            expected_formats=["zip", "csv"], spatial_scope="Recife/estação A301", temporal_scope=f"2017–{current_year}",
            claim_ceiling="controle meteorológico e choques observados"
        ),
        SourceSpec(
            source_id="anp_combustiveis", phases=["0", "2", "4", "5"], agency="ANP",
            dataset="Série histórica de preços de combustíveis", source_class="price_microdata",
            measurement_status="observado_mercado", strategy="anp_page_links",
            discovery_url="https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/serie-historica-de-precos-de-combustiveis",
            years=CORE_YEARS, expected_formats=["csv", "pdf"], spatial_scope="Município/posto",
            temporal_scope=f"2017–{current_year}", claim_ceiling="choques de custo e pass-through; não custo individual"
        ),
        SourceSpec(
            source_id="bcb_ipca", phases=["0", "1", "2", "4", "5"], agency="Banco Central do Brasil/IBGE",
            dataset="SGS 433 — IPCA mensal", source_class="price_index", measurement_status="observado_indice",
            strategy="bcb_api", discovery_url="https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json",
            expected_formats=["json"], spatial_scope="Brasil", temporal_scope="1980–atual",
            claim_ceiling="harmonização monetária"
        ),
        SourceSpec(
            source_id="osm_recife", phases=["0", "3A", "3B", "4", "5"], agency="OpenStreetMap",
            dataset="Rede viária multimodal Recife", source_class="volunteered_geodata",
            measurement_status="observado_cartografico", strategy="osmnx_place",
            discovery_url="https://www.openstreetmap.org/", expected_formats=["graphml", "gpkg"],
            spatial_scope="Recife", temporal_scope="snapshot da execução", license="ODbL",
            claim_ceiling="estrutura de rede; qualidade depende da cobertura OSM"
        ),
    ]
    return specs


RECIFE_CKAN_PACKAGES: dict[str, dict[str, Any]] = {
    "velocidade_2016": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2016", "required": False},
    "velocidade_2017": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2017", "required": False},
    "velocidade_2018": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media", "required": False},
    "velocidade_2019": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2019", "required": False},
    "velocidade_2020": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2020", "required": False},
    "velocidade_2021": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2021", "required": False},
    "velocidade_2022": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2022", "required": True},
    "velocidade_2023": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2023", "required": True},
    "velocidade_2024": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2024", "required": True},
    "equipamentos_transito": {"slug": "equipamentos-de-monitoramento-e-fiscalizacao-de-transito", "required": True},
    "sinistros_transito": {"slug": "acidentes-de-transito-com-e-sem-vitimas", "required": True},
    "empresas": {"slug": "empresas-da-cidade-do-recife", "required": True},
    "bares_restaurantes": {"slug": "bares-e-restaurantes", "required": True},
    "itbi": {"slug": "imposto-sobre-transmissao-de-bens-imoveis-itbi", "required": True},
    # Os itens abaixo usam busca CKAN se o slug mudar.
    "iluminacao": {"query": "iluminação pública", "required": False},
    "ciclovias": {"query": "malha cicloviária", "required": False},
    "salva_bike": {"query": "Salva Bike", "required": False},
    "mercados_publicos": {"query": "mercados públicos", "required": False},
    "parques_pracas": {"query": "parques praças", "required": False},
    "equipamentos_publicos": {"query": "equipamentos públicos", "required": False},
    "terminais": {"query": "terminais transporte", "required": False},
    "malha_viaria": {"query": "malha viária logradouros eixos viários", "required": False},
}


# -----------------------------------------------------------------------------
# Dicionário semântico, contratos, estimandos e DAGs
# -----------------------------------------------------------------------------

def semantic_dictionary() -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []

    def add(source: str, var: str, concept: str, role: str, status: str,
            allowed_use: str, forbidden_use: str = "", notes: str = "") -> None:
        rows.append({
            "schema_version": SCHEMA_VERSION, "source": source, "variable": var,
            "concept": concept, "role": role, "measurement_status": status,
            "allowed_use": allowed_use, "forbidden_use": forbidden_use, "notes": notes,
        })

    # PNADc direta e regular
    add("PNADc", "S140093", "uso de plataforma de entrega", "treatment/direct identifier",
        "observado_direto", "identificação direta no módulo especial",
        "não substituir silenciosamente por CBO/CNAE quando ausente")
    add("PNADc", "SD14001", "trabalho por plataforma em ao menos um tipo", "direct identifier",
        "observado_direto", "prevalência geral de plataforma", "não equivale especificamente a entrega")
    for v, c in [("S140091", "táxi"), ("S140092", "transporte particular de passageiros"),
                 ("S140094", "serviços gerais/profissionais")]:
        add("PNADc", v, f"uso de plataforma — {c}", "direct identifier", "observado_direto",
            "tipologia de plataforma", "não usar como entrega")
    add("PNADc", "V4010", "ocupação no trabalho principal", "covariate/occupation",
        "observado_survey", "ocupação e proxy histórica calibrada", "não é plataforma por si só")
    add("PNADc", "V4012", "posição na ocupação", "covariate/employment position",
        "observado_survey", "comparadores e formalidade", "código 4 não deve ser tratado como conta própria")
    add("PNADc", "V4013", "atividade do empreendimento", "covariate/activity",
        "observado_survey", "ocupação × atividade e proxy histórica", "não confundir CNAE classe 53202 com código domiciliar 53002")
    add("PNADc", "V4039", "horas habitualmente trabalhadas", "outcome/exposure",
        "observado_survey", "jornada", "não combinar mecanicamente com renda-hora no mesmo lado da regressão")
    add("PNADc", "VD4016", "rendimento habitual mensal do trabalho principal", "outcome",
        "observado_survey", "renda bruta principal", "não subtrair custos apenas do tratado antes da estimação")
    add("PNADc", "VD4020", "rendimento efetivo mensal de todos os trabalhos", "outcome",
        "observado_survey", "renda efetiva, após confirmação no dicionário", "não tratar como ocupação")
    add("PNADc", "V1028", "peso final", "survey design", "observado_survey",
        "estimação survey", "não usar como variável substantiva")
    add("PNADc", "UPA", "unidade primária de amostragem", "survey design", "observado_survey",
        "variância survey", "não ignorar em inferência")
    add("PNADc", "Estrato", "estrato amostral", "survey design", "observado_survey",
        "variância survey", "não ignorar em inferência")

    # PNAD COVID
    add("PNAD_COVID", "C007C", "ocupação em categorias especiais", "occupation identifier",
        "observado_ocupacional", "16=motoboy; 17=entrega de mercadorias",
        "não chamar diretamente de uso de aplicativo")
    add("PNAD_COVID", "C007", "posição na ocupação", "covariate", "observado_survey",
        "formalidade/conta própria", "não equiparar todos os demais códigos a formal")
    add("PNAD_COVID", "C001", "trabalhou na semana", "universe filter", "observado_survey",
        "sensibilidade trabalhador ativo", "não confundir com todo o universo ocupado")
    add("PNAD_COVID", "C014", "contribuição ao INSS", "social protection", "observado_survey",
        "proteção previdenciária", "não confundir com local de trabalho")

    # RAIS/CAGED
    add("RAIS", "CBO 2002", "ocupação formal", "occupation identifier", "observado_administrativo",
        "baseline formal", "não representa entregadores informais")
    add("RAIS", "Vl Remun Média Nom", "remuneração média nominal", "outcome", "observado_administrativo",
        "renda formal", "não comparar nominalmente entre anos sem deflator")
    add("RAIS", "Qtd Hora Contr", "horas contratuais", "outcome/exposure", "observado_administrativo",
        "jornada contratual", "não equivale a horas efetivas")

    # Espacial/temporal
    add("SINTAXE", "NAIN_r", "integração angular normalizada por raio", "spatial prior",
        "derivado_observado", "estrutura configuracional", "não chamar de demanda observada")
    add("SINTAXE", "NACH_r", "choice angular normalizada por raio", "spatial prior",
        "derivado_observado", "potencial de intermediação", "não chamar de fluxo real")
    add("CTTU", "fluxo_15min", "contagem veicular por equipamento e intervalo", "dynamic outcome",
        "observado_instrumental", "treino/validação do grafo dinâmico", "não equivale a fluxo de entregadores")
    add("SIM", "pedidos_potenciais", "pedidos gerados pelo emulador", "latent simulation",
        "simulado", "cenários e identificação de conjunto", "não reportar como pedidos observados")
    add("SIM", "TFD_contrafactual", "carga de custos sob política simulada", "counterfactual outcome",
        "simulado_parcialmente_identificado", "comparação de regimes", "não reportar como valor individual observado")
    return rows


def data_contracts() -> dict[str, Any]:
    return {
        "schema_version": SCHEMA_VERSION,
        "global": {
            "timezone": "America/Recife",
            "crs_geographic": "EPSG:4674",
            "crs_projected_recife": "EPSG:31985",
            "currency_nominal": "BRL",
            "currency_real_base": "IPCA, base configurável no pipeline",
            "missing_policy": "nunca converter missing em zero sem regra explícita",
            "fallback_policy": "fail_closed",
        },
        "pnadc_direct": {
            "required_variables": sorted(PNADC_REQUIRED_DIRECT),
            "treatment": "S140093",
            "survey_design": ["V1028", "UPA", "Estrato"],
            "forbidden_fallbacks": ["V4010 sozinho", "VD4019/VD4020 como ocupação", "proxy quando S140093 ausente"],
            "gates": [
                "S140093 presente no layout oficial",
                "tratados e controles positivos",
                "totais ponderados replicam tabelas oficiais dentro da tolerância",
                "amostra de outcome separada do universo identificado",
            ],
        },
        "pnad_covid": {
            "required_variables": sorted(PNAD_COVID_REQUIRED),
            "narrow_delivery": "C007C == 17",
            "broad_logistics": "C007C in {16,17}",
            "platform_direct": False,
            "claim_ceiling": "ocupação de entrega no choque pandêmico",
        },
        "rais": {
            "delimiter": ";",
            "expected_encoding_candidates": ["latin1", "cp1252", "utf-8"],
            "occupation_aliases": ["CBO Ocupação 2002", "CBO 2002 Ocupação", "CBO Ocupação 2002"],
            "income_aliases": ["Vl Remun Média Nom", "Vl Remun Média (SM)", "Vl Remun Dezembro Nom"],
            "hours_aliases": ["Qtd Hora Contr"],
            "geography_aliases": ["Mun Trab", "Município"],
            "claim_ceiling": "formalidade registrada",
        },
        "cttu_speed": {
            "time_resolution_expected_minutes": 15,
            "minimum_stable_sensors": 20,
            "minimum_continuous_months": 6,
            "maximum_missing_fraction": 0.30,
            "minimum_map_match_rate": 0.80,
            "claim_ceiling": "tráfego/velocidade observados; não demanda de plataforma",
        },
        "spatial": {
            "required_modes": ["drive", "bike", "walk"],
            "syntax_radii_m": [400, 800, 1200, 2000, 5000, "global"],
            "preserve_tags": ["bridge", "tunnel", "layer", "oneway", "access", "highway"],
            "hexagons_m": [250, 500],
            "aggregation": "length_weighted",
        },
    }


def estimand_registry() -> list[dict[str, Any]]:
    return [
        {
            "estimand_id": "E1_PLATFORM_GAP",
            "phase": "2",
            "question": "Diferença ajustada entre entregadores de plataforma e comparadores não-plataforma",
            "population": "suporte comum dentro da PNADc especial do mesmo trimestre",
            "treatment": "S140093",
            "outcomes": "renda bruta, jornada, renda-hora, informalidade, previdência",
            "identification": "seleção em observáveis + desenho survey",
            "claim_ceiling": "diferença ajustada; não efeito causal forte",
        },
        {
            "estimand_id": "E2_ICA_ASSOC",
            "phase": "2",
            "question": "Associação entre intensidade de controle algorítmico e resultados laborais",
            "population": "trabalhadores de plataforma",
            "treatment": "Índice de Controle Algorítmico",
            "outcomes": "renda, horas, informalidade, TFD parcialmente identificado",
            "identification": "associação ajustada",
            "claim_ceiling": "intensificação compatível; causalidade condicionada",
        },
        {
            "estimand_id": "E3_TFD_BOUNDS",
            "phase": "2/4",
            "question": "Limites da carga de custos necessários e não compensados",
            "population": "entregadores sob cenários de custo plausíveis",
            "treatment": "não aplicável",
            "outcomes": "[TFD_L, TFD_U] e carga sobre renda",
            "identification": "partial identification + Monte Carlo",
            "claim_ceiling": "intervalo populacional/cenário; não contabilidade individual",
        },
        {
            "estimand_id": "E4_COST_PASS_THROUGH",
            "phase": "2",
            "question": "Quanto choques de combustível são compensados na remuneração",
            "population": "trabalhadores motorizados e comparadores",
            "treatment": "choque exógeno/semiexógeno no preço do combustível",
            "outcomes": "renda nominal/real e custo esperado",
            "identification": "painel agregado/repeated cross-section com controles e placebos",
            "claim_ceiling": "pass-through incompleto; não fórmula tarifária interna",
        },
        {
            "estimand_id": "E5_SYNTAX_DYNAMIC_VALUE",
            "phase": "3B",
            "question": "Contribuição incremental da Sintaxe Espacial à previsão urbana fora da amostra",
            "population": "sensores/corredores/períodos observados",
            "treatment": "ablação de NAIN/NACH e grafo angular",
            "outcomes": "erro e calibração de fluxo/velocidade/fricção",
            "identification": "experimento computacional preditivo com holdout espaço-temporal",
            "claim_ceiling": "valor informacional da configuração; não causalidade social",
        },
        {
            "estimand_id": "E6_ALGO_SET",
            "phase": "4",
            "question": "Conjunto de regimes de despacho compatíveis com momentos observados",
            "population": "população sintética calibrada",
            "treatment": "família de políticas de despacho",
            "outcomes": "momentos laborais, territoriais e distribuição do TFD",
            "identification": "SMM/indirect inference/ABC; set identification",
            "claim_ceiling": "propriedades compatíveis; não recuperação do algoritmo real",
        },
        {
            "estimand_id": "E7_POLICY_EX_ANTE",
            "phase": "5",
            "question": "Ganho de cobertura e equidade de uma rede de suporte",
            "population": "superfícies estimadas/simuladas por cenário",
            "treatment": "localização-alocação contrafactual",
            "outcomes": "distância, cobertura, P90, desigualdade e TFD modelado",
            "identification": "otimização ex ante",
            "claim_ceiling": "impacto modelado; não efeito realizado",
        },
    ]


def dag_files() -> dict[str, str]:
    return {
        "dag_platform_gap.dot": r'''digraph G {
  rankdir=LR;
  Platform [shape=box]; Outcome [shape=box];
  Occupation; Activity; Position; Region; Period; Age; Sex; Race; Education; Assets;
  Occupation -> Platform; Activity -> Platform; Position -> Platform; Region -> Platform;
  Age -> Platform; Sex -> Platform; Race -> Platform; Education -> Platform; Assets -> Platform;
  Occupation -> Outcome; Activity -> Outcome; Position -> Outcome; Region -> Outcome; Period -> Outcome;
  Age -> Outcome; Sex -> Outcome; Race -> Outcome; Education -> Outcome; Assets -> Outcome;
  Platform -> Outcome;
}''',
        "dag_tfd_mechanism.dot": r'''digraph G {
  rankdir=LR;
  StructuralInequality -> ResidentialLocation;
  StructuralInequality -> LaborPosition;
  ResidentialLocation -> SpatialAccess;
  UrbanConfiguration -> SpatialAccess;
  PlatformRegime -> AlgorithmicControl;
  PlatformRegime -> Dispatch;
  SpatialAccess -> Dispatch;
  Dispatch -> UnpaidTime;
  Dispatch -> EmptyDistance;
  Dispatch -> Risk;
  FuelPrice -> MonetaryCost;
  EmptyDistance -> MonetaryCost;
  UnpaidTime -> TFD;
  MonetaryCost -> TFD;
  Risk -> TFD;
  Compensation -> TFD [label="reduz"];
  AlgorithmicControl -> Dispatch;
}''',
        "dag_policy.dot": r'''digraph G {
  rankdir=LR;
  EstimatedExposure -> CandidateWeights;
  StructuralFriction -> CandidateWeights;
  Vulnerability -> EquityConstraint;
  Budget -> FacilityChoice;
  CandidateWeights -> FacilityChoice;
  EquityConstraint -> FacilityChoice;
  FacilityChoice -> AccessDistance;
  AccessDistance -> ModeledTFD;
  UrbanDynamics -> AccessDistance;
}''',
    }


# -----------------------------------------------------------------------------
# Cliente HTTP seguro e download versionado
# -----------------------------------------------------------------------------

class SafeHTTP:
    def __init__(self, logger: logging.Logger, timeout: int = 90, retries: int = 4):
        import requests
        from requests.adapters import HTTPAdapter
        from urllib3.util.retry import Retry

        self.logger = logger
        self.timeout = timeout
        self.session = requests.Session()
        retry = Retry(
            total=retries, connect=retries, read=retries,
            backoff_factor=1.5,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=frozenset(["HEAD", "GET"]),
            respect_retry_after_header=True,
        )
        self.session.mount("https://", HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=10))
        self.session.headers.update({
            "User-Agent": f"SPINE-GPEv7-Phase0/{SCRIPT_VERSION} academic-research",
            "Accept-Encoding": "gzip, deflate",
        })

    @staticmethod
    def validate_url(url: str) -> None:
        parsed = urllib.parse.urlparse(url)
        if parsed.scheme != "https":
            raise ValueError(f"Somente HTTPS é permitido: {url}")
        if parsed.hostname not in OFFICIAL_HTTPS_HOSTS:
            raise ValueError(f"Host fora da allowlist: {parsed.hostname}")

    def get(self, url: str, **kwargs: Any):
        self.validate_url(url)
        return self.session.get(url, timeout=kwargs.pop("timeout", self.timeout), **kwargs)

    def head(self, url: str, **kwargs: Any):
        self.validate_url(url)
        return self.session.head(url, timeout=kwargs.pop("timeout", self.timeout), allow_redirects=True, **kwargs)

    def download(self, url: str, destination: Path, source: SourceSpec,
                 run_id: str, phase: str = "0", overwrite: bool = False,
                 max_bytes: int | None = None) -> ArtifactRecord:
        self.validate_url(url)
        destination.parent.mkdir(parents=True, exist_ok=True)
        discovered = utc_now()
        try:
            with self.get(url, stream=True) as r:
                r.raise_for_status()
                content_type = (r.headers.get("Content-Type") or "").lower()
                content_length = int(r.headers.get("Content-Length") or 0)
                if max_bytes and content_length and content_length > max_bytes:
                    return ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="SKIPPED_SIZE_LIMIT",
                        phase=phase, agency=source.agency, dataset=source.dataset,
                        measurement_status=source.measurement_status, source_url=url,
                        file_name=destination.name, bytes=content_length, security="https",
                        license=source.license, discovered_at_utc=discovered,
                        notes=f"Limite: {max_bytes} bytes"
                    )
                if "text/html" in content_type and destination.suffix.lower() not in {".html", ".htm"}:
                    raise ValueError(f"Resposta HTML inesperada para {destination.name}")
                if destination.exists() and not overwrite:
                    return ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="EXISTS",
                        phase=phase, agency=source.agency, dataset=source.dataset,
                        measurement_status=source.measurement_status, source_url=url,
                        local_path=str(destination), file_name=destination.name,
                        bytes=destination.stat().st_size, sha256=sha256_file(destination),
                        etag=r.headers.get("ETag"), last_modified=r.headers.get("Last-Modified"),
                        security="https", license=source.license, discovered_at_utc=discovered,
                    )
                tmp = destination.with_suffix(destination.suffix + ".part")
                h = hashlib.sha256()
                total = 0
                with tmp.open("wb") as f:
                    for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                        if not chunk:
                            continue
                        total += len(chunk)
                        if max_bytes and total > max_bytes:
                            f.close()
                            tmp.unlink(missing_ok=True)
                            return ArtifactRecord(
                                run_id=run_id, source_id=source.source_id, status="SKIPPED_SIZE_LIMIT",
                                phase=phase, agency=source.agency, dataset=source.dataset,
                                measurement_status=source.measurement_status, source_url=url,
                                file_name=destination.name, bytes=total, security="https",
                                license=source.license, discovered_at_utc=discovered,
                            )
                        h.update(chunk)
                        f.write(chunk)
                digest = h.hexdigest()
                if tmp.stat().st_size < 32:
                    tmp.unlink(missing_ok=True)
                    raise ValueError("Arquivo baixado é pequeno demais; possível resposta de erro")
                tmp.replace(destination)
                return ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DOWNLOADED",
                    phase=phase, agency=source.agency, dataset=source.dataset,
                    measurement_status=source.measurement_status, source_url=url,
                    local_path=str(destination), file_name=destination.name,
                    media_type=content_type, bytes=total, sha256=digest,
                    etag=r.headers.get("ETag"), last_modified=r.headers.get("Last-Modified"),
                    downloaded_at_utc=utc_now(), discovered_at_utc=discovered,
                    security="https", license=source.license,
                )
        except Exception as exc:
            self.logger.error("Falha no download %s: %s", url, exc)
            return ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="ERROR",
                phase=phase, agency=source.agency, dataset=source.dataset,
                measurement_status=source.measurement_status, source_url=url,
                local_path=str(destination), file_name=destination.name,
                security="https", license=source.license, discovered_at_utc=discovered,
                error=f"{type(exc).__name__}: {exc}",
            )


# -----------------------------------------------------------------------------
# Descoberta HTTP/CKAN
# -----------------------------------------------------------------------------

def html_links(http: SafeHTTP, url: str) -> list[tuple[str, str]]:
    from bs4 import BeautifulSoup

    r = http.get(url)
    r.raise_for_status()
    soup = BeautifulSoup(r.content, "lxml")
    out: list[tuple[str, str]] = []
    for a in soup.find_all("a", href=True):
        href = urllib.parse.urljoin(url, a.get("href"))
        text = " ".join(a.get_text(" ", strip=True).split())
        if urllib.parse.urlparse(href).scheme == "https":
            out.append((text, href))
    return out


def select_links(links: Sequence[tuple[str, str]], pattern: str,
                 extensions: Sequence[str] | None = None) -> list[tuple[str, str]]:
    rx = re.compile(pattern, flags=re.I)
    out = []
    for text, href in links:
        blob = f"{text} {href}"
        path = urllib.parse.urlparse(href).path.lower()
        if rx.search(blob) and (not extensions or any(path.endswith("." + e.lower()) or f".{e.lower()}/" in path for e in extensions)):
            out.append((text, href))
    return out


def ckan_action(http: SafeHTTP, action: str, params: Mapping[str, Any]) -> dict[str, Any]:
    url = f"https://dados.recife.pe.gov.br/api/3/action/{action}"
    r = http.get(url, params=params)
    r.raise_for_status()
    data = r.json()
    if not data.get("success"):
        raise RuntimeError(f"CKAN retornou success=false em {action}")
    return data["result"]


def resolve_ckan_package(http: SafeHTTP, item: Mapping[str, Any]) -> dict[str, Any] | None:
    if item.get("slug"):
        try:
            return ckan_action(http, "package_show", {"id": item["slug"]})
        except Exception:
            pass
    query = item.get("query") or item.get("slug")
    if not query:
        return None
    result = ckan_action(http, "package_search", {"q": query, "rows": 5})
    candidates = result.get("results", [])
    return candidates[0] if candidates else None


def choose_ckan_resources(package: Mapping[str, Any], mode: str) -> list[dict[str, Any]]:
    allowed = {"CSV", "JSON", "GEOJSON", "ZIP", "SHP", "GPKG", "XLS", "XLSX", "PDF", "KML"}
    resources = []
    for res in package.get("resources", []):
        fmt = str(res.get("format") or "").upper().strip()
        url = str(res.get("url") or "")
        name = str(res.get("name") or "")
        if fmt in allowed and url.startswith("https://"):
            if mode == "core":
                # No core: dicionários, geometrias, recursos recentes e anos 2022–2024.
                if any(token in name.lower() for token in ["dicion", "2022", "2023", "2024", "equipamento", "ativa", "bares"]):
                    resources.append(dict(res))
                elif len(package.get("resources", [])) <= 5:
                    resources.append(dict(res))
            else:
                resources.append(dict(res))
    # Deduplicação por URL.
    seen = set()
    dedup = []
    for r in resources:
        if r["url"] not in seen:
            seen.add(r["url"])
            dedup.append(r)
    return dedup


# -----------------------------------------------------------------------------
# FTP oficial MTE — somente opt-in
# -----------------------------------------------------------------------------

def ftp_connect() -> ftplib.FTP:
    ftp = ftplib.FTP("ftp.mtps.gov.br", timeout=90)
    ftp.login()
    ftp.set_pasv(True)
    return ftp


def ftp_nlst_safe(ftp: ftplib.FTP, path: str) -> list[str]:
    try:
        return ftp.nlst(path)
    except ftplib.error_perm:
        return []


def ftp_download_file(ftp: ftplib.FTP, remote_path: str, destination: Path,
                      source: SourceSpec, run_id: str, logger: logging.Logger) -> ArtifactRecord:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        return ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="EXISTS", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=f"ftp://ftp.mtps.gov.br{remote_path}", local_path=str(destination),
            file_name=destination.name, bytes=destination.stat().st_size,
            sha256=sha256_file(destination), security="official_plaintext_ftp_opt_in",
            discovered_at_utc=utc_now(), notes="Transporte sem criptografia; integridade local por SHA-256"
        )
    tmp = destination.with_suffix(destination.suffix + ".part")
    h = hashlib.sha256()
    total = 0
    try:
        with tmp.open("wb") as f:
            def callback(chunk: bytes) -> None:
                nonlocal total
                total += len(chunk)
                h.update(chunk)
                f.write(chunk)
            ftp.retrbinary(f"RETR {remote_path}", callback, blocksize=8 * 1024 * 1024)
        tmp.replace(destination)
        return ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="DOWNLOADED", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=f"ftp://ftp.mtps.gov.br{remote_path}", local_path=str(destination),
            file_name=destination.name, bytes=total, sha256=h.hexdigest(),
            security="official_plaintext_ftp_opt_in", downloaded_at_utc=utc_now(),
            discovered_at_utc=utc_now(), notes="FTP oficial MTE sem criptografia; SHA-256 calculado após download"
        )
    except Exception as exc:
        tmp.unlink(missing_ok=True)
        logger.error("FTP falhou em %s: %s", remote_path, exc)
        return ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=f"ftp://ftp.mtps.gov.br{remote_path}", local_path=str(destination),
            file_name=destination.name, security="official_plaintext_ftp_opt_in",
            error=f"{type(exc).__name__}: {exc}", discovered_at_utc=utc_now(),
        )


def download_mte_rais(source: SourceSpec, dest: Path, run_id: str,
                      logger: logging.Logger, years: Sequence[int]) -> list[ArtifactRecord]:
    records: list[ArtifactRecord] = []
    ftp = ftp_connect()
    try:
        for year in years:
            base = f"/pdet/microdados/RAIS/{year}"
            names = ftp_nlst_safe(ftp, base)
            matches = [n for n in names if re.search(r"NORDESTE.*\.(7z|zip)$", n, re.I)]
            if not matches:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                    agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                    source_url=f"ftp://ftp.mtps.gov.br{base}/", security=source.security,
                    discovered_at_utc=utc_now(), notes="Nenhum arquivo Nordeste encontrado"
                ))
                continue
            for remote in matches:
                name = sanitize_filename(Path(remote).name)
                records.append(ftp_download_file(ftp, remote, dest / str(year) / name, source, run_id, logger))
    finally:
        try:
            ftp.quit()
        except Exception:
            ftp.close()
    return records


def download_mte_novo_caged(source: SourceSpec, dest: Path, run_id: str,
                            logger: logging.Logger, years: Sequence[int]) -> list[ArtifactRecord]:
    """Baixa somente arquivos CAGEDMOV, evitando FOR/EXC salvo mudança explícita."""
    records: list[ArtifactRecord] = []
    ftp = ftp_connect()
    try:
        root_candidates = ["/pdet/microdados/NOVO CAGED", "/pdet/microdados/NOVO%20CAGED"]
        root = None
        for candidate in root_candidates:
            if ftp_nlst_safe(ftp, candidate):
                root = candidate
                break
        if root is None:
            return [ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security=source.security,
                discovered_at_utc=utc_now(), error="Diretório NOVO CAGED não encontrado"
            )]
        for year in years:
            year_dir = f"{root}/{year}"
            month_dirs = ftp_nlst_safe(ftp, year_dir)
            for month_dir in month_dirs:
                names = ftp_nlst_safe(ftp, month_dir)
                matches = [n for n in names if re.search(r"CAGEDMOV.*\.(7z|zip)$", n, re.I)]
                for remote in matches:
                    month = re.search(r"20\d{4}", remote)
                    sub = month.group(0) if month else str(year)
                    records.append(ftp_download_file(
                        ftp, remote, dest / str(year) / sub / sanitize_filename(Path(remote).name),
                        source, run_id, logger
                    ))
    finally:
        try:
            ftp.quit()
        except Exception:
            ftp.close()
    return records


# -----------------------------------------------------------------------------
# Extração, layout PNAD e auditoria de arquivos
# -----------------------------------------------------------------------------

def extract_archive(path: Path, destination: Path, logger: logging.Logger,
                    member_pattern: str | None = None) -> list[Path]:
    destination.mkdir(parents=True, exist_ok=True)
    extracted: list[Path] = []
    rx = re.compile(member_pattern, re.I) if member_pattern else None
    suffix = path.suffix.lower()
    try:
        if suffix == ".zip":
            with zipfile.ZipFile(path) as zf:
                for info in zf.infolist():
                    if info.is_dir():
                        continue
                    if rx and not rx.search(info.filename):
                        continue
                    target = destination / sanitize_filename(Path(info.filename).name)
                    with zf.open(info) as src, target.open("wb") as dst:
                        shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
                    extracted.append(target)
        elif suffix == ".7z":
            import py7zr
            with py7zr.SevenZipFile(path, mode="r") as archive:
                names = archive.getnames()
                selected = [n for n in names if not rx or rx.search(n)]
                archive.extract(path=destination, targets=selected or None)
            extracted = [p for p in destination.rglob("*") if p.is_file()]
        else:
            logger.warning("Formato de arquivo não extraível automaticamente: %s", path)
    except Exception as exc:
        logger.error("Falha ao extrair %s: %s", path, exc)
    return extracted


def detect_encoding(path: Path, sample_bytes: int = 2_000_000) -> str:
    from charset_normalizer import from_bytes
    with path.open("rb") as f:
        raw = f.read(sample_bytes)
    match = from_bytes(raw).best()
    return match.encoding if match and match.encoding else "utf-8"


def audit_fixed_width(path: Path, source_id: str, max_lines: int = 20_000) -> AuditResult:
    lengths: Counter[int] = Counter()
    sample = 0
    with path.open("rb") as f:
        for line in f:
            lengths[len(line.rstrip(b"\r\n"))] += 1
            sample += 1
            if sample >= max_lines:
                break
    mode = lengths.most_common(1)[0][0] if lengths else None
    result = AuditResult(
        source_id=source_id, path=str(path), kind="fixed_width_text",
        size_bytes=path.stat().st_size, sha256=sha256_file(path),
        sample_rows=sample, estimated_rows=estimate_line_count(path),
        line_length_min=min(lengths) if lengths else None,
        line_length_max=max(lengths) if lengths else None,
        line_length_mode=mode,
    )
    if len(lengths) > 5:
        result.warnings.append(f"Muitos comprimentos de linha no sample: {dict(lengths.most_common(10))}")
    return result


def sniff_delimiter(sample: str) -> str | None:
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=";,\t|")
        return dialect.delimiter
    except csv.Error:
        return None


def audit_delimited(path: Path, source_id: str, sample_rows: int = 50_000) -> AuditResult:
    import pandas as pd

    encoding = detect_encoding(path)
    with path.open("r", encoding=encoding, errors="replace") as f:
        sample_text = f.read(100_000)
    delim = sniff_delimiter(sample_text) or ";"
    warnings: list[str] = []
    try:
        df = pd.read_csv(path, sep=delim, encoding=encoding, nrows=sample_rows, low_memory=False)
        columns = [str(c) for c in df.columns]
    except Exception as exc:
        columns = []
        warnings.append(f"Falha ao ler amostra: {type(exc).__name__}: {exc}")
    return AuditResult(
        source_id=source_id, path=str(path), kind="delimited_text",
        size_bytes=path.stat().st_size, sha256=sha256_file(path), encoding=encoding,
        delimiter=delim, sample_rows=sample_rows, estimated_rows=estimate_line_count(path),
        columns=columns, warnings=warnings,
    )


def audit_generic(path: Path, source_id: str) -> AuditResult:
    ext = path.suffix.lower()
    if ext == ".txt":
        # PNADc é fixed-width; demais TXT tendem a ser delimitados.
        if "PNADC_" in path.name.upper() or "PNAD_COVID" in path.name.upper():
            return audit_fixed_width(path, source_id)
        return audit_delimited(path, source_id)
    if ext in {".csv", ".tsv"}:
        return audit_delimited(path, source_id)
    return AuditResult(
        source_id=source_id, path=str(path), kind=ext.lstrip(".") or "binary",
        size_bytes=path.stat().st_size, sha256=sha256_file(path),
    )


def parse_sas_input_layout(text: str) -> list[dict[str, Any]]:
    """Extrai layout de instruções SAS do tipo @posição variável $largura. ou largura."""
    rows = []
    rx = re.compile(
        r"@(?P<start>\d+)\s+(?P<name>[A-Za-z_][A-Za-z0-9_]*)\s+(?P<char>\$)?(?P<width>\d+)(?:\.\d+)?",
        flags=re.I,
    )
    for line in text.splitlines():
        m = rx.search(line)
        if not m:
            continue
        start = int(m.group("start"))
        width = int(m.group("width"))
        rows.append({
            "variable": m.group("name"), "start_1based": start,
            "end_1based": start + width - 1, "width": width,
            "type": "string" if m.group("char") else "numeric",
        })
    return rows


def discover_layout_files(root: Path) -> list[Path]:
    patterns = ["*.sas", "*input*.txt", "*INPUT*.txt", "*layout*.txt", "*dicion*.xls*", "*Dicion*.xls*"]
    out: list[Path] = []
    for pattern in patterns:
        out.extend(root.rglob(pattern))
    return sorted(set(out))


def extract_variable_names_from_docs(paths: Sequence[Path]) -> set[str]:
    names: set[str] = set()
    for path in paths:
        try:
            if path.suffix.lower() in {".txt", ".sas"}:
                enc = detect_encoding(path)
                text = path.read_text(encoding=enc, errors="replace")
                for row in parse_sas_input_layout(text):
                    names.add(row["variable"])
                names.update(re.findall(r"\b(?:S14\d{4}|SD14\d{3}|V\d{4,5}|VD\d{4}|C\d{3}[A-Z]?)\b", text))
            elif path.suffix.lower() in {".xls", ".xlsx"}:
                import pandas as pd
                book = pd.ExcelFile(path)
                for sheet in book.sheet_names:
                    df = pd.read_excel(path, sheet_name=sheet, header=None, nrows=5000)
                    for value in df.astype(str).to_numpy().ravel():
                        names.update(re.findall(r"\b(?:S14\d{4}|SD14\d{3}|V\d{4,5}|VD\d{4}|C\d{3}[A-Z]?)\b", value))
        except Exception:
            continue
    return names


# -----------------------------------------------------------------------------
# Golden tests e fail-closed gates
# -----------------------------------------------------------------------------

def run_local_pnadc_gates(local_paths: Mapping[str, Path], audits: list[AuditResult]) -> list[GateResult]:
    gates: list[GateResult] = []
    for source_id, path in local_paths.items():
        if not path.exists():
            gates.append(GateResult(
                gate_id=f"{source_id}.exists", status="FAIL", severity="critical",
                source_id=source_id,
                message="Arquivo local PNADc obrigatório não encontrado.",
                evidence={"expected_path": str(path)},
            ))
            continue
        size = path.stat().st_size
        gates.append(GateResult(
            gate_id=f"{source_id}.exists", status="PASS", severity="critical",
            source_id=source_id, message="Arquivo local encontrado.",
            evidence={"path": str(path), "bytes": size},
        ))
        gates.append(GateResult(
            gate_id=f"{source_id}.size", status="PASS" if size > 10_000_000 else "FAIL",
            severity="critical", source_id=source_id,
            message="Tamanho plausível para microdado PNADc." if size > 10_000_000 else "Arquivo pequeno demais.",
            evidence={"bytes": size},
        ))
        audit = next((a for a in audits if a.source_id == source_id), None)
        if audit:
            stable = audit.line_length_mode is not None and audit.line_length_min == audit.line_length_max
            gates.append(GateResult(
                gate_id=f"{source_id}.fixed_width_stability",
                status="PASS" if stable else "WARN", severity="high", source_id=source_id,
                message="Comprimento fixed-width estável na amostra." if stable else "Variação de comprimento detectada; investigar CR/LF ou corrupção.",
                evidence={
                    "min": audit.line_length_min, "max": audit.line_length_max,
                    "mode": audit.line_length_mode, "sample_rows": audit.sample_rows,
                },
            ))
    return gates


def run_semantic_gates(variable_names: set[str], docs_found: bool) -> list[GateResult]:
    gates: list[GateResult] = []
    if not docs_found:
        gates.append(GateResult(
            gate_id="pnadc.docs.available", status="BLOCKED", severity="critical",
            source_id="pnadc_documentation",
            message="Documentação/layout oficial ainda não foi materializado; modelos ficam bloqueados.",
        ))
        return gates
    missing_direct = sorted(PNADC_REQUIRED_DIRECT - variable_names)
    missing_core = sorted(PNADC_REQUIRED_CORE - variable_names)
    gates.append(GateResult(
        gate_id="pnadc.layout.core_variables",
        status="PASS" if not missing_core else "FAIL", severity="critical",
        source_id="pnadc_documentation",
        message="Variáveis centrais presentes no layout." if not missing_core else "Variáveis centrais ausentes no layout detectado.",
        evidence={"missing": missing_core, "detected_count": len(variable_names)},
    ))
    gates.append(GateResult(
        gate_id="pnadc.layout.direct_platform",
        status="PASS" if "S140093" in variable_names else "FAIL", severity="critical",
        source_id="pnadc_documentation",
        message="S140093 validada no layout oficial." if "S140093" in variable_names else "S140093 não encontrada: PROIBIDO usar proxy como substituição silenciosa.",
        evidence={"missing_direct": missing_direct},
    ))
    gates.append(GateResult(
        gate_id="pnadc.no_silent_proxy_fallback", status="PASS", severity="critical",
        source_id="pnadc_direct",
        message="Política fail-closed registrada: ausência de S140093 bloqueia comparações diretas.",
    ))
    return gates


def global_fail_closed_gates() -> list[GateResult]:
    return [
        GateResult("global.no_synthetic_geography_as_observed", "PASS", "critical", "global",
                   "Geografia sintética nunca será tratada como localização observada."),
        GateResult("global.no_treatment_specific_outcome", "PASS", "critical", "global",
                   "É proibido subtrair custos somente do grupo tratado antes da comparação."),
        GateResult("global.no_random_labels_for_stgnn", "PASS", "critical", "global",
                   "STGNN só pode usar targets temporais observados; séries pseudoaleatórias são bloqueadas."),
        GateResult("global.no_causal_without_overlap", "PASS", "critical", "global",
                   "Estimadores causais/duplamente robustos exigem tratamento direto, controles e suporte."),
        GateResult("global.observed_estimated_simulated_separation", "PASS", "critical", "global",
                   "Todo artefato deve registrar observado/proxy/imputado/simulado."),
        GateResult("global.outcome_universe_separation", "PASS", "high", "global",
                   "Universo identificado e amostra válida de outcome serão preservados separadamente."),
    ]


# -----------------------------------------------------------------------------
# Downloaders por fonte
# -----------------------------------------------------------------------------

def download_ibge_directory(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                            mode: str, logger: logging.Logger) -> list[ArtifactRecord]:
    records: list[ArtifactRecord] = []
    assert source.discovery_url
    try:
        links = html_links(http, source.discovery_url)
        if source.source_id == "pnadc_official_2022q4_archive":
            chosen = select_links(links, r"PNADC_042022.*\.zip", ["zip"])
        elif source.source_id == "pnadc_official_2024q3_archive":
            chosen = select_links(links, r"PNADC_032024.*\.zip", ["zip"])
        else:
            chosen = [x for x in links if re.search(r"\.(zip|xlsx?|pdf|txt|sas)$", urllib.parse.urlparse(x[1]).path, re.I)]
        if mode == "inventory":
            for text, url in chosen:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                    agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                    source_url=url, file_name=sanitize_filename(Path(urllib.parse.urlparse(url).path).name or text),
                    security="https", license=source.license, discovered_at_utc=utc_now(),
                ))
            return records
        # Arquivos PNADc brutos são enormes e já existem localmente: core baixa docs, full baixa arquivos oficiais.
        if source.source_id.startswith("pnadc_official_") and mode != "full":
            for text, url in chosen:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DISCOVERED_NOT_DOWNLOADED_LOCAL_BASE_EXISTS",
                    phase="0", agency=source.agency, dataset=source.dataset,
                    measurement_status=source.measurement_status, source_url=url,
                    file_name=sanitize_filename(Path(urllib.parse.urlparse(url).path).name or text),
                    security="https", discovered_at_utc=utc_now(),
                    notes="Microdado local é a base; archive oficial será baixado apenas em mode=full."
                ))
            return records
        for text, url in chosen:
            name = sanitize_filename(Path(urllib.parse.urlparse(url).path).name or text)
            records.append(http.download(url, dest / name, source, run_id))
        if not chosen:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                error="Nenhum link compatível descoberto"
            ))
    except Exception as exc:
        logger.error("Descoberta IBGE falhou para %s: %s", source.source_id, exc)
        records.append(ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
            error=f"{type(exc).__name__}: {exc}"
        ))
    return records


def download_pnadc_historical(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                              mode: str, logger: logging.Logger) -> list[ArtifactRecord]:
    """Descobre/baixa os quatro trimestres PNADc por ano sem hardcode de sufixo de revisão."""
    records: list[ArtifactRecord] = []
    for year in source.years:
        year_url = urllib.parse.urljoin(source.discovery_url.rstrip("/") + "/", f"{year}/")
        try:
            links = html_links(http, year_url)
            chosen = select_links(links, rf"PNADC_0[1-4]{year}.*\.zip", ["zip"])
            for text, url in chosen:
                name = sanitize_filename(Path(urllib.parse.urlparse(url).path).name or text)
                if mode == "inventory" or mode == "core":
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                        agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                        source_url=url, file_name=name, security="https", discovered_at_utc=utc_now(),
                        notes="Download efetivo reservado a mode=full para controlar armazenamento."
                    ))
                else:
                    records.append(http.download(url, dest / str(year) / name, source, run_id))
            if not chosen:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                    agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                    source_url=year_url, security="https", discovered_at_utc=utc_now(),
                    notes=f"Nenhum trimestre localizado para {year}"
                ))
        except Exception as exc:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=year_url, security="https", discovered_at_utc=utc_now(),
                error=f"{type(exc).__name__}: {exc}"
            ))
    return records


def discover_ibge_sidra_tables(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                               mode: str, logger: logging.Logger) -> tuple[list[ArtifactRecord], dict[str, Any]]:
    """Registra IDs de tabelas SIDRA linkadas pela página oficial sem fazer consulta irrestrita gigante."""
    records: list[ArtifactRecord] = []
    meta: dict[str, Any] = {"source_page": source.discovery_url, "discovered_at_utc": utc_now(), "tables": []}
    try:
        r = http.get(source.discovery_url)
        r.raise_for_status()
        html = r.text
        ids = sorted(set(re.findall(r"sidra\.ibge\.gov\.br/(?:tabela|Tabela)/(\d+)", html, flags=re.I)))
        # Alguns links são redirecionados/embutidos com apenas /tabela/ID.
        ids += [x for x in sorted(set(re.findall(r"(?:/tabela/|Tabela/)(\d+)", html, flags=re.I))) if x not in ids]
        meta["tables"] = [{"table_id": x, "url": f"https://sidra.ibge.gov.br/tabela/{x}"} for x in ids]
        dest.mkdir(parents=True, exist_ok=True)
        html_path = dest / "pnadc_platform_2024_official_page.html"
        if mode != "inventory":
            html_path.write_text(html, encoding="utf-8")
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="DOWNLOADED", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, local_path=str(html_path), file_name=html_path.name,
                bytes=html_path.stat().st_size, sha256=sha256_file(html_path), security="https",
                downloaded_at_utc=utc_now(), discovered_at_utc=utc_now(),
                notes=f"IDs SIDRA descobertos: {len(ids)}"
            ))
        else:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                notes=f"IDs SIDRA descobertos: {len(ids)}"
            ))
        if not ids:
            logger.warning("Nenhum ID SIDRA foi extraído da página oficial; a página pode carregar links por JavaScript.")
    except Exception as exc:
        records.append(ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
            error=f"{type(exc).__name__}: {exc}"
        ))
    return records, meta


def download_pnad_covid(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                        mode: str, logger: logging.Logger) -> list[ArtifactRecord]:
    records: list[ArtifactRecord] = []
    base = source.discovery_url.rstrip("/") + "/"
    dirs = {
        "dados": urllib.parse.urljoin(base, "Dados/"),
        "docs": urllib.parse.urljoin(base, "Documentacao/"),
    }
    for kind, url in dirs.items():
        try:
            links = html_links(http, url)
            if kind == "dados":
                pattern = r"PNAD_COVID_(092020|052020|062020|072020|082020|102020|112020)\.zip"
                chosen = select_links(links, pattern, ["zip"])
                if mode == "core":
                    chosen = [x for x in chosen if "092020" in x[1]]
            else:
                chosen = [x for x in links if re.search(r"\.(zip|xlsx?|pdf|txt)$", urllib.parse.urlparse(x[1]).path, re.I)]
            for text, href in chosen:
                name = sanitize_filename(Path(urllib.parse.urlparse(href).path).name or text)
                if mode == "inventory":
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                        agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                        source_url=href, file_name=name, security="https", discovered_at_utc=utc_now()
                    ))
                else:
                    records.append(http.download(href, dest / kind / name, source, run_id))
        except Exception as exc:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=url, security="https", discovered_at_utc=utc_now(), error=f"{type(exc).__name__}: {exc}"
            ))
    return records


def download_recife_ckan(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                         mode: str, logger: logging.Logger, core_max_mb: int) -> tuple[list[ArtifactRecord], list[dict[str, Any]]]:
    records: list[ArtifactRecord] = []
    packages_meta: list[dict[str, Any]] = []
    for alias, item in RECIFE_CKAN_PACKAGES.items():
        try:
            pkg = resolve_ckan_package(http, item)
            if not pkg:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=f"recife_{alias}", status="NOT_FOUND", phase="0",
                    agency=source.agency, dataset=alias, measurement_status=source.measurement_status,
                    source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                    notes=f"required={item.get('required', False)}"
                ))
                continue
            packages_meta.append({
                "alias": alias, "id": pkg.get("id"), "name": pkg.get("name"), "title": pkg.get("title"),
                "metadata_modified": pkg.get("metadata_modified"), "license_title": pkg.get("license_title"),
                "num_resources": len(pkg.get("resources", [])), "url": pkg.get("url"),
            })
            resources = choose_ckan_resources(pkg, mode)
            if mode == "inventory":
                resources = [dict(r) for r in pkg.get("resources", [])]
            for res in resources:
                url = str(res.get("url") or "")
                if not url.startswith("https://"):
                    continue
                raw_name = res.get("name") or Path(urllib.parse.urlparse(url).path).name or res.get("id")
                ext = str(res.get("format") or "bin").lower()
                name = sanitize_filename(str(raw_name))
                if "." not in Path(name).name:
                    name += f".{ext}"
                subdir = dest / alias
                if mode == "inventory":
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=f"recife_{alias}", status="DISCOVERED", phase="0",
                        agency=source.agency, dataset=str(pkg.get("title") or alias),
                        measurement_status=source.measurement_status, source_url=url,
                        file_name=name, media_type=str(res.get("mimetype") or ext), security="https",
                        license=str(pkg.get("license_title") or source.license), discovered_at_utc=utc_now(),
                    ))
                else:
                    max_bytes = core_max_mb * 1024 * 1024 if mode == "core" else None
                    spec = dataclasses.replace(
                        source, source_id=f"recife_{alias}", dataset=str(pkg.get("title") or alias),
                        license=str(pkg.get("license_title") or source.license)
                    )
                    records.append(http.download(url, subdir / name, spec, run_id, max_bytes=max_bytes))
        except Exception as exc:
            logger.error("CKAN %s falhou: %s", alias, exc)
            records.append(ArtifactRecord(
                run_id=run_id, source_id=f"recife_{alias}", status="ERROR", phase="0",
                agency=source.agency, dataset=alias, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                error=f"{type(exc).__name__}: {exc}", notes=f"required={item.get('required', False)}"
            ))
    return records, packages_meta


def download_inmet(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                    mode: str, logger: logging.Logger) -> list[ArtifactRecord]:
    records: list[ArtifactRecord] = []
    try:
        links = html_links(http, source.discovery_url)
        year_rx = re.compile(r"(20\d{2})")
        candidates = []
        for text, url in links:
            match = year_rx.search(text + " " + url)
            if match and (url.lower().endswith(".zip") or ".zip" in url.lower()):
                year = int(match.group(1))
                if year in source.years:
                    candidates.append((year, text, url))
        if not candidates:
            # Padrão público histórico do portal INMET; o download ainda passa por validação HTTPS/allowlist.
            candidates = [
                (year, f"INMET {year}", f"https://portal.inmet.gov.br/uploads/dadoshistoricos/{year}.zip")
                for year in source.years
            ]
        if mode == "core":
            candidates = [x for x in candidates if x[0] >= 2020]
        for year, text, url in sorted(set(candidates)):
            name = sanitize_filename(Path(urllib.parse.urlparse(url).path).name or f"INMET_{year}.zip")
            if mode == "inventory":
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                    agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                    source_url=url, file_name=name, security="https", discovered_at_utc=utc_now(),
                ))
            else:
                rec = http.download(url, dest / "archives" / name, source, run_id)
                records.append(rec)
                if rec.status in {"DOWNLOADED", "EXISTS"} and rec.local_path:
                    extracted = extract_archive(Path(rec.local_path), dest / "A301", logger, r"(A301|RECIFE)")
                    for file in extracted:
                        records.append(ArtifactRecord(
                            run_id=run_id, source_id=source.source_id, status="EXTRACTED", phase="0",
                            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                            source_url=url, local_path=str(file), file_name=file.name,
                            bytes=file.stat().st_size, sha256=sha256_file(file), security="https",
                            downloaded_at_utc=utc_now(), discovered_at_utc=utc_now(),
                            notes="Subconjunto estação A301/Recife extraído do arquivo anual"
                        ))
        if not candidates:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                error="Links ZIP anuais não detectados; revisar HTML do portal INMET"
            ))
    except Exception as exc:
        records.append(ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
            error=f"{type(exc).__name__}: {exc}"
        ))
    return records


def download_anp(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                 mode: str, logger: logging.Logger) -> list[ArtifactRecord]:
    records: list[ArtifactRecord] = []
    try:
        links = html_links(http, source.discovery_url)
        candidates: list[tuple[int, str, str]] = []
        for text, url in links:
            blob = text + " " + url
            year_m = re.search(r"(20\d{2})", blob)
            if not year_m:
                continue
            year = int(year_m.group(1))
            if year not in source.years:
                continue
            if re.search(r"(csv|combust|automotiv|semestre)", blob, re.I):
                candidates.append((year, text, url))
        # Em core, limitar aos anos diretamente usados no choque e módulos.
        if mode == "core":
            candidates = [x for x in candidates if x[0] >= 2020]
        seen = set()
        for year, text, url in candidates:
            if url in seen:
                continue
            seen.add(url)
            name = sanitize_filename(Path(urllib.parse.urlparse(url).path).name or f"ANP_{year}.csv")
            if mode == "inventory":
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                    agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                    source_url=url, file_name=name, security="https", discovered_at_utc=utc_now(),
                ))
            else:
                records.append(http.download(url, dest / str(year) / name, source, run_id))
        if not candidates:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                error="Links de dados não detectados automaticamente; metadado permanece registrado"
            ))
    except Exception as exc:
        records.append(ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
            error=f"{type(exc).__name__}: {exc}"
        ))
    return records


def download_bcb_ipca(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                      mode: str) -> list[ArtifactRecord]:
    if mode == "inventory":
        return [ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, file_name="bcb_sgs_433_ipca.json",
            security="https", discovered_at_utc=utc_now(),
        )]
    return [http.download(source.discovery_url, dest / "bcb_sgs_433_ipca.json", source, run_id)]


def download_osm(source: SourceSpec, dest: Path, run_id: str,
                 logger: logging.Logger) -> list[ArtifactRecord]:
    records: list[ArtifactRecord] = []
    try:
        import osmnx as ox
        ox.settings.use_cache = True
        ox.settings.cache_folder = str(dest / "cache")
        ox.settings.requests_timeout = 180
        for network_type in ["drive", "bike", "walk"]:
            logger.info("Baixando OSM Recife (%s)...", network_type)
            graph = ox.graph_from_place("Recife, Pernambuco, Brazil", network_type=network_type, simplify=False)
            graphml = dest / f"recife_{network_type}_unsimplified.graphml"
            gpkg = dest / f"recife_{network_type}_unsimplified.gpkg"
            ox.save_graphml(graph, graphml)
            ox.save_graph_geopackage(graph, gpkg, directed=True)
            for path in [graphml, gpkg]:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DOWNLOADED", phase="0",
                    agency=source.agency, dataset=f"{source.dataset} — {network_type}",
                    measurement_status=source.measurement_status, source_url=source.discovery_url,
                    local_path=str(path), file_name=path.name, bytes=path.stat().st_size,
                    sha256=sha256_file(path), security="https via OSMnx/Nominatim/Overpass",
                    license=source.license, downloaded_at_utc=utc_now(), discovered_at_utc=utc_now(),
                ))
    except Exception as exc:
        records.append(ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
            error=f"{type(exc).__name__}: {exc}",
        ))
    return records


# -----------------------------------------------------------------------------
# Inventário local, manifests e relatórios
# -----------------------------------------------------------------------------

def register_local_reference(source: SourceSpec, run_id: str, refs_dir: Path) -> tuple[ArtifactRecord, AuditResult | None]:
    path = Path(source.local_path or "")
    if not path.exists():
        rec = ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="MISSING_LOCAL_REQUIRED", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            local_path=str(path), file_name=path.name, security="local", discovered_at_utc=utc_now(),
            error="Arquivo local não encontrado"
        )
        return rec, None
    audit = audit_fixed_width(path, source.source_id)
    pointer = {
        "source_id": source.source_id,
        "absolute_path": str(path.resolve()),
        "bytes": path.stat().st_size,
        "sha256": audit.sha256,
        "estimated_rows": audit.estimated_rows,
        "registered_at_utc": utc_now(),
        "copy_policy": "reference_only_no_duplication",
    }
    write_json(refs_dir / f"{source.source_id}.link.json", pointer)
    rec = ArtifactRecord(
        run_id=run_id, source_id=source.source_id, status="REGISTERED_LOCAL", phase="0",
        agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
        local_path=str(path.resolve()), file_name=path.name, bytes=path.stat().st_size,
        sha256=audit.sha256, security="local", discovered_at_utc=utc_now(),
        notes="Arquivo referenciado sem duplicação para preservar espaço"
    )
    return rec, audit


def manifest_to_csv(records: Sequence[ArtifactRecord], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = [f.name for f in dataclasses.fields(ArtifactRecord)]
    with path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for record in records:
            writer.writerow(asdict(record))


def gates_to_csv(gates: Sequence[GateResult], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=["gate_id", "status", "severity", "source_id", "message", "evidence"])
        writer.writeheader()
        for gate in gates:
            row = asdict(gate)
            row["evidence"] = json.dumps(row["evidence"], ensure_ascii=False)
            writer.writerow(row)


def write_environment_lock(path: Path, run_id: str, root: Path, root_mode: str) -> None:
    packages = {}
    for pkg in ["requests", "beautifulsoup4", "lxml", "pandas", "pyarrow", "polars", "py7zr", "openpyxl", "pyyaml", "osmnx", "geopandas"]:
        try:
            packages[pkg] = importlib.metadata.version(pkg)
        except importlib.metadata.PackageNotFoundError:
            packages[pkg] = None
    data = {
        "run_id": run_id,
        "script_version": SCRIPT_VERSION,
        "schema_version": SCHEMA_VERSION,
        "created_at_utc": utc_now(),
        "root": str(root),
        "root_resolution": root_mode,
        "python": sys.version,
        "platform": platform.platform(),
        "hostname": socket.gethostname(),
        "packages": packages,
        "git_commit": _git_commit(),
    }
    write_json(path, data)


def _git_commit() -> str | None:
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL, text=True).strip()
    except Exception:
        return None


def initial_decisions(run_id: str) -> list[dict[str, Any]]:
    return [
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D001", "decision": "S140093 é o identificador direto primário de entrega por plataforma.", "rationale": "Evita proxy silenciosa.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D002", "decision": "PNAD COVID C007C=17 é ocupação de entrega, não uso direto de plataforma.", "rationale": "Limite semântico oficial.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D003", "decision": "RAIS/PNAD COVID/PNADc não serão raw-pooled como painel causal.", "rationale": "Universos e desenhos incompatíveis.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D004", "decision": "Custos do TFD serão outcomes/cenários separados e nunca subtraídos somente do tratado antes da estimação.", "rationale": "Evita inserir mecanicamente a penalidade.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D005", "decision": "Pedidos, despacho, espera e retorno vazio sem logs serão sempre rotulados como simulados.", "rationale": "Separação observado/estimado/simulado.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D006", "decision": "STGNN usará targets urbanos observados e validação espaço-temporal; random labels são proibidos.", "rationale": "Validade preditiva.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D007", "decision": "Fase 4 identifica conjunto de regimes compatíveis; não recupera o algoritmo proprietário real.", "rationale": "Identificação parcial e equifinalidade.", "status": "locked"},
    ]


def generate_report(root: Path, run_id: str, mode: str, records: Sequence[ArtifactRecord],
                    audits: Sequence[AuditResult], gates: Sequence[GateResult], root_mode: str) -> str:
    status_counts = Counter(r.status for r in records)
    gate_counts = Counter(g.status for g in gates)
    required_failures = [g for g in gates if g.status in {"FAIL", "BLOCKED"} and g.severity == "critical"]
    lines = [
        "# SPINE-GPE v7 — Relatório da Fase 0",
        "",
        f"- **Run ID:** `{run_id}`",
        f"- **Versão:** `{SCRIPT_VERSION}`",
        f"- **Modo:** `{mode}`",
        f"- **Diretório:** `{root}`",
        f"- **Resolução do diretório:** `{root_mode}`",
        f"- **Gerado em:** `{utc_now()}`",
        "",
        "## Resultado executivo",
        "",
    ]
    if required_failures:
        lines.append("**LOCK: BLOQUEADO.** Há gates críticos não atendidos; nenhuma fase inferencial deve prosseguir.")
    else:
        lines.append("**LOCK: LIBERADO PARA A PRÓXIMA ETAPA**, condicionado aos limites registrados no Claim/Evidence Book.")
    lines += [
        "",
        "## Artefatos por status",
        "",
        "| Status | N |",
        "|---|---:|",
    ]
    for status, n in sorted(status_counts.items()):
        lines.append(f"| {status} | {n} |")
    lines += ["", "## Gates", "", "| Status | N |", "|---|---:|"]
    for status, n in sorted(gate_counts.items()):
        lines.append(f"| {status} | {n} |")
    lines += ["", "## Gates críticos não atendidos", ""]
    if not required_failures:
        lines.append("Nenhum.")
    else:
        for g in required_failures:
            lines.append(f"- **{g.gate_id}** — {g.message}")
    lines += [
        "",
        "## Arquivos locais auditados",
        "",
        "| Fonte | Caminho | GiB | SHA-256 | Linhas estimadas | Comprimento modal |",
        "|---|---|---:|---|---:|---:|",
    ]
    for a in audits:
        gib = a.size_bytes / 1024**3
        lines.append(f"| {a.source_id} | `{a.path}` | {gib:.3f} | `{a.sha256[:16]}…` | {a.estimated_rows or ''} | {a.line_length_mode or ''} |")
    lines += [
        "",
        "## Regras epistemológicas congeladas",
        "",
        "1. Observado, proxy, imputado e simulado permanecem separados em todos os manifests.",
        "2. Ausência de `S140093` bloqueia identificação direta; não ativa proxy automaticamente.",
        "3. PNAD COVID mede ocupação de entrega no choque pandêmico, não plataforma confirmada.",
        "4. RAIS mede formalidade registrada e não representa o universo informal.",
        "5. CTTU mede tráfego veicular geral, não demanda observada de entregadores.",
        "6. NAIN/NACH representam configuração; não são fluxo ou demanda observados.",
        "7. Outputs da Fase 4 são contrafactuais dentro do modelo e do conjunto compatível.",
        "",
        "## Próximo artefato esperado",
        "",
        "Executar o parser dos layouts oficiais PNADc e produzir a primeira `certified_table` com golden tests de prevalência, desenho survey e identificação direta.",
    ]
    return "\n".join(lines) + "\n"


# -----------------------------------------------------------------------------
# Orquestrador
# -----------------------------------------------------------------------------

def _sanitize_notebook_argv(argv: Sequence[str] | None = None) -> list[str]:
    """Remove apenas argumentos internos do kernel Jupyter/Colab.

    Ao executar um arquivo com ``%run``, ``exec`` ou ``runpy`` dentro de um
    notebook, o ipykernel pode acrescentar ``-f <kernel-....json>`` a
    ``sys.argv``. Esse argumento não pertence ao pipeline e fazia o argparse
    encerrar a execução com SystemExit(2). Argumentos desconhecidos reais
    continuam sendo rejeitados pelo argparse.
    """
    raw = list(sys.argv[1:] if argv is None else argv)
    in_notebook = (
        "ipykernel" in sys.modules
        or "google.colab" in sys.modules
        or os.environ.get("COLAB_RELEASE_TAG") is not None
    )
    if not in_notebook:
        return raw

    cleaned: list[str] = []
    i = 0
    while i < len(raw):
        token = raw[i]
        if token == "-f" and i + 1 < len(raw):
            candidate = raw[i + 1]
            if candidate.endswith(".json") and ("kernel-" in candidate or "jupyter" in candidate):
                i += 2
                continue
        if token.startswith("-f="):
            candidate = token.split("=", 1)[1]
            if candidate.endswith(".json") and ("kernel-" in candidate or "jupyter" in candidate):
                i += 1
                continue
        cleaned.append(token)
        i += 1
    return cleaned


def parse_args(argv: Sequence[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="SPINE-GPE v7 — Fase 0")
    parser.add_argument("--root", help="Diretório raiz; padrão D:\\aCidadeAlgoritmica\\SPINE-GPEv7 no Windows")
    parser.add_argument("--mode", choices=["inventory", "core", "full"], default="core")
    parser.add_argument("--bootstrap", action="store_true", help="Instala dependências mínimas")
    parser.add_argument("--allow-official-ftp", action="store_true", help="Autoriza FTP oficial MTE sem criptografia")
    parser.add_argument("--download-osm", action="store_true", help="Baixa redes OSM Recife via OSMnx")
    parser.add_argument("--core-max-resource-mb", type=int, default=600, help="Limite por recurso CKAN em mode=core")
    parser.add_argument("--overwrite", action="store_true", help="Reservado para futuras versões; downloads existentes são preservados")
    parser.add_argument("--strict", action="store_true", help="Retorna exit code 2 quando gate crítico falha")
    return parser.parse_args(_sanitize_notebook_argv(argv))


def main(argv: Sequence[str] | None = None) -> int:
    args = parse_args(argv)
    root, root_mode = resolve_root(args.root)
    tree = create_project_tree(root)
    run_id = dt.datetime.now().strftime("%Y%m%dT%H%M%S") + "Z"
    logger = setup_logger(tree["logs"] / f"phase0_{run_id}.log")
    logger.info("SPINE-GPE v7 Fase 0 | run=%s | mode=%s | root=%s", run_id, args.mode, root)

    if args.bootstrap:
        bootstrap_packages(logger)

    # Imports externos validados após bootstrap.
    try:
        import requests  # noqa: F401
        import pandas  # noqa: F401
        import bs4  # noqa: F401
    except ImportError as exc:
        logger.error("Dependência ausente: %s. Execute com --bootstrap.", exc)
        return 1

    write_environment_lock(tree["admin"] / f"environment_lock_{run_id}.json", run_id, root, root_mode)
    local_paths = resolve_local_pnadc_paths()
    sources = build_source_registry(local_paths)
    write_json(tree["registry"] / "source_registry.json", [asdict(s) for s in sources])
    write_json(tree["registry"] / "semantic_dictionary.json", semantic_dictionary())
    write_json(tree["contracts"] / "data_contracts.json", data_contracts())
    write_json(tree["registry"] / "estimand_registry.json", estimand_registry())
    for name, content in dag_files().items():
        (tree["dags"] / name).write_text(content + "\n", encoding="utf-8")
    for decision in initial_decisions(run_id):
        append_jsonl(tree["decisions"] / "analysis_decisions.jsonl", decision)
    # Inicializa registros mutáveis sem inventar decisões futuras.
    for name in ["exclusions.jsonl", "data_losses.jsonl", "protocol_deviations.jsonl"]:
        (tree["decisions"] / name).touch(exist_ok=True)

    records: list[ArtifactRecord] = []
    audits: list[AuditResult] = []
    gates: list[GateResult] = global_fail_closed_gates()

    # 1) Arquivos locais fundamentais.
    for source in sources:
        if source.strategy == "local_reference":
            rec, audit = register_local_reference(source, run_id, tree["raw_local_refs"])
            records.append(rec)
            if audit:
                audits.append(audit)
    gates.extend(run_local_pnadc_gates(local_paths, audits))

    # 2) Descoberta/download público.
    http = SafeHTTP(logger)
    packages_meta: list[dict[str, Any]] = []
    for source in sources:
        try:
            logger.info("Fonte: %s | estratégia=%s", source.source_id, source.strategy)
            if source.strategy == "local_reference":
                continue
            if source.strategy in {"ibge_directory_regex", "ibge_directory_all"}:
                records.extend(download_ibge_directory(
                    http, source, tree["raw_ibge"] / source.source_id, run_id, args.mode, logger
                ))
            elif source.strategy == "pnadc_historical_quarters":
                records.extend(download_pnadc_historical(
                    http, source, tree["raw_ibge"] / "pnadc_regular_historical", run_id, args.mode, logger
                ))
            elif source.strategy == "ibge_product_sidra_links":
                recs, sidra_meta = discover_ibge_sidra_tables(
                    http, source, tree["raw_ibge"] / "official_benchmarks", run_id, args.mode, logger
                )
                records.extend(recs)
                write_json(tree["registry"] / "sidra_platform_tables.json", sidra_meta)
            elif source.strategy == "ibge_pnad_covid":
                records.extend(download_pnad_covid(
                    http, source, tree["raw_ibge"] / "pnad_covid", run_id, args.mode, logger
                ))
            elif source.strategy == "direct_https":
                if args.mode == "inventory":
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                        agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                        source_url=source.discovery_url, file_name=Path(urllib.parse.urlparse(source.discovery_url).path).name,
                        security="https", discovered_at_utc=utc_now(),
                    ))
                else:
                    name = sanitize_filename(Path(urllib.parse.urlparse(source.discovery_url).path).name)
                    records.append(http.download(source.discovery_url, tree["raw_ibge"] / "censo2022" / name, source, run_id))
            elif source.strategy == "recife_ckan_bundle":
                recs, meta = download_recife_ckan(
                    http, source, tree["raw_recife"], run_id, args.mode, logger, args.core_max_resource_mb
                )
                records.extend(recs)
                packages_meta.extend(meta)
            elif source.strategy == "inmet_annual_station":
                records.extend(download_inmet(http, source, tree["raw_inmet"], run_id, args.mode, logger))
            elif source.strategy == "anp_page_links":
                records.extend(download_anp(http, source, tree["raw_anp"], run_id, args.mode, logger))
            elif source.strategy == "bcb_api":
                records.extend(download_bcb_ipca(http, source, tree["raw_bcb"], run_id, args.mode))
            elif source.strategy in {"mte_ftp_rais_nordeste", "mte_ftp_novo_caged"}:
                if not args.allow_official_ftp:
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="BLOCKED_SECURITY_OPT_IN", phase="0",
                        agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                        source_url=source.discovery_url, security=source.security, discovered_at_utc=utc_now(),
                        notes="Use --allow-official-ftp para autorizar o FTP oficial MTE sem criptografia."
                    ))
                    gates.append(GateResult(
                        gate_id=f"{source.source_id}.transport_opt_in", status="BLOCKED", severity="high",
                        source_id=source.source_id,
                        message="Download automático bloqueado porque a fonte oficial oferece FTP sem criptografia.",
                    ))
                elif args.mode == "full":
                    if source.strategy == "mte_ftp_rais_nordeste":
                        records.extend(download_mte_rais(source, tree["raw_mte"] / "RAIS", run_id, logger, source.years))
                    else:
                        records.extend(download_mte_novo_caged(source, tree["raw_mte"] / "NOVO_CAGED", run_id, logger, source.years))
                else:
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="DISCOVERED_NOT_DOWNLOADED_MODE_FULL_REQUIRED",
                        phase="0", agency=source.agency, dataset=source.dataset,
                        measurement_status=source.measurement_status, source_url=source.discovery_url,
                        security=source.security, discovered_at_utc=utc_now(),
                        notes="Use --mode full --allow-official-ftp."
                    ))
            elif source.strategy == "osmnx_place":
                if args.download_osm and args.mode == "full":
                    records.extend(download_osm(source, tree["raw_osm"], run_id, logger))
                else:
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="DISCOVERED_NOT_DOWNLOADED",
                        phase="0", agency=source.agency, dataset=source.dataset,
                        measurement_status=source.measurement_status, source_url=source.discovery_url,
                        security="https", license=source.license, discovered_at_utc=utc_now(),
                        notes="Use --mode full --download-osm."
                    ))
        except Exception as exc:
            logger.error("Erro não tratado em %s: %s", source.source_id, exc)
            logger.debug(traceback.format_exc())
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, local_path=source.local_path,
                security=source.security, discovered_at_utc=utc_now(),
                error=f"{type(exc).__name__}: {exc}",
            ))

    write_json(tree["registry"] / "recife_ckan_packages_resolved.json", packages_meta)

    # 3) Extração de documentação baixada e resolução de nomes de variáveis.
    doc_dir = tree["raw_ibge"] / "pnadc_documentation"
    for archive in doc_dir.rglob("*.zip"):
        extract_archive(archive, tree["interim"] / "pnadc_documentation_extracted" / archive.stem, logger)
    doc_paths = discover_layout_files(tree["raw_ibge"]) + discover_layout_files(tree["interim"])
    variable_names = extract_variable_names_from_docs(doc_paths)
    write_json(tree["registry"] / "pnadc_detected_variables.json", {
        "schema_version": SCHEMA_VERSION,
        "detected_at_utc": utc_now(),
        "documentation_files": [str(p) for p in doc_paths],
        "variables": sorted(variable_names),
    })
    gates.extend(run_semantic_gates(variable_names, docs_found=bool(doc_paths)))

    # 4) Auditoria leve de arquivos tabulares baixados (evita reprocessar arquivos gigantes).
    for rec in records:
        if rec.status not in {"DOWNLOADED", "EXISTS", "EXTRACTED"} or not rec.local_path:
            continue
        path = Path(rec.local_path)
        if not path.exists() or path.stat().st_size > 3 * 1024**3:
            continue
        if path.suffix.lower() in {".csv", ".tsv", ".txt"}:
            try:
                audits.append(audit_generic(path, rec.source_id))
            except Exception as exc:
                logger.warning("Auditoria tabular falhou em %s: %s", path, exc)

    # 5) Gates de disponibilidade CKAN essenciais.
    required_aliases = [k for k, v in RECIFE_CKAN_PACKAGES.items() if v.get("required")]
    for alias in required_aliases:
        matched = [r for r in records if r.source_id == f"recife_{alias}" and r.status not in {"ERROR", "NOT_FOUND"}]
        gates.append(GateResult(
            gate_id=f"recife.{alias}.available",
            status="PASS" if matched else "FAIL",
            severity="critical" if alias in {"velocidade_2022", "velocidade_2023", "velocidade_2024", "equipamentos_transito"} else "high",
            source_id=f"recife_{alias}",
            message="Pacote/recurso CKAN localizado." if matched else "Pacote/recurso CKAN obrigatório não localizado.",
            evidence={"records": len(matched)},
        ))

    # 6) Manifest, auditorias, gates e relatório.
    write_json(tree["manifests"] / f"artifacts_{run_id}.json", [asdict(r) for r in records])
    manifest_to_csv(records, tree["manifests"] / f"artifacts_{run_id}.csv")
    write_json(tree["reports"] / f"audits_{run_id}.json", [asdict(a) for a in audits])
    write_json(tree["reports"] / f"gates_{run_id}.json", [asdict(g) for g in gates])
    gates_to_csv(gates, tree["reports"] / f"gates_{run_id}.csv")

    report = generate_report(root, run_id, args.mode, records, audits, gates, root_mode)
    report_path = tree["reports"] / f"PHASE0_REPORT_{run_id}.md"
    report_path.write_text(report, encoding="utf-8")
    (tree["reports"] / "PHASE0_REPORT_LATEST.md").write_text(report, encoding="utf-8")

    # Lock final: qualquer FAIL/BLOCKED critical impede avanço.
    critical_fail = [g for g in gates if g.severity == "critical" and g.status in {"FAIL", "BLOCKED"}]
    lock = {
        "run_id": run_id,
        "script_version": SCRIPT_VERSION,
        "schema_version": SCHEMA_VERSION,
        "status": "BLOCKED" if critical_fail else "RELEASED",
        "critical_failures": [asdict(g) for g in critical_fail],
        "report": str(report_path),
        "created_at_utc": utc_now(),
    }
    write_json(tree["admin"] / "PHASE0_LOCK.json", lock)

    logger.info("Fase 0 concluída. LOCK=%s | relatório=%s", lock["status"], report_path)
    if critical_fail:
        for gate in critical_fail:
            logger.error("GATE CRÍTICO: %s — %s", gate.gate_id, gate.message)
    return 2 if args.strict and critical_fail else 0


if __name__ == "__main__":
    raise SystemExit(main())

2026-07-20 01:04:44,140 | INFO | SPINE-GPE v7 Fase 0 | run=20260720T010444Z | mode=core | root=/content/SPINE-GPEv7


INFO:spine_phase0:SPINE-GPE v7 Fase 0 | run=20260720T010444Z | mode=core | root=/content/SPINE-GPEv7


2026-07-20 01:04:46,275 | INFO | Fonte: pnadc_2022q4_direct_local | estratégia=local_reference


INFO:spine_phase0:Fonte: pnadc_2022q4_direct_local | estratégia=local_reference


2026-07-20 01:04:46,290 | INFO | Fonte: pnadc_2024q3_direct_local | estratégia=local_reference


...

ERROR:spine_phase0:GATE CRÍTICO: pnadc_2024q3_direct.exists — Arquivo local PNADc obrigatório não encontrado.


2026-07-20 01:38:01,831 | ERROR | GATE CRÍTICO: pnadc.layout.direct_platform — S140093 não encontrada: PROIBIDO usar proxy como substituição silenciosa.


ERROR:spine_phase0:GATE CRÍTICO: pnadc.layout.direct_platform — S140093 não encontrada: PROIBIDO usar proxy como substituição silenciosa.


SystemExit: 0

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
from pathlib import Path
import subprocess

ORIGEM = Path("/content/SPINE-GPEv7")
DESTINO = Path(
    "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7"
)

DESTINO.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        "rsync",
        "-a",
        "--ignore-existing",
        str(ORIGEM) + "/",
        str(DESTINO) + "/",
    ],
    check=False,
)

print("Cópia concluída:", DESTINO)

Cópia concluída: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
SPINE-GPE v7 — FASE 0: DATA, IDENTIFIABILITY & REPRODUCIBILITY LOCK
===================================================================

Objetivo
--------
Criar a fundação auditável e fail-closed da tese "A Cidade Algorítmica":

1. inventariar e versionar as fontes;
2. registrar hashes e proveniência;
3. baixar dados públicos apenas de domínios autorizados;
4. auditar os dois arquivos locais PNADc usados como base direta;
5. criar dicionário semântico, contratos, schema registry e golden tests;
6. auditar cobertura temporal/espacial;
7. registrar observado/proxy/imputado/simulado;
8. congelar DAGs, estimandos e limites de identificação;
9. impedir fallbacks silenciosos e leakage;
10. produzir um relatório de viabilidade para TODAS as fases da SPINE-GPE v7.

Execução recomendada
--------------------
Windows / Colab Local Runtime (acessa D:\\):
    python SPINE_GPEv7_FASE0.py --mode core
    python SPINE_GPEv7_FASE0.py --mode full --allow-official-ftp --download-osm

Google Colab hospedado (não acessa D:\\):
    O script monta/usa Google Drive quando disponível e grava em:
    /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7

Modos
-----
inventory : cria registry/contratos/DAGs, audita arquivos locais e descobre metadados.
core      : baixa fontes essenciais e de tamanho moderado.
full      : tenta baixar todo o acervo configurado; RAIS/CAGED exigem opt-in de FTP.

Segurança
---------
- HTTPS é obrigatório por padrão.
- O FTP oficial do MTE é texto claro; só é usado com --allow-official-ftp.
- Todos os arquivos recebem SHA-256 e registro de cabeçalhos/proveniência.
- Respostas HTML disfarçadas de dados são rejeitadas.
- Fallback de identificação direta para proxy é PROIBIDO.

Este script é a Fase 0. Ele NÃO executa ainda as estimativas finais das Fases 1–6.
"""

from __future__ import annotations

import argparse
import csv
import dataclasses
import datetime as dt
import ftplib
import hashlib
import importlib.metadata
import io
import json
import logging
import mimetypes
import os
import platform
import re
import shutil
import socket
import subprocess
import sys
import tempfile
import textwrap
import time
import traceback
import urllib.parse
import zipfile
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterable, Iterator, Mapping, Sequence

# Dependências externas são carregadas depois do bootstrap.

SCRIPT_VERSION = "7.0.2-phase0-colab-data-recovery"
SCHEMA_VERSION = "spine-gpe-v7-schema-1.0.0"
DEFAULT_WINDOWS_ROOT = Path(r"D:\aCidadeAlgoritmica\SPINE-GPEv7")
DEFAULT_COLAB_ROOT = Path("/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7")
DEFAULT_COLAB_EPHEMERAL_ROOT = Path("/content/SPINE-GPEv7")

LOCAL_PNADC_PATHS = {
    "pnadc_2022q4_direct": Path(r"D:\aCidadeAlgoritmica\PNADC_042022_20250815\PNADC_042022.txt"),
    "pnadc_2024q3_direct": Path(r"D:\aCidadeAlgoritmica\PNADC_032024_20250815\PNADC_032024.txt"),
}

OFFICIAL_HTTPS_HOSTS = {
    "ftp.ibge.gov.br",
    "www.ibge.gov.br",
    "servicodados.ibge.gov.br",
    "apisidra.ibge.gov.br",
    "dados.recife.pe.gov.br",
    "portal.inmet.gov.br",
    "www.gov.br",
    "api.bcb.gov.br",
    "download.geofabrik.de",
    "overpass-api.de",
    "nominatim.openstreetmap.org",
}

CORE_YEARS = list(range(2017, dt.datetime.now().year + 1))
PNADC_REQUIRED_DIRECT = {
    "UF", "UPA", "Estrato", "V1028", "V2007", "V2009", "V2010",
    "V4010", "V4012", "V4013", "V4039", "S140093", "SD14001",
}
PNADC_REQUIRED_CORE = {"UF", "UPA", "Estrato", "V1028", "V4010", "V4012", "V4013", "V4039"}
PNAD_COVID_REQUIRED = {"UF", "V1012", "V1032", "C001", "C007", "C007C", "C009", "C010"}

# -----------------------------------------------------------------------------
# Modelos de metadados
# -----------------------------------------------------------------------------

@dataclass(slots=True)
class SourceSpec:
    source_id: str
    phases: list[str]
    agency: str
    dataset: str
    source_class: str
    measurement_status: str
    strategy: str
    discovery_url: str | None = None
    local_path: str | None = None
    package_slug: str | None = None
    years: list[int] = field(default_factory=list)
    required: bool = True
    expected_formats: list[str] = field(default_factory=list)
    spatial_scope: str = ""
    temporal_scope: str = ""
    license: str = ""
    security: str = "https"
    claim_ceiling: str = "descriptive"
    notes: str = ""


@dataclass(slots=True)
class ArtifactRecord:
    run_id: str
    source_id: str
    status: str
    phase: str
    agency: str
    dataset: str
    measurement_status: str
    source_url: str | None = None
    local_path: str | None = None
    file_name: str | None = None
    media_type: str | None = None
    bytes: int | None = None
    sha256: str | None = None
    etag: str | None = None
    last_modified: str | None = None
    downloaded_at_utc: str | None = None
    discovered_at_utc: str | None = None
    security: str | None = None
    license: str | None = None
    error: str | None = None
    notes: str | None = None


@dataclass(slots=True)
class GateResult:
    gate_id: str
    status: str  # PASS/WARN/FAIL/BLOCKED/SKIP
    severity: str
    source_id: str
    message: str
    evidence: dict[str, Any] = field(default_factory=dict)


@dataclass(slots=True)
class AuditResult:
    source_id: str
    path: str
    kind: str
    size_bytes: int
    sha256: str
    encoding: str | None = None
    delimiter: str | None = None
    sample_rows: int | None = None
    estimated_rows: int | None = None
    columns: list[str] = field(default_factory=list)
    line_length_min: int | None = None
    line_length_max: int | None = None
    line_length_mode: int | None = None
    date_min: str | None = None
    date_max: str | None = None
    spatial_bounds: list[float] | None = None
    warnings: list[str] = field(default_factory=list)


# -----------------------------------------------------------------------------
# Logging e utilidades
# -----------------------------------------------------------------------------

def utc_now() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def json_default(obj: Any) -> Any:
    if isinstance(obj, Path):
        return str(obj)
    if dataclasses.is_dataclass(obj):
        return asdict(obj)
    if isinstance(obj, set):
        return sorted(obj)
    raise TypeError(f"Tipo não serializável: {type(obj)!r}")


def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(data, ensure_ascii=False, indent=2, default=json_default), encoding="utf-8")
    tmp.replace(path)


def append_jsonl(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(data, ensure_ascii=False, default=json_default) + "\n")


def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def estimate_line_count(path: Path, sample_bytes: int = 64 * 1024 * 1024) -> int | None:
    size = path.stat().st_size
    if size == 0:
        return 0
    read_bytes = min(size, sample_bytes)
    with path.open("rb") as f:
        data = f.read(read_bytes)
    n = data.count(b"\n")
    if n == 0:
        return None
    return int(round(n * size / read_bytes))


def sanitize_filename(name: str) -> str:
    name = urllib.parse.unquote(name).strip().replace("\\", "_").replace("/", "_")
    name = re.sub(r"[^0-9A-Za-zÀ-ÿ._()\- ]+", "_", name)
    name = re.sub(r"\s+", "_", name)
    return name[:220] or "download.bin"


def setup_logger(log_path: Path) -> logging.Logger:
    logger = logging.getLogger("spine_phase0")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(sh)
    logger.addHandler(fh)
    return logger


def is_colab() -> bool:
    return "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))


def resolve_root(cli_root: str | None) -> tuple[Path, str]:
    if cli_root:
        return Path(cli_root).expanduser(), "cli"
    env_root = os.environ.get("SPINE_GPE_ROOT")
    if env_root:
        return Path(env_root).expanduser(), "env"
    if os.name == "nt":
        return DEFAULT_WINDOWS_ROOT, "windows_default"
    if is_colab():
        if DEFAULT_COLAB_ROOT.parent.exists():
            return DEFAULT_COLAB_ROOT, "colab_drive_fallback"
        return DEFAULT_COLAB_EPHEMERAL_ROOT, "colab_ephemeral_fallback"
    return Path.cwd() / "SPINE-GPEv7", "portable_fallback"


def resolve_local_pnadc_paths(
    cli_2022: str | None = None,
    cli_2024: str | None = None,
    root: Path | None = None,
) -> dict[str, Path]:
    """Resolve PNADc paths across Windows, hosted Colab, Drive and project cache.

    Priority: explicit CLI > environment > original Windows path > known Colab/Drive
    locations > exact-name search inside the project root.
    """
    out = dict(LOCAL_PNADC_PATHS)
    explicit = {
        "pnadc_2022q4_direct": cli_2022 or os.environ.get("PNADC_2022Q4_PATH"),
        "pnadc_2024q3_direct": cli_2024 or os.environ.get("PNADC_2024Q3_PATH"),
    }
    for key, value in explicit.items():
        if value:
            out[key] = Path(value).expanduser()

    expected_names = {
        "pnadc_2022q4_direct": "PNADC_042022.txt",
        "pnadc_2024q3_direct": "PNADC_032024.txt",
    }
    candidate_bases = [
        Path("/content/drive/MyDrive/aCidadeAlgoritmica"),
        Path("/content/aCidadeAlgoritmica"),
        Path("/content"),
    ]
    if root is not None:
        candidate_bases.extend([root, root / "01_raw", root / "02_interim"])

    logical_dirs = {
        "pnadc_2022q4_direct": "PNADC_042022_20250815",
        "pnadc_2024q3_direct": "PNADC_032024_20250815",
    }
    for key, name in expected_names.items():
        if out[key].exists():
            continue
        direct_candidates = [base / logical_dirs[key] / name for base in candidate_bases]
        direct_candidates += [base / name for base in candidate_bases]
        found = next((c for c in direct_candidates if c.exists()), None)
        if found is None and root is not None and root.exists():
            try:
                found = next(root.rglob(name), None)
            except OSError:
                found = None
        if found is not None:
            out[key] = found
    return out


def create_project_tree(root: Path) -> dict[str, Path]:
    tree = {
        "root": root,
        "admin": root / "00_admin",
        "registry": root / "00_admin/registry",
        "contracts": root / "00_admin/contracts",
        "manifests": root / "00_admin/manifests",
        "logs": root / "00_admin/logs",
        "reports": root / "00_admin/reports",
        "dags": root / "00_admin/dags",
        "decisions": root / "00_admin/decisions",
        "raw": root / "01_raw",
        "raw_local_refs": root / "01_raw/00_local_references",
        "raw_ibge": root / "01_raw/10_ibge",
        "raw_mte": root / "01_raw/20_mte",
        "raw_recife": root / "01_raw/30_recife_ckan",
        "raw_inmet": root / "01_raw/40_inmet",
        "raw_anp": root / "01_raw/50_anp",
        "raw_bcb": root / "01_raw/60_bcb",
        "raw_osm": root / "01_raw/70_osm",
        "interim": root / "02_interim",
        "processed": root / "03_processed",
        "models": root / "04_models",
        "outputs": root / "05_outputs",
        "cache": root / "99_cache",
    }
    for p in tree.values():
        p.mkdir(parents=True, exist_ok=True)
    return tree


def bootstrap_packages(logger: logging.Logger) -> None:
    packages = [
        "requests>=2.31", "beautifulsoup4>=4.12", "lxml>=5.0",
        "pandas>=2.1", "pyarrow>=15", "polars>=0.20",
        "charset-normalizer>=3.3", "py7zr>=0.21", "openpyxl>=3.1",
        "pyyaml>=6.0", "tqdm>=4.66", "rich>=13.7",
        "xlrd>=2.0", "pypdf>=4.0",
        "osmnx>=1.9", "geopandas>=0.14",
    ]
    logger.info("Bootstrap de dependências Python...")
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    subprocess.run(cmd, check=True)


# -----------------------------------------------------------------------------
# Registry completo de fontes
# -----------------------------------------------------------------------------

def build_source_registry(local_paths: Mapping[str, Path]) -> list[SourceSpec]:
    current_year = dt.datetime.now().year
    specs: list[SourceSpec] = [
        SourceSpec(
            source_id="pnadc_2022q4_direct_local",
            phases=["0", "1", "2"], agency="IBGE", dataset="PNADc 2022T4 — microdado local",
            source_class="survey_microdata", measurement_status="observado_direto",
            strategy="local_reference", local_path=str(local_paths["pnadc_2022q4_direct"]),
            spatial_scope="Brasil/UF/RM conforme desenho", temporal_scope="2022T4",
            claim_ceiling="inferência survey; comparação ajustada; não causal sem hipóteses adicionais",
            notes="Base direta de plataforma; S140093 é obrigatório; fallback para proxy é proibido."
        ),
        SourceSpec(
            source_id="pnadc_2024q3_direct_local",
            phases=["0", "1", "2"], agency="IBGE", dataset="PNADc 2024T3 — microdado local",
            source_class="survey_microdata", measurement_status="observado_direto",
            strategy="local_reference", local_path=str(local_paths["pnadc_2024q3_direct"]),
            spatial_scope="Brasil/UF/RM conforme desenho", temporal_scope="2024T3",
            claim_ceiling="inferência survey; comparação ajustada; não causal sem hipóteses adicionais",
            notes="Base direta de plataforma; S140093 é obrigatório; comparação 2022–2024 sofre sazonalidade."
        ),
        SourceSpec(
            source_id="pnadc_official_2022q4_archive", phases=["0", "1", "2"], agency="IBGE",
            dataset="PNADc 2022T4 — arquivo oficial", source_class="survey_microdata",
            measurement_status="observado_direto", strategy="ibge_directory_regex",
            discovery_url="https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2022/",
            years=[2022], expected_formats=["zip"], temporal_scope="2022T4", spatial_scope="Brasil",
            claim_ceiling="proveniência e reprodução", notes=r"Regex: PNADC_042022.*\.zip"
        ),
        SourceSpec(
            source_id="pnadc_official_2024q3_archive", phases=["0", "1", "2"], agency="IBGE",
            dataset="PNADc 2024T3 — arquivo oficial", source_class="survey_microdata",
            measurement_status="observado_direto", strategy="ibge_directory_regex",
            discovery_url="https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2024/",
            years=[2024], expected_formats=["zip"], temporal_scope="2024T3", spatial_scope="Brasil",
            claim_ceiling="proveniência e reprodução", notes=r"Regex: PNADC_032024.*\.zip"
        ),
        SourceSpec(
            source_id="pnadc_documentation", phases=["0", "1", "2"], agency="IBGE",
            dataset="PNADc trimestral — documentação, inputs, dicionários e deflatores",
            source_class="official_metadata", measurement_status="observado_metadado",
            strategy="ibge_directory_all",
            discovery_url="https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/Documentacao/",
            expected_formats=["zip", "xls", "xlsx", "pdf", "txt", "sas"], spatial_scope="Brasil",
            temporal_scope="versão vigente", claim_ceiling="definição semântica oficial"
        ),
        SourceSpec(
            source_id="pnadc_regular_historical", phases=["0", "1", "2", "4"], agency="IBGE",
            dataset="PNADc trimestral regular — série histórica", source_class="survey_microdata",
            measurement_status="observado_survey_proxy_calibravel", strategy="pnadc_historical_quarters",
            discovery_url="https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/",
            years=list(range(2017, 2025)), expected_formats=["zip"], spatial_scope="Brasil/UF/RM conforme desenho",
            temporal_scope="2017T1–2024T4", claim_ceiling="reconstrução probabilística histórica; plataforma não direta fora dos módulos",
            notes="Mode=full baixa os quatro trimestres por ano; core apenas descobre o inventário remoto."
        ),
        SourceSpec(
            source_id="pnadc_platform_official_tables", phases=["0", "1"], agency="IBGE/SIDRA",
            dataset="Tabelas oficiais — trabalho por plataformas digitais 2024", source_class="official_benchmark",
            measurement_status="observado_agregado_oficial", strategy="ibge_product_sidra_links",
            discovery_url="https://www.ibge.gov.br/estatisticas/sociais/trabalho/17270-pnad-%20continua.html/17270-pnad-continua.html?edicao=44741",
            expected_formats=["html", "json"], spatial_scope="Brasil/Grandes Regiões/UF/RM conforme tabela",
            temporal_scope="2022T4 e 2024T3", claim_ceiling="golden tests e reprodução de totais oficiais"
        ),
        SourceSpec(
            source_id="pnad_covid_2020", phases=["0", "1", "2"], agency="IBGE",
            dataset="PNAD COVID19 — microdados mensais", source_class="survey_microdata",
            measurement_status="observado_ocupacional", strategy="ibge_pnad_covid",
            discovery_url="https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_PNAD_COVID19/Microdados/",
            years=[2020], expected_formats=["zip", "xls", "xlsx", "pdf"], spatial_scope="Brasil/UF",
            temporal_scope="2020-05 a 2020-11", claim_ceiling="ponte pandêmica ocupacional; não plataforma direta"
        ),
        SourceSpec(
            source_id="rais_nordeste_2017_latest", phases=["0", "1", "2", "4"], agency="MTE/PDET",
            dataset="RAIS vínculos públicos — Nordeste", source_class="administrative_microdata",
            measurement_status="observado_administrativo", strategy="mte_ftp_rais_nordeste",
            discovery_url="ftp://ftp.mtps.gov.br/pdet/microdados/RAIS/",
            years=list(range(2017, current_year)), expected_formats=["7z", "txt"], spatial_scope="Nordeste/município",
            temporal_scope=f"2017–{current_year-1}", security="official_plaintext_ftp_opt_in",
            claim_ceiling="baseline formal; registro administrativo; não representa informalidade"
        ),
        SourceSpec(
            source_id="novo_caged_2020_latest", phases=["0", "1", "2"], agency="MTE/PDET",
            dataset="Novo CAGED — movimentações", source_class="administrative_microdata",
            measurement_status="observado_administrativo", strategy="mte_ftp_novo_caged",
            discovery_url="ftp://ftp.mtps.gov.br/pdet/microdados/NOVO%20CAGED/",
            years=list(range(2020, current_year + 1)), expected_formats=["7z", "txt"], spatial_scope="Brasil/município",
            temporal_scope=f"2020–{current_year}", security="official_plaintext_ftp_opt_in",
            claim_ceiling="dinâmica mensal formal; não plataforma direta"
        ),
        SourceSpec(
            source_id="censo2022_pe_setores", phases=["0", "3A", "4", "5"], agency="IBGE",
            dataset="Censo 2022 — setores com atributos PE", source_class="census_geodata",
            measurement_status="observado_censitario", strategy="direct_https",
            discovery_url="https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/malha_com_atributos/setores/gpkg/UF/PE/PE_setores_CD2022.gpkg",
            expected_formats=["gpkg"], spatial_scope="Pernambuco/setor", temporal_scope="2022",
            claim_ceiling="estrutura territorial censitária; SAE/MRP model-based"
        ),
        SourceSpec(
            source_id="recife_ckan_bundle", phases=["0", "3A", "3B", "4", "5"], agency="Prefeitura do Recife",
            dataset="Pacote CKAN urbano Recife", source_class="municipal_open_data",
            measurement_status="observado_municipal", strategy="recife_ckan_bundle",
            discovery_url="https://dados.recife.pe.gov.br/api/3/action/", spatial_scope="Recife",
            temporal_scope="conforme recurso", license="ODbL/licença indicada no pacote",
            claim_ceiling="dinâmica urbana observada ou potencial locacional conforme conjunto"
        ),
        SourceSpec(
            source_id="inmet_recife", phases=["0", "3B", "4", "5"], agency="INMET",
            dataset="Dados meteorológicos históricos — estação Recife A301", source_class="weather_timeseries",
            measurement_status="observado_instrumental", strategy="inmet_annual_station",
            discovery_url="https://portal.inmet.gov.br/dadoshistoricos", years=CORE_YEARS,
            expected_formats=["zip", "csv"], spatial_scope="Recife/estação A301", temporal_scope=f"2017–{current_year}",
            claim_ceiling="controle meteorológico e choques observados"
        ),
        SourceSpec(
            source_id="anp_combustiveis", phases=["0", "2", "4", "5"], agency="ANP",
            dataset="Série histórica de preços de combustíveis", source_class="price_microdata",
            measurement_status="observado_mercado", strategy="anp_page_links",
            discovery_url="https://www.gov.br/anp/pt-br/assuntos/precos-e-defesa-da-concorrencia/precos/precos-revenda-e-de-distribuicao-combustiveis/serie-historica-do-levantamento-de-precos",
            years=CORE_YEARS, expected_formats=["csv", "xls", "xlsx", "ods"], spatial_scope="Município/posto",
            temporal_scope=f"2017–{current_year}", claim_ceiling="choques de custo e pass-through; não custo individual"
        ),
        SourceSpec(
            source_id="bcb_ipca", phases=["0", "1", "2", "4", "5"], agency="Banco Central do Brasil/IBGE",
            dataset="SGS 433 — IPCA mensal", source_class="price_index", measurement_status="observado_indice",
            strategy="bcb_api", discovery_url="https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json",
            expected_formats=["json"], spatial_scope="Brasil", temporal_scope="1980–atual",
            claim_ceiling="harmonização monetária"
        ),
        SourceSpec(
            source_id="osm_recife", phases=["0", "3A", "3B", "4", "5"], agency="OpenStreetMap",
            dataset="Rede viária multimodal Recife", source_class="volunteered_geodata",
            measurement_status="observado_cartografico", strategy="osmnx_place",
            discovery_url="https://www.openstreetmap.org/", expected_formats=["graphml", "gpkg"],
            spatial_scope="Recife", temporal_scope="snapshot da execução", license="ODbL",
            claim_ceiling="estrutura de rede; qualidade depende da cobertura OSM"
        ),
    ]
    return specs


RECIFE_CKAN_PACKAGES: dict[str, dict[str, Any]] = {
    "velocidade_2016": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2016", "required": False},
    "velocidade_2017": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2017", "required": False},
    "velocidade_2018": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media", "required": False},
    "velocidade_2019": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2019", "required": False},
    "velocidade_2020": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2020", "required": False},
    "velocidade_2021": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2021", "required": False},
    "velocidade_2022": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2022", "required": True},
    "velocidade_2023": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2023", "required": True},
    "velocidade_2024": {"slug": "velocidade-das-vias-quantitativo-por-velocidade-media-2024", "required": True},
    "equipamentos_transito": {"slug": "equipamentos-de-monitoramento-e-fiscalizacao-de-transito", "required": True},
    "sinistros_transito": {"slug": "acidentes-de-transito-com-e-sem-vitimas", "required": True},
    "empresas": {"slug": "empresas-da-cidade-do-recife", "required": True},
    "bares_restaurantes": {"slug": "bares-e-restaurantes", "required": True},
    "itbi": {"slug": "imposto-sobre-transmissao-de-bens-imoveis-itbi", "required": True},
    # Os itens abaixo usam busca CKAN se o slug mudar.
    "iluminacao": {"query": "iluminação pública", "required": False},
    "ciclovias": {"query": "malha cicloviária", "required": False},
    "salva_bike": {"query": "Salva Bike", "required": False},
    "mercados_publicos": {"query": "mercados públicos", "required": False},
    "parques_pracas": {"query": "parques praças", "required": False},
    "equipamentos_publicos": {"query": "equipamentos públicos", "required": False},
    "terminais": {"query": "terminais transporte", "required": False},
    "malha_viaria": {"query": "malha viária logradouros eixos viários", "required": False},
}


# -----------------------------------------------------------------------------
# Dicionário semântico, contratos, estimandos e DAGs
# -----------------------------------------------------------------------------

def semantic_dictionary() -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []

    def add(source: str, var: str, concept: str, role: str, status: str,
            allowed_use: str, forbidden_use: str = "", notes: str = "") -> None:
        rows.append({
            "schema_version": SCHEMA_VERSION, "source": source, "variable": var,
            "concept": concept, "role": role, "measurement_status": status,
            "allowed_use": allowed_use, "forbidden_use": forbidden_use, "notes": notes,
        })

    # PNADc direta e regular
    add("PNADc", "S140093", "uso de plataforma de entrega", "treatment/direct identifier",
        "observado_direto", "identificação direta no módulo especial",
        "não substituir silenciosamente por CBO/CNAE quando ausente")
    add("PNADc", "SD14001", "trabalho por plataforma em ao menos um tipo", "direct identifier",
        "observado_direto", "prevalência geral de plataforma", "não equivale especificamente a entrega")
    for v, c in [("S140091", "táxi"), ("S140092", "transporte particular de passageiros"),
                 ("S140094", "serviços gerais/profissionais")]:
        add("PNADc", v, f"uso de plataforma — {c}", "direct identifier", "observado_direto",
            "tipologia de plataforma", "não usar como entrega")
    add("PNADc", "V4010", "ocupação no trabalho principal", "covariate/occupation",
        "observado_survey", "ocupação e proxy histórica calibrada", "não é plataforma por si só")
    add("PNADc", "V4012", "posição na ocupação", "covariate/employment position",
        "observado_survey", "comparadores e formalidade", "código 4 não deve ser tratado como conta própria")
    add("PNADc", "V4013", "atividade do empreendimento", "covariate/activity",
        "observado_survey", "ocupação × atividade e proxy histórica", "não confundir CNAE classe 53202 com código domiciliar 53002")
    add("PNADc", "V4039", "horas habitualmente trabalhadas", "outcome/exposure",
        "observado_survey", "jornada", "não combinar mecanicamente com renda-hora no mesmo lado da regressão")
    add("PNADc", "VD4016", "rendimento habitual mensal do trabalho principal", "outcome",
        "observado_survey", "renda bruta principal", "não subtrair custos apenas do tratado antes da estimação")
    add("PNADc", "VD4020", "rendimento efetivo mensal de todos os trabalhos", "outcome",
        "observado_survey", "renda efetiva, após confirmação no dicionário", "não tratar como ocupação")
    add("PNADc", "V1028", "peso final", "survey design", "observado_survey",
        "estimação survey", "não usar como variável substantiva")
    add("PNADc", "UPA", "unidade primária de amostragem", "survey design", "observado_survey",
        "variância survey", "não ignorar em inferência")
    add("PNADc", "Estrato", "estrato amostral", "survey design", "observado_survey",
        "variância survey", "não ignorar em inferência")

    # PNAD COVID
    add("PNAD_COVID", "C007C", "ocupação em categorias especiais", "occupation identifier",
        "observado_ocupacional", "16=motoboy; 17=entrega de mercadorias",
        "não chamar diretamente de uso de aplicativo")
    add("PNAD_COVID", "C007", "posição na ocupação", "covariate", "observado_survey",
        "formalidade/conta própria", "não equiparar todos os demais códigos a formal")
    add("PNAD_COVID", "C001", "trabalhou na semana", "universe filter", "observado_survey",
        "sensibilidade trabalhador ativo", "não confundir com todo o universo ocupado")
    add("PNAD_COVID", "C014", "contribuição ao INSS", "social protection", "observado_survey",
        "proteção previdenciária", "não confundir com local de trabalho")

    # RAIS/CAGED
    add("RAIS", "CBO 2002", "ocupação formal", "occupation identifier", "observado_administrativo",
        "baseline formal", "não representa entregadores informais")
    add("RAIS", "Vl Remun Média Nom", "remuneração média nominal", "outcome", "observado_administrativo",
        "renda formal", "não comparar nominalmente entre anos sem deflator")
    add("RAIS", "Qtd Hora Contr", "horas contratuais", "outcome/exposure", "observado_administrativo",
        "jornada contratual", "não equivale a horas efetivas")

    # Espacial/temporal
    add("SINTAXE", "NAIN_r", "integração angular normalizada por raio", "spatial prior",
        "derivado_observado", "estrutura configuracional", "não chamar de demanda observada")
    add("SINTAXE", "NACH_r", "choice angular normalizada por raio", "spatial prior",
        "derivado_observado", "potencial de intermediação", "não chamar de fluxo real")
    add("CTTU", "fluxo_15min", "contagem veicular por equipamento e intervalo", "dynamic outcome",
        "observado_instrumental", "treino/validação do grafo dinâmico", "não equivale a fluxo de entregadores")
    add("SIM", "pedidos_potenciais", "pedidos gerados pelo emulador", "latent simulation",
        "simulado", "cenários e identificação de conjunto", "não reportar como pedidos observados")
    add("SIM", "TFD_contrafactual", "carga de custos sob política simulada", "counterfactual outcome",
        "simulado_parcialmente_identificado", "comparação de regimes", "não reportar como valor individual observado")
    return rows


def data_contracts() -> dict[str, Any]:
    return {
        "schema_version": SCHEMA_VERSION,
        "global": {
            "timezone": "America/Recife",
            "crs_geographic": "EPSG:4674",
            "crs_projected_recife": "EPSG:31985",
            "currency_nominal": "BRL",
            "currency_real_base": "IPCA, base configurável no pipeline",
            "missing_policy": "nunca converter missing em zero sem regra explícita",
            "fallback_policy": "fail_closed",
        },
        "pnadc_direct": {
            "required_variables": sorted(PNADC_REQUIRED_DIRECT),
            "treatment": "S140093",
            "survey_design": ["V1028", "UPA", "Estrato"],
            "forbidden_fallbacks": ["V4010 sozinho", "VD4019/VD4020 como ocupação", "proxy quando S140093 ausente"],
            "gates": [
                "S140093 presente no layout oficial",
                "tratados e controles positivos",
                "totais ponderados replicam tabelas oficiais dentro da tolerância",
                "amostra de outcome separada do universo identificado",
            ],
        },
        "pnad_covid": {
            "required_variables": sorted(PNAD_COVID_REQUIRED),
            "narrow_delivery": "C007C == 17",
            "broad_logistics": "C007C in {16,17}",
            "platform_direct": False,
            "claim_ceiling": "ocupação de entrega no choque pandêmico",
        },
        "rais": {
            "delimiter": ";",
            "expected_encoding_candidates": ["latin1", "cp1252", "utf-8"],
            "occupation_aliases": ["CBO Ocupação 2002", "CBO 2002 Ocupação", "CBO Ocupação 2002"],
            "income_aliases": ["Vl Remun Média Nom", "Vl Remun Média (SM)", "Vl Remun Dezembro Nom"],
            "hours_aliases": ["Qtd Hora Contr"],
            "geography_aliases": ["Mun Trab", "Município"],
            "claim_ceiling": "formalidade registrada",
        },
        "cttu_speed": {
            "time_resolution_expected_minutes": 15,
            "minimum_stable_sensors": 20,
            "minimum_continuous_months": 6,
            "maximum_missing_fraction": 0.30,
            "minimum_map_match_rate": 0.80,
            "claim_ceiling": "tráfego/velocidade observados; não demanda de plataforma",
        },
        "spatial": {
            "required_modes": ["drive", "bike", "walk"],
            "syntax_radii_m": [400, 800, 1200, 2000, 5000, "global"],
            "preserve_tags": ["bridge", "tunnel", "layer", "oneway", "access", "highway"],
            "hexagons_m": [250, 500],
            "aggregation": "length_weighted",
        },
    }


def estimand_registry() -> list[dict[str, Any]]:
    return [
        {
            "estimand_id": "E1_PLATFORM_GAP",
            "phase": "2",
            "question": "Diferença ajustada entre entregadores de plataforma e comparadores não-plataforma",
            "population": "suporte comum dentro da PNADc especial do mesmo trimestre",
            "treatment": "S140093",
            "outcomes": "renda bruta, jornada, renda-hora, informalidade, previdência",
            "identification": "seleção em observáveis + desenho survey",
            "claim_ceiling": "diferença ajustada; não efeito causal forte",
        },
        {
            "estimand_id": "E2_ICA_ASSOC",
            "phase": "2",
            "question": "Associação entre intensidade de controle algorítmico e resultados laborais",
            "population": "trabalhadores de plataforma",
            "treatment": "Índice de Controle Algorítmico",
            "outcomes": "renda, horas, informalidade, TFD parcialmente identificado",
            "identification": "associação ajustada",
            "claim_ceiling": "intensificação compatível; causalidade condicionada",
        },
        {
            "estimand_id": "E3_TFD_BOUNDS",
            "phase": "2/4",
            "question": "Limites da carga de custos necessários e não compensados",
            "population": "entregadores sob cenários de custo plausíveis",
            "treatment": "não aplicável",
            "outcomes": "[TFD_L, TFD_U] e carga sobre renda",
            "identification": "partial identification + Monte Carlo",
            "claim_ceiling": "intervalo populacional/cenário; não contabilidade individual",
        },
        {
            "estimand_id": "E4_COST_PASS_THROUGH",
            "phase": "2",
            "question": "Quanto choques de combustível são compensados na remuneração",
            "population": "trabalhadores motorizados e comparadores",
            "treatment": "choque exógeno/semiexógeno no preço do combustível",
            "outcomes": "renda nominal/real e custo esperado",
            "identification": "painel agregado/repeated cross-section com controles e placebos",
            "claim_ceiling": "pass-through incompleto; não fórmula tarifária interna",
        },
        {
            "estimand_id": "E5_SYNTAX_DYNAMIC_VALUE",
            "phase": "3B",
            "question": "Contribuição incremental da Sintaxe Espacial à previsão urbana fora da amostra",
            "population": "sensores/corredores/períodos observados",
            "treatment": "ablação de NAIN/NACH e grafo angular",
            "outcomes": "erro e calibração de fluxo/velocidade/fricção",
            "identification": "experimento computacional preditivo com holdout espaço-temporal",
            "claim_ceiling": "valor informacional da configuração; não causalidade social",
        },
        {
            "estimand_id": "E6_ALGO_SET",
            "phase": "4",
            "question": "Conjunto de regimes de despacho compatíveis com momentos observados",
            "population": "população sintética calibrada",
            "treatment": "família de políticas de despacho",
            "outcomes": "momentos laborais, territoriais e distribuição do TFD",
            "identification": "SMM/indirect inference/ABC; set identification",
            "claim_ceiling": "propriedades compatíveis; não recuperação do algoritmo real",
        },
        {
            "estimand_id": "E7_POLICY_EX_ANTE",
            "phase": "5",
            "question": "Ganho de cobertura e equidade de uma rede de suporte",
            "population": "superfícies estimadas/simuladas por cenário",
            "treatment": "localização-alocação contrafactual",
            "outcomes": "distância, cobertura, P90, desigualdade e TFD modelado",
            "identification": "otimização ex ante",
            "claim_ceiling": "impacto modelado; não efeito realizado",
        },
    ]


def dag_files() -> dict[str, str]:
    return {
        "dag_platform_gap.dot": r'''digraph G {
  rankdir=LR;
  Platform [shape=box]; Outcome [shape=box];
  Occupation; Activity; Position; Region; Period; Age; Sex; Race; Education; Assets;
  Occupation -> Platform; Activity -> Platform; Position -> Platform; Region -> Platform;
  Age -> Platform; Sex -> Platform; Race -> Platform; Education -> Platform; Assets -> Platform;
  Occupation -> Outcome; Activity -> Outcome; Position -> Outcome; Region -> Outcome; Period -> Outcome;
  Age -> Outcome; Sex -> Outcome; Race -> Outcome; Education -> Outcome; Assets -> Outcome;
  Platform -> Outcome;
}''',
        "dag_tfd_mechanism.dot": r'''digraph G {
  rankdir=LR;
  StructuralInequality -> ResidentialLocation;
  StructuralInequality -> LaborPosition;
  ResidentialLocation -> SpatialAccess;
  UrbanConfiguration -> SpatialAccess;
  PlatformRegime -> AlgorithmicControl;
  PlatformRegime -> Dispatch;
  SpatialAccess -> Dispatch;
  Dispatch -> UnpaidTime;
  Dispatch -> EmptyDistance;
  Dispatch -> Risk;
  FuelPrice -> MonetaryCost;
  EmptyDistance -> MonetaryCost;
  UnpaidTime -> TFD;
  MonetaryCost -> TFD;
  Risk -> TFD;
  Compensation -> TFD [label="reduz"];
  AlgorithmicControl -> Dispatch;
}''',
        "dag_policy.dot": r'''digraph G {
  rankdir=LR;
  EstimatedExposure -> CandidateWeights;
  StructuralFriction -> CandidateWeights;
  Vulnerability -> EquityConstraint;
  Budget -> FacilityChoice;
  CandidateWeights -> FacilityChoice;
  EquityConstraint -> FacilityChoice;
  FacilityChoice -> AccessDistance;
  AccessDistance -> ModeledTFD;
  UrbanDynamics -> AccessDistance;
}''',
    }


# -----------------------------------------------------------------------------
# Cliente HTTP seguro e download versionado
# -----------------------------------------------------------------------------

class SafeHTTP:
    def __init__(self, logger: logging.Logger, timeout: int = 90, retries: int = 4):
        import requests
        from requests.adapters import HTTPAdapter
        from urllib3.util.retry import Retry

        self.logger = logger
        self.timeout = timeout
        self.session = requests.Session()
        retry = Retry(
            total=retries, connect=retries, read=retries,
            backoff_factor=1.5,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=frozenset(["HEAD", "GET"]),
            respect_retry_after_header=True,
        )
        self.session.mount("https://", HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=10))
        self.session.headers.update({
            "User-Agent": f"SPINE-GPEv7-Phase0/{SCRIPT_VERSION} academic-research",
            "Accept-Encoding": "gzip, deflate",
        })

    @staticmethod
    def validate_url(url: str) -> None:
        parsed = urllib.parse.urlparse(url)
        if parsed.scheme != "https":
            raise ValueError(f"Somente HTTPS é permitido: {url}")
        if parsed.hostname not in OFFICIAL_HTTPS_HOSTS:
            raise ValueError(f"Host fora da allowlist: {parsed.hostname}")

    def get(self, url: str, **kwargs: Any):
        self.validate_url(url)
        return self.session.get(url, timeout=kwargs.pop("timeout", self.timeout), **kwargs)

    def head(self, url: str, **kwargs: Any):
        self.validate_url(url)
        return self.session.head(url, timeout=kwargs.pop("timeout", self.timeout), allow_redirects=True, **kwargs)

    def download(self, url: str, destination: Path, source: SourceSpec,
                 run_id: str, phase: str = "0", overwrite: bool = False,
                 max_bytes: int | None = None) -> ArtifactRecord:
        self.validate_url(url)
        destination.parent.mkdir(parents=True, exist_ok=True)
        discovered = utc_now()
        try:
            with self.get(url, stream=True) as r:
                r.raise_for_status()
                content_type = (r.headers.get("Content-Type") or "").lower()
                content_length = int(r.headers.get("Content-Length") or 0)
                if max_bytes and content_length and content_length > max_bytes:
                    return ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="SKIPPED_SIZE_LIMIT",
                        phase=phase, agency=source.agency, dataset=source.dataset,
                        measurement_status=source.measurement_status, source_url=url,
                        file_name=destination.name, bytes=content_length, security="https",
                        license=source.license, discovered_at_utc=discovered,
                        notes=f"Limite: {max_bytes} bytes"
                    )
                if "text/html" in content_type and destination.suffix.lower() not in {".html", ".htm"}:
                    raise ValueError(f"Resposta HTML inesperada para {destination.name}")
                if destination.exists() and not overwrite:
                    return ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="EXISTS",
                        phase=phase, agency=source.agency, dataset=source.dataset,
                        measurement_status=source.measurement_status, source_url=url,
                        local_path=str(destination), file_name=destination.name,
                        bytes=destination.stat().st_size, sha256=sha256_file(destination),
                        etag=r.headers.get("ETag"), last_modified=r.headers.get("Last-Modified"),
                        security="https", license=source.license, discovered_at_utc=discovered,
                    )
                tmp = destination.with_suffix(destination.suffix + ".part")
                h = hashlib.sha256()
                total = 0
                with tmp.open("wb") as f:
                    for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                        if not chunk:
                            continue
                        total += len(chunk)
                        if max_bytes and total > max_bytes:
                            f.close()
                            tmp.unlink(missing_ok=True)
                            return ArtifactRecord(
                                run_id=run_id, source_id=source.source_id, status="SKIPPED_SIZE_LIMIT",
                                phase=phase, agency=source.agency, dataset=source.dataset,
                                measurement_status=source.measurement_status, source_url=url,
                                file_name=destination.name, bytes=total, security="https",
                                license=source.license, discovered_at_utc=discovered,
                            )
                        h.update(chunk)
                        f.write(chunk)
                digest = h.hexdigest()
                if tmp.stat().st_size < 32:
                    tmp.unlink(missing_ok=True)
                    raise ValueError("Arquivo baixado é pequeno demais; possível resposta de erro")
                tmp.replace(destination)
                return ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DOWNLOADED",
                    phase=phase, agency=source.agency, dataset=source.dataset,
                    measurement_status=source.measurement_status, source_url=url,
                    local_path=str(destination), file_name=destination.name,
                    media_type=content_type, bytes=total, sha256=digest,
                    etag=r.headers.get("ETag"), last_modified=r.headers.get("Last-Modified"),
                    downloaded_at_utc=utc_now(), discovered_at_utc=discovered,
                    security="https", license=source.license,
                )
        except Exception as exc:
            self.logger.error("Falha no download %s: %s", url, exc)
            return ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="ERROR",
                phase=phase, agency=source.agency, dataset=source.dataset,
                measurement_status=source.measurement_status, source_url=url,
                local_path=str(destination), file_name=destination.name,
                security="https", license=source.license, discovered_at_utc=discovered,
                error=f"{type(exc).__name__}: {exc}",
            )


# -----------------------------------------------------------------------------
# Descoberta HTTP/CKAN
# -----------------------------------------------------------------------------

def html_links(http: SafeHTTP, url: str) -> list[tuple[str, str]]:
    from bs4 import BeautifulSoup

    r = http.get(url)
    r.raise_for_status()
    soup = BeautifulSoup(r.content, "lxml")
    out: list[tuple[str, str]] = []
    for a in soup.find_all("a", href=True):
        href = urllib.parse.urljoin(url, a.get("href"))
        text = " ".join(a.get_text(" ", strip=True).split())
        if urllib.parse.urlparse(href).scheme == "https":
            out.append((text, href))
    return out


def select_links(links: Sequence[tuple[str, str]], pattern: str,
                 extensions: Sequence[str] | None = None) -> list[tuple[str, str]]:
    rx = re.compile(pattern, flags=re.I)
    out = []
    for text, href in links:
        blob = f"{text} {href}"
        path = urllib.parse.urlparse(href).path.lower()
        if rx.search(blob) and (not extensions or any(path.endswith("." + e.lower()) or f".{e.lower()}/" in path for e in extensions)):
            out.append((text, href))
    return out


def ckan_action(http: SafeHTTP, action: str, params: Mapping[str, Any]) -> dict[str, Any]:
    url = f"https://dados.recife.pe.gov.br/api/3/action/{action}"
    r = http.get(url, params=params)
    r.raise_for_status()
    data = r.json()
    if not data.get("success"):
        raise RuntimeError(f"CKAN retornou success=false em {action}")
    return data["result"]


def resolve_ckan_package(http: SafeHTTP, item: Mapping[str, Any]) -> dict[str, Any] | None:
    if item.get("slug"):
        try:
            return ckan_action(http, "package_show", {"id": item["slug"]})
        except Exception:
            pass
    query = item.get("query") or item.get("slug")
    if not query:
        return None
    result = ckan_action(http, "package_search", {"q": query, "rows": 5})
    candidates = result.get("results", [])
    return candidates[0] if candidates else None


def choose_ckan_resources(package: Mapping[str, Any], mode: str) -> list[dict[str, Any]]:
    allowed = {"CSV", "JSON", "GEOJSON", "ZIP", "SHP", "GPKG", "XLS", "XLSX", "PDF", "KML"}
    resources = []
    for res in package.get("resources", []):
        fmt = str(res.get("format") or "").upper().strip()
        url = str(res.get("url") or "")
        name = str(res.get("name") or "")
        if fmt in allowed and url.startswith("https://"):
            if mode == "core":
                # No core: dicionários, geometrias, recursos recentes e anos 2022–2024.
                if any(token in name.lower() for token in ["dicion", "2022", "2023", "2024", "equipamento", "ativa", "bares"]):
                    resources.append(dict(res))
                elif len(package.get("resources", [])) <= 5:
                    resources.append(dict(res))
            else:
                resources.append(dict(res))
    # Deduplicação por URL.
    seen = set()
    dedup = []
    for r in resources:
        if r["url"] not in seen:
            seen.add(r["url"])
            dedup.append(r)
    return dedup


# -----------------------------------------------------------------------------
# FTP oficial MTE — somente opt-in
# -----------------------------------------------------------------------------

def ftp_connect() -> ftplib.FTP:
    ftp = ftplib.FTP("ftp.mtps.gov.br", timeout=90)
    ftp.login()
    ftp.set_pasv(True)
    return ftp


def ftp_nlst_safe(ftp: ftplib.FTP, path: str) -> list[str]:
    try:
        return ftp.nlst(path)
    except ftplib.error_perm:
        return []


def ftp_download_file(ftp: ftplib.FTP, remote_path: str, destination: Path,
                      source: SourceSpec, run_id: str, logger: logging.Logger) -> ArtifactRecord:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        return ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="EXISTS", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=f"ftp://ftp.mtps.gov.br{remote_path}", local_path=str(destination),
            file_name=destination.name, bytes=destination.stat().st_size,
            sha256=sha256_file(destination), security="official_plaintext_ftp_opt_in",
            discovered_at_utc=utc_now(), notes="Transporte sem criptografia; integridade local por SHA-256"
        )
    tmp = destination.with_suffix(destination.suffix + ".part")
    h = hashlib.sha256()
    total = 0
    try:
        with tmp.open("wb") as f:
            def callback(chunk: bytes) -> None:
                nonlocal total
                total += len(chunk)
                h.update(chunk)
                f.write(chunk)
            ftp.retrbinary(f"RETR {remote_path}", callback, blocksize=8 * 1024 * 1024)
        tmp.replace(destination)
        return ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="DOWNLOADED", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=f"ftp://ftp.mtps.gov.br{remote_path}", local_path=str(destination),
            file_name=destination.name, bytes=total, sha256=h.hexdigest(),
            security="official_plaintext_ftp_opt_in", downloaded_at_utc=utc_now(),
            discovered_at_utc=utc_now(), notes="FTP oficial MTE sem criptografia; SHA-256 calculado após download"
        )
    except Exception as exc:
        tmp.unlink(missing_ok=True)
        logger.error("FTP falhou em %s: %s", remote_path, exc)
        return ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=f"ftp://ftp.mtps.gov.br{remote_path}", local_path=str(destination),
            file_name=destination.name, security="official_plaintext_ftp_opt_in",
            error=f"{type(exc).__name__}: {exc}", discovered_at_utc=utc_now(),
        )


def download_mte_rais(source: SourceSpec, dest: Path, run_id: str,
                      logger: logging.Logger, years: Sequence[int]) -> list[ArtifactRecord]:
    records: list[ArtifactRecord] = []
    ftp = ftp_connect()
    try:
        for year in years:
            base = f"/pdet/microdados/RAIS/{year}"
            names = ftp_nlst_safe(ftp, base)
            matches = [n for n in names if re.search(r"NORDESTE.*\.(7z|zip)$", n, re.I)]
            if not matches:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                    agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                    source_url=f"ftp://ftp.mtps.gov.br{base}/", security=source.security,
                    discovered_at_utc=utc_now(), notes="Nenhum arquivo Nordeste encontrado"
                ))
                continue
            for remote in matches:
                name = sanitize_filename(Path(remote).name)
                records.append(ftp_download_file(ftp, remote, dest / str(year) / name, source, run_id, logger))
    finally:
        try:
            ftp.quit()
        except Exception:
            ftp.close()
    return records


def download_mte_novo_caged(source: SourceSpec, dest: Path, run_id: str,
                            logger: logging.Logger, years: Sequence[int]) -> list[ArtifactRecord]:
    """Baixa somente arquivos CAGEDMOV, evitando FOR/EXC salvo mudança explícita."""
    records: list[ArtifactRecord] = []
    ftp = ftp_connect()
    try:
        root_candidates = ["/pdet/microdados/NOVO CAGED", "/pdet/microdados/NOVO%20CAGED"]
        root = None
        for candidate in root_candidates:
            if ftp_nlst_safe(ftp, candidate):
                root = candidate
                break
        if root is None:
            return [ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security=source.security,
                discovered_at_utc=utc_now(), error="Diretório NOVO CAGED não encontrado"
            )]
        for year in years:
            year_dir = f"{root}/{year}"
            month_dirs = ftp_nlst_safe(ftp, year_dir)
            for month_dir in month_dirs:
                names = ftp_nlst_safe(ftp, month_dir)
                matches = [n for n in names if re.search(r"CAGEDMOV.*\.(7z|zip)$", n, re.I)]
                for remote in matches:
                    month = re.search(r"20\d{4}", remote)
                    sub = month.group(0) if month else str(year)
                    records.append(ftp_download_file(
                        ftp, remote, dest / str(year) / sub / sanitize_filename(Path(remote).name),
                        source, run_id, logger
                    ))
    finally:
        try:
            ftp.quit()
        except Exception:
            ftp.close()
    return records


# -----------------------------------------------------------------------------
# Extração, layout PNAD e auditoria de arquivos
# -----------------------------------------------------------------------------

def extract_archive(path: Path, destination: Path, logger: logging.Logger,
                    member_pattern: str | None = None) -> list[Path]:
    destination.mkdir(parents=True, exist_ok=True)
    extracted: list[Path] = []
    rx = re.compile(member_pattern, re.I) if member_pattern else None
    suffix = path.suffix.lower()
    try:
        if suffix == ".zip":
            with zipfile.ZipFile(path) as zf:
                for info in zf.infolist():
                    if info.is_dir():
                        continue
                    if rx and not rx.search(info.filename):
                        continue
                    target = destination / sanitize_filename(Path(info.filename).name)
                    with zf.open(info) as src, target.open("wb") as dst:
                        shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
                    extracted.append(target)
        elif suffix == ".7z":
            import py7zr
            with py7zr.SevenZipFile(path, mode="r") as archive:
                names = archive.getnames()
                selected = [n for n in names if not rx or rx.search(n)]
                archive.extract(path=destination, targets=selected or None)
            extracted = [p for p in destination.rglob("*") if p.is_file()]
        else:
            logger.warning("Formato de arquivo não extraível automaticamente: %s", path)
    except Exception as exc:
        logger.error("Falha ao extrair %s: %s", path, exc)
    return extracted


def detect_encoding(path: Path, sample_bytes: int = 2_000_000) -> str:
    from charset_normalizer import from_bytes
    with path.open("rb") as f:
        raw = f.read(sample_bytes)
    match = from_bytes(raw).best()
    return match.encoding if match and match.encoding else "utf-8"


def audit_fixed_width(path: Path, source_id: str, max_lines: int = 20_000) -> AuditResult:
    lengths: Counter[int] = Counter()
    sample = 0
    with path.open("rb") as f:
        for line in f:
            lengths[len(line.rstrip(b"\r\n"))] += 1
            sample += 1
            if sample >= max_lines:
                break
    mode = lengths.most_common(1)[0][0] if lengths else None
    result = AuditResult(
        source_id=source_id, path=str(path), kind="fixed_width_text",
        size_bytes=path.stat().st_size, sha256=sha256_file(path),
        sample_rows=sample, estimated_rows=estimate_line_count(path),
        line_length_min=min(lengths) if lengths else None,
        line_length_max=max(lengths) if lengths else None,
        line_length_mode=mode,
    )
    if len(lengths) > 5:
        result.warnings.append(f"Muitos comprimentos de linha no sample: {dict(lengths.most_common(10))}")
    return result


def sniff_delimiter(sample: str) -> str | None:
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=";,\t|")
        return dialect.delimiter
    except csv.Error:
        return None


def audit_delimited(path: Path, source_id: str, sample_rows: int = 50_000) -> AuditResult:
    import pandas as pd

    encoding = detect_encoding(path)
    with path.open("r", encoding=encoding, errors="replace") as f:
        sample_text = f.read(100_000)
    delim = sniff_delimiter(sample_text) or ";"
    warnings: list[str] = []
    try:
        df = pd.read_csv(path, sep=delim, encoding=encoding, nrows=sample_rows, low_memory=False)
        columns = [str(c) for c in df.columns]
    except Exception as exc:
        columns = []
        warnings.append(f"Falha ao ler amostra: {type(exc).__name__}: {exc}")
    return AuditResult(
        source_id=source_id, path=str(path), kind="delimited_text",
        size_bytes=path.stat().st_size, sha256=sha256_file(path), encoding=encoding,
        delimiter=delim, sample_rows=sample_rows, estimated_rows=estimate_line_count(path),
        columns=columns, warnings=warnings,
    )


def audit_generic(path: Path, source_id: str) -> AuditResult:
    ext = path.suffix.lower()
    if ext == ".txt":
        # PNADc é fixed-width; demais TXT tendem a ser delimitados.
        if "PNADC_" in path.name.upper() or "PNAD_COVID" in path.name.upper():
            return audit_fixed_width(path, source_id)
        return audit_delimited(path, source_id)
    if ext in {".csv", ".tsv"}:
        return audit_delimited(path, source_id)
    return AuditResult(
        source_id=source_id, path=str(path), kind=ext.lstrip(".") or "binary",
        size_bytes=path.stat().st_size, sha256=sha256_file(path),
    )


def parse_sas_input_layout(text: str) -> list[dict[str, Any]]:
    """Extrai layout de instruções SAS do tipo @posição variável $largura. ou largura."""
    rows = []
    rx = re.compile(
        r"@(?P<start>\d+)\s+(?P<name>[A-Za-z_][A-Za-z0-9_]*)\s+(?P<char>\$)?(?P<width>\d+)(?:\.\d+)?",
        flags=re.I,
    )
    for line in text.splitlines():
        m = rx.search(line)
        if not m:
            continue
        start = int(m.group("start"))
        width = int(m.group("width"))
        rows.append({
            "variable": m.group("name"), "start_1based": start,
            "end_1based": start + width - 1, "width": width,
            "type": "string" if m.group("char") else "numeric",
        })
    return rows


def discover_layout_files(root: Path) -> list[Path]:
    patterns = ["*.sas", "*input*.txt", "*INPUT*.txt", "*layout*.txt", "*dicion*.xls*", "*Dicion*.xls*"]
    out: list[Path] = []
    for pattern in patterns:
        out.extend(root.rglob(pattern))
    return sorted(set(out))


def extract_variable_names_from_docs(paths: Sequence[Path]) -> set[str]:
    names: set[str] = set()
    for path in paths:
        try:
            if path.suffix.lower() in {".txt", ".sas"}:
                enc = detect_encoding(path)
                text = path.read_text(encoding=enc, errors="replace")
                for row in parse_sas_input_layout(text):
                    names.add(row["variable"])
                names.update(re.findall(r"\b(?:S14\d{4}|SD14\d{3}|V\d{4,5}|VD\d{4}|C\d{3}[A-Z]?)\b", text))
            elif path.suffix.lower() in {".xls", ".xlsx"}:
                import pandas as pd
                # .xls requires xlrd; .xlsx uses openpyxl. Both are installed by bootstrap.
                book = pd.ExcelFile(path)
                for sheet in book.sheet_names:
                    df = pd.read_excel(path, sheet_name=sheet, header=None, nrows=10000, dtype=str)
                    for value in df.fillna("").astype(str).to_numpy().ravel():
                        names.update(re.findall(r"\b(?:S14\d{4}|SD14\d{3}|V\d{4,5}|VD\d{4}|C\d{3}[A-Z]?)\b", value))
            elif path.suffix.lower() == ".pdf":
                from pypdf import PdfReader
                reader = PdfReader(str(path))
                text = "\n".join((page.extract_text() or "") for page in reader.pages)
                names.update(re.findall(r"\b(?:S14\d{4}|SD14\d{3}|V\d{4,5}|VD\d{4}|C\d{3}[A-Z]?)\b", text))
        except Exception:
            continue
    return names


# -----------------------------------------------------------------------------
# Golden tests e fail-closed gates
# -----------------------------------------------------------------------------

def run_local_pnadc_gates(local_paths: Mapping[str, Path], audits: list[AuditResult]) -> list[GateResult]:
    gates: list[GateResult] = []
    for source_id, path in local_paths.items():
        if not path.exists():
            gates.append(GateResult(
                gate_id=f"{source_id}.exists", status="FAIL", severity="critical",
                source_id=source_id,
                message="Arquivo local PNADc obrigatório não encontrado.",
                evidence={"expected_path": str(path)},
            ))
            continue
        size = path.stat().st_size
        gates.append(GateResult(
            gate_id=f"{source_id}.exists", status="PASS", severity="critical",
            source_id=source_id, message="Arquivo local encontrado.",
            evidence={"path": str(path), "bytes": size},
        ))
        gates.append(GateResult(
            gate_id=f"{source_id}.size", status="PASS" if size > 10_000_000 else "FAIL",
            severity="critical", source_id=source_id,
            message="Tamanho plausível para microdado PNADc." if size > 10_000_000 else "Arquivo pequeno demais.",
            evidence={"bytes": size},
        ))
        audit = next((a for a in audits if a.source_id == source_id), None)
        if audit:
            stable = audit.line_length_mode is not None and audit.line_length_min == audit.line_length_max
            gates.append(GateResult(
                gate_id=f"{source_id}.fixed_width_stability",
                status="PASS" if stable else "WARN", severity="high", source_id=source_id,
                message="Comprimento fixed-width estável na amostra." if stable else "Variação de comprimento detectada; investigar CR/LF ou corrupção.",
                evidence={
                    "min": audit.line_length_min, "max": audit.line_length_max,
                    "mode": audit.line_length_mode, "sample_rows": audit.sample_rows,
                },
            ))
    return gates


def run_semantic_gates(variable_names: set[str], docs_found: bool) -> list[GateResult]:
    gates: list[GateResult] = []
    if not docs_found:
        gates.append(GateResult(
            gate_id="pnadc.docs.available", status="BLOCKED", severity="critical",
            source_id="pnadc_documentation",
            message="Documentação/layout oficial ainda não foi materializado; modelos ficam bloqueados.",
        ))
        return gates
    missing_direct = sorted(PNADC_REQUIRED_DIRECT - variable_names)
    missing_core = sorted(PNADC_REQUIRED_CORE - variable_names)
    gates.append(GateResult(
        gate_id="pnadc.layout.core_variables",
        status="PASS" if not missing_core else "FAIL", severity="critical",
        source_id="pnadc_documentation",
        message="Variáveis centrais presentes no layout." if not missing_core else "Variáveis centrais ausentes no layout detectado.",
        evidence={"missing": missing_core, "detected_count": len(variable_names)},
    ))
    gates.append(GateResult(
        gate_id="pnadc.layout.direct_platform",
        status="PASS" if "S140093" in variable_names else "FAIL", severity="critical",
        source_id="pnadc_documentation",
        message="S140093 validada no layout oficial." if "S140093" in variable_names else "S140093 não encontrada: PROIBIDO usar proxy como substituição silenciosa.",
        evidence={"missing_direct": missing_direct},
    ))
    gates.append(GateResult(
        gate_id="pnadc.no_silent_proxy_fallback", status="PASS", severity="critical",
        source_id="pnadc_direct",
        message="Política fail-closed registrada: ausência de S140093 bloqueia comparações diretas.",
    ))
    return gates


def global_fail_closed_gates() -> list[GateResult]:
    return [
        GateResult("global.no_synthetic_geography_as_observed", "PASS", "critical", "global",
                   "Geografia sintética nunca será tratada como localização observada."),
        GateResult("global.no_treatment_specific_outcome", "PASS", "critical", "global",
                   "É proibido subtrair custos somente do grupo tratado antes da comparação."),
        GateResult("global.no_random_labels_for_stgnn", "PASS", "critical", "global",
                   "STGNN só pode usar targets temporais observados; séries pseudoaleatórias são bloqueadas."),
        GateResult("global.no_causal_without_overlap", "PASS", "critical", "global",
                   "Estimadores causais/duplamente robustos exigem tratamento direto, controles e suporte."),
        GateResult("global.observed_estimated_simulated_separation", "PASS", "critical", "global",
                   "Todo artefato deve registrar observado/proxy/imputado/simulado."),
        GateResult("global.outcome_universe_separation", "PASS", "high", "global",
                   "Universo identificado e amostra válida de outcome serão preservados separadamente."),
    ]


# -----------------------------------------------------------------------------
# Downloaders por fonte
# -----------------------------------------------------------------------------

def download_ibge_directory(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                            mode: str, logger: logging.Logger) -> list[ArtifactRecord]:
    records: list[ArtifactRecord] = []
    assert source.discovery_url
    try:
        links = html_links(http, source.discovery_url)
        if source.source_id == "pnadc_official_2022q4_archive":
            chosen = select_links(links, r"PNADC_042022.*\.zip", ["zip"])
        elif source.source_id == "pnadc_official_2024q3_archive":
            chosen = select_links(links, r"PNADC_032024.*\.zip", ["zip"])
        else:
            chosen = [x for x in links if re.search(r"\.(zip|xlsx?|pdf|txt|sas)$", urllib.parse.urlparse(x[1]).path, re.I)]
        if mode == "inventory":
            for text, url in chosen:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                    agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                    source_url=url, file_name=sanitize_filename(Path(urllib.parse.urlparse(url).path).name or text),
                    security="https", license=source.license, discovered_at_utc=utc_now(),
                ))
            return records
        # Arquivos PNADc brutos são enormes e já existem localmente: core baixa docs, full baixa arquivos oficiais.
        if source.source_id.startswith("pnadc_official_") and mode != "full":
            for text, url in chosen:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DISCOVERED_NOT_DOWNLOADED_LOCAL_BASE_EXISTS",
                    phase="0", agency=source.agency, dataset=source.dataset,
                    measurement_status=source.measurement_status, source_url=url,
                    file_name=sanitize_filename(Path(urllib.parse.urlparse(url).path).name or text),
                    security="https", discovered_at_utc=utc_now(),
                    notes="Microdado local é a base; archive oficial será baixado apenas em mode=full."
                ))
            return records
        for text, url in chosen:
            name = sanitize_filename(Path(urllib.parse.urlparse(url).path).name or text)
            records.append(http.download(url, dest / name, source, run_id))
        if not chosen:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                error="Nenhum link compatível descoberto"
            ))
    except Exception as exc:
        logger.error("Descoberta IBGE falhou para %s: %s", source.source_id, exc)
        records.append(ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
            error=f"{type(exc).__name__}: {exc}"
        ))
    return records


def download_pnadc_historical(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                              mode: str, logger: logging.Logger) -> list[ArtifactRecord]:
    """Descobre/baixa os quatro trimestres PNADc por ano sem hardcode de sufixo de revisão."""
    records: list[ArtifactRecord] = []
    for year in source.years:
        year_url = urllib.parse.urljoin(source.discovery_url.rstrip("/") + "/", f"{year}/")
        try:
            links = html_links(http, year_url)
            chosen = select_links(links, rf"PNADC_0[1-4]{year}.*\.zip", ["zip"])
            for text, url in chosen:
                name = sanitize_filename(Path(urllib.parse.urlparse(url).path).name or text)
                if mode == "inventory" or mode == "core":
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                        agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                        source_url=url, file_name=name, security="https", discovered_at_utc=utc_now(),
                        notes="Download efetivo reservado a mode=full para controlar armazenamento."
                    ))
                else:
                    records.append(http.download(url, dest / str(year) / name, source, run_id))
            if not chosen:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                    agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                    source_url=year_url, security="https", discovered_at_utc=utc_now(),
                    notes=f"Nenhum trimestre localizado para {year}"
                ))
        except Exception as exc:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=year_url, security="https", discovered_at_utc=utc_now(),
                error=f"{type(exc).__name__}: {exc}"
            ))
    return records


def discover_ibge_sidra_tables(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                               mode: str, logger: logging.Logger) -> tuple[list[ArtifactRecord], dict[str, Any]]:
    """Registra IDs de tabelas SIDRA linkadas pela página oficial sem fazer consulta irrestrita gigante."""
    records: list[ArtifactRecord] = []
    meta: dict[str, Any] = {"source_page": source.discovery_url, "discovered_at_utc": utc_now(), "tables": []}
    try:
        r = http.get(source.discovery_url)
        r.raise_for_status()
        html = r.text
        ids = sorted(set(re.findall(r"sidra\.ibge\.gov\.br/(?:tabela|Tabela)/(\d+)", html, flags=re.I)))
        # Alguns links são redirecionados/embutidos com apenas /tabela/ID.
        ids += [x for x in sorted(set(re.findall(r"(?:/tabela/|Tabela/)(\d+)", html, flags=re.I))) if x not in ids]
        meta["tables"] = [{"table_id": x, "url": f"https://sidra.ibge.gov.br/tabela/{x}"} for x in ids]
        dest.mkdir(parents=True, exist_ok=True)
        html_path = dest / "pnadc_platform_2024_official_page.html"
        if mode != "inventory":
            html_path.write_text(html, encoding="utf-8")
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="DOWNLOADED", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, local_path=str(html_path), file_name=html_path.name,
                bytes=html_path.stat().st_size, sha256=sha256_file(html_path), security="https",
                downloaded_at_utc=utc_now(), discovered_at_utc=utc_now(),
                notes=f"IDs SIDRA descobertos: {len(ids)}"
            ))
        else:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                notes=f"IDs SIDRA descobertos: {len(ids)}"
            ))
        if not ids:
            logger.warning("Nenhum ID SIDRA foi extraído da página oficial; a página pode carregar links por JavaScript.")
    except Exception as exc:
        records.append(ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
            error=f"{type(exc).__name__}: {exc}"
        ))
    return records, meta


def download_pnad_covid(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                        mode: str, logger: logging.Logger) -> list[ArtifactRecord]:
    records: list[ArtifactRecord] = []
    base = source.discovery_url.rstrip("/") + "/"
    dirs = {
        "dados": urllib.parse.urljoin(base, "Dados/"),
        "docs": urllib.parse.urljoin(base, "Documentacao/"),
    }
    for kind, url in dirs.items():
        try:
            links = html_links(http, url)
            if kind == "dados":
                pattern = r"PNAD_COVID_(092020|052020|062020|072020|082020|102020|112020)\.zip"
                chosen = select_links(links, pattern, ["zip"])
                if mode == "core":
                    chosen = [x for x in chosen if "092020" in x[1]]
            else:
                chosen = [x for x in links if re.search(r"\.(zip|xlsx?|pdf|txt)$", urllib.parse.urlparse(x[1]).path, re.I)]
            for text, href in chosen:
                name = sanitize_filename(Path(urllib.parse.urlparse(href).path).name or text)
                if mode == "inventory":
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                        agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                        source_url=href, file_name=name, security="https", discovered_at_utc=utc_now()
                    ))
                else:
                    records.append(http.download(href, dest / kind / name, source, run_id))
        except Exception as exc:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=url, security="https", discovered_at_utc=utc_now(), error=f"{type(exc).__name__}: {exc}"
            ))
    return records


def download_recife_ckan(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                         mode: str, logger: logging.Logger, core_max_mb: int) -> tuple[list[ArtifactRecord], list[dict[str, Any]]]:
    records: list[ArtifactRecord] = []
    packages_meta: list[dict[str, Any]] = []
    for alias, item in RECIFE_CKAN_PACKAGES.items():
        try:
            pkg = resolve_ckan_package(http, item)
            if not pkg:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=f"recife_{alias}", status="NOT_FOUND", phase="0",
                    agency=source.agency, dataset=alias, measurement_status=source.measurement_status,
                    source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                    notes=f"required={item.get('required', False)}"
                ))
                continue
            packages_meta.append({
                "alias": alias, "id": pkg.get("id"), "name": pkg.get("name"), "title": pkg.get("title"),
                "metadata_modified": pkg.get("metadata_modified"), "license_title": pkg.get("license_title"),
                "num_resources": len(pkg.get("resources", [])), "url": pkg.get("url"),
            })
            resources = choose_ckan_resources(pkg, mode)
            if mode == "inventory":
                resources = [dict(r) for r in pkg.get("resources", [])]
            for res in resources:
                url = str(res.get("url") or "")
                if not url.startswith("https://"):
                    continue
                raw_name = res.get("name") or Path(urllib.parse.urlparse(url).path).name or res.get("id")
                ext = str(res.get("format") or "bin").lower()
                name = sanitize_filename(str(raw_name))
                if "." not in Path(name).name:
                    name += f".{ext}"
                subdir = dest / alias
                if mode == "inventory":
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=f"recife_{alias}", status="DISCOVERED", phase="0",
                        agency=source.agency, dataset=str(pkg.get("title") or alias),
                        measurement_status=source.measurement_status, source_url=url,
                        file_name=name, media_type=str(res.get("mimetype") or ext), security="https",
                        license=str(pkg.get("license_title") or source.license), discovered_at_utc=utc_now(),
                    ))
                else:
                    max_bytes = core_max_mb * 1024 * 1024 if mode == "core" else None
                    spec = dataclasses.replace(
                        source, source_id=f"recife_{alias}", dataset=str(pkg.get("title") or alias),
                        license=str(pkg.get("license_title") or source.license)
                    )
                    records.append(http.download(url, subdir / name, spec, run_id, max_bytes=max_bytes))
        except Exception as exc:
            logger.error("CKAN %s falhou: %s", alias, exc)
            records.append(ArtifactRecord(
                run_id=run_id, source_id=f"recife_{alias}", status="ERROR", phase="0",
                agency=source.agency, dataset=alias, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                error=f"{type(exc).__name__}: {exc}", notes=f"required={item.get('required', False)}"
            ))
    return records, packages_meta


def download_inmet(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                    mode: str, logger: logging.Logger) -> list[ArtifactRecord]:
    records: list[ArtifactRecord] = []
    try:
        links = html_links(http, source.discovery_url)
        year_rx = re.compile(r"(20\d{2})")
        candidates = []
        for text, url in links:
            match = year_rx.search(text + " " + url)
            if match and (url.lower().endswith(".zip") or ".zip" in url.lower()):
                year = int(match.group(1))
                if year in source.years:
                    candidates.append((year, text, url))
        if not candidates:
            # Padrão público histórico do portal INMET; o download ainda passa por validação HTTPS/allowlist.
            candidates = [
                (year, f"INMET {year}", f"https://portal.inmet.gov.br/uploads/dadoshistoricos/{year}.zip")
                for year in source.years
            ]
        if mode == "core":
            # Core only needs the direct-module and urban-dynamics benchmark years.
            # Full mode remains responsible for the complete historical weather series.
            candidates = [x for x in candidates if x[0] in {2022, 2023, 2024}]
        for year, text, url in sorted(set(candidates)):
            name = sanitize_filename(Path(urllib.parse.urlparse(url).path).name or f"INMET_{year}.zip")
            if mode == "inventory":
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                    agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                    source_url=url, file_name=name, security="https", discovered_at_utc=utc_now(),
                ))
            else:
                rec = http.download(url, dest / "archives" / name, source, run_id)
                records.append(rec)
                if rec.status in {"DOWNLOADED", "EXISTS"} and rec.local_path:
                    extracted = extract_archive(Path(rec.local_path), dest / "A301", logger, r"(A301|RECIFE)")
                    for file in extracted:
                        records.append(ArtifactRecord(
                            run_id=run_id, source_id=source.source_id, status="EXTRACTED", phase="0",
                            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                            source_url=url, local_path=str(file), file_name=file.name,
                            bytes=file.stat().st_size, sha256=sha256_file(file), security="https",
                            downloaded_at_utc=utc_now(), discovered_at_utc=utc_now(),
                            notes="Subconjunto estação A301/Recife extraído do arquivo anual"
                        ))
        if not candidates:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                error="Links ZIP anuais não detectados; revisar HTML do portal INMET"
            ))
    except Exception as exc:
        records.append(ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
            error=f"{type(exc).__name__}: {exc}"
        ))
    return records


def download_anp(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                 mode: str, logger: logging.Logger) -> list[ArtifactRecord]:
    """Download only the aggregated historical fuel-price files needed for pass-through.

    The previous broad scraper could follow unrelated ANP pages and very large DSAS
    microdata. This version only accepts downloadable spreadsheet/CSV resources from
    the official historical-price page and explicitly excludes DSAS/registry assets.
    """
    records: list[ArtifactRecord] = []
    try:
        links = html_links(http, source.discovery_url)
        candidates: list[tuple[str, str]] = []
        for text, url in links:
            low_url = url.lower()
            blob = (text + " " + url).lower()
            path = urllib.parse.urlparse(url).path.lower()
            if "/dsas/" in path or "renovabio" in low_url:
                continue
            if not re.search(r"\.(csv|xlsx?|ods)(?:$|\?)", low_url, re.I):
                continue
            # Prefer municipality-level aggregated series; retain state/Brazil only in full.
            is_municipal = "munic" in blob
            if mode == "core" and not is_municipal:
                continue
            if not re.search(r"pre[cç]o|combust|levantamento|serie|s[eé]rie", blob, re.I):
                continue
            candidates.append((text, url))

        seen: set[str] = set()
        for text, url in candidates:
            if url in seen:
                continue
            seen.add(url)
            name = sanitize_filename(Path(urllib.parse.urlparse(url).path).name or "ANP_precos_historicos.xlsx")
            if mode == "inventory":
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                    agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                    source_url=url, file_name=name, security="https", discovered_at_utc=utc_now(),
                ))
            else:
                records.append(http.download(url, dest / name, source, run_id))
        if not candidates:
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="NOT_FOUND", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
                error="Nenhuma planilha agregada municipal foi detectada; fonte registrada para revisão manual"
            ))
    except Exception as exc:
        records.append(ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
            error=f"{type(exc).__name__}: {exc}"
        ))
    return records


def download_bcb_ipca(http: SafeHTTP, source: SourceSpec, dest: Path, run_id: str,
                      mode: str) -> list[ArtifactRecord]:
    if mode == "inventory":
        return [ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, file_name="bcb_sgs_433_ipca.json",
            security="https", discovered_at_utc=utc_now(),
        )]
    return [http.download(source.discovery_url, dest / "bcb_sgs_433_ipca.json", source, run_id)]


def download_osm(source: SourceSpec, dest: Path, run_id: str,
                 logger: logging.Logger) -> list[ArtifactRecord]:
    records: list[ArtifactRecord] = []
    try:
        import osmnx as ox
        ox.settings.use_cache = True
        ox.settings.cache_folder = str(dest / "cache")
        ox.settings.requests_timeout = 180
        for network_type in ["drive", "bike", "walk"]:
            logger.info("Baixando OSM Recife (%s)...", network_type)
            graph = ox.graph_from_place("Recife, Pernambuco, Brazil", network_type=network_type, simplify=False)
            graphml = dest / f"recife_{network_type}_unsimplified.graphml"
            gpkg = dest / f"recife_{network_type}_unsimplified.gpkg"
            ox.save_graphml(graph, graphml)
            ox.save_graph_geopackage(graph, gpkg, directed=True)
            for path in [graphml, gpkg]:
                records.append(ArtifactRecord(
                    run_id=run_id, source_id=source.source_id, status="DOWNLOADED", phase="0",
                    agency=source.agency, dataset=f"{source.dataset} — {network_type}",
                    measurement_status=source.measurement_status, source_url=source.discovery_url,
                    local_path=str(path), file_name=path.name, bytes=path.stat().st_size,
                    sha256=sha256_file(path), security="https via OSMnx/Nominatim/Overpass",
                    license=source.license, downloaded_at_utc=utc_now(), discovered_at_utc=utc_now(),
                ))
    except Exception as exc:
        records.append(ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            source_url=source.discovery_url, security="https", discovered_at_utc=utc_now(),
            error=f"{type(exc).__name__}: {exc}",
        ))
    return records


def recover_required_pnadc_from_archives(
    tree: Mapping[str, Path],
    records: Sequence[ArtifactRecord],
    local_paths: dict[str, Path],
    run_id: str,
    logger: logging.Logger,
) -> tuple[list[ArtifactRecord], list[AuditResult]]:
    """Extract the two required direct PNADc TXT files from official archives.

    This is activated only when the user-supplied/local file is absent. It lets hosted
    Colab recover from the official IBGE archives without silently substituting a proxy.
    """
    mapping = {
        "pnadc_2022q4_direct": ("pnadc_official_2022q4_archive", "PNADC_042022.txt", "2022Q4"),
        "pnadc_2024q3_direct": ("pnadc_official_2024q3_archive", "PNADC_032024.txt", "2024Q3"),
    }
    out_records: list[ArtifactRecord] = []
    out_audits: list[AuditResult] = []
    for key, (archive_source, target_name, period) in mapping.items():
        if local_paths[key].exists():
            continue
        archives = [
            Path(r.local_path) for r in records
            if r.source_id == archive_source and r.local_path and Path(r.local_path).suffix.lower() == ".zip"
            and Path(r.local_path).exists()
        ]
        if not archives:
            archives = list(tree["raw_ibge"].rglob(f"*{target_name.replace('.txt', '')}*.zip"))
        if not archives:
            logger.warning("Arquivo oficial %s ainda não disponível para recuperação.", target_name)
            continue
        archive = archives[0]
        destination = tree["raw_ibge"] / "direct_required" / period / target_name
        destination.parent.mkdir(parents=True, exist_ok=True)
        try:
            with zipfile.ZipFile(archive) as zf:
                member = next((n for n in zf.namelist() if Path(n).name.upper() == target_name.upper()), None)
                if member is None:
                    member = next((n for n in zf.namelist() if target_name.replace('.txt','') in Path(n).name.upper()), None)
                if member is None:
                    raise FileNotFoundError(f"{target_name} não encontrado dentro de {archive.name}")
                with zf.open(member) as src, destination.open("wb") as dst:
                    shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
            local_paths[key] = destination
            audit = audit_fixed_width(destination, key)
            out_audits.append(audit)
            out_records.append(ArtifactRecord(
                run_id=run_id, source_id=key, status="RECOVERED_FROM_OFFICIAL_ARCHIVE", phase="0",
                agency="IBGE", dataset=f"PNADc direta {period}", measurement_status="observado_direto",
                source_url=next((r.source_url for r in records if r.source_id == archive_source and r.local_path == str(archive)), None),
                local_path=str(destination), file_name=destination.name, bytes=destination.stat().st_size,
                sha256=audit.sha256, security="https+local_extract", downloaded_at_utc=utc_now(),
                discovered_at_utc=utc_now(), notes=f"Extraído de {archive.name}; sem proxy."
            ))
            logger.info("PNADc recuperada do arquivo oficial: %s", destination)
        except Exception as exc:
            logger.error("Falha ao recuperar %s de %s: %s", target_name, archive, exc)
            out_records.append(ArtifactRecord(
                run_id=run_id, source_id=key, status="RECOVERY_ERROR", phase="0",
                agency="IBGE", dataset=f"PNADc direta {period}", measurement_status="observado_direto",
                local_path=str(destination), file_name=target_name, security="https+local_extract",
                discovered_at_utc=utc_now(), error=f"{type(exc).__name__}: {exc}"
            ))
    return out_records, out_audits


# -----------------------------------------------------------------------------
# Inventário local, manifests e relatórios
# -----------------------------------------------------------------------------

def register_local_reference(source: SourceSpec, run_id: str, refs_dir: Path) -> tuple[ArtifactRecord, AuditResult | None]:
    path = Path(source.local_path or "")
    if not path.exists():
        rec = ArtifactRecord(
            run_id=run_id, source_id=source.source_id, status="MISSING_LOCAL_REQUIRED", phase="0",
            agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
            local_path=str(path), file_name=path.name, security="local", discovered_at_utc=utc_now(),
            error="Arquivo local não encontrado"
        )
        return rec, None
    audit = audit_fixed_width(path, source.source_id)
    pointer = {
        "source_id": source.source_id,
        "absolute_path": str(path.resolve()),
        "bytes": path.stat().st_size,
        "sha256": audit.sha256,
        "estimated_rows": audit.estimated_rows,
        "registered_at_utc": utc_now(),
        "copy_policy": "reference_only_no_duplication",
    }
    write_json(refs_dir / f"{source.source_id}.link.json", pointer)
    rec = ArtifactRecord(
        run_id=run_id, source_id=source.source_id, status="REGISTERED_LOCAL", phase="0",
        agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
        local_path=str(path.resolve()), file_name=path.name, bytes=path.stat().st_size,
        sha256=audit.sha256, security="local", discovered_at_utc=utc_now(),
        notes="Arquivo referenciado sem duplicação para preservar espaço"
    )
    return rec, audit


def manifest_to_csv(records: Sequence[ArtifactRecord], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = [f.name for f in dataclasses.fields(ArtifactRecord)]
    with path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for record in records:
            writer.writerow(asdict(record))


def gates_to_csv(gates: Sequence[GateResult], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=["gate_id", "status", "severity", "source_id", "message", "evidence"])
        writer.writeheader()
        for gate in gates:
            row = asdict(gate)
            row["evidence"] = json.dumps(row["evidence"], ensure_ascii=False)
            writer.writerow(row)


def write_environment_lock(path: Path, run_id: str, root: Path, root_mode: str) -> None:
    packages = {}
    for pkg in ["requests", "beautifulsoup4", "lxml", "pandas", "pyarrow", "polars", "py7zr", "openpyxl", "pyyaml", "osmnx", "geopandas"]:
        try:
            packages[pkg] = importlib.metadata.version(pkg)
        except importlib.metadata.PackageNotFoundError:
            packages[pkg] = None
    data = {
        "run_id": run_id,
        "script_version": SCRIPT_VERSION,
        "schema_version": SCHEMA_VERSION,
        "created_at_utc": utc_now(),
        "root": str(root),
        "root_resolution": root_mode,
        "python": sys.version,
        "platform": platform.platform(),
        "hostname": socket.gethostname(),
        "packages": packages,
        "git_commit": _git_commit(),
    }
    write_json(path, data)


def _git_commit() -> str | None:
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL, text=True).strip()
    except Exception:
        return None


def initial_decisions(run_id: str) -> list[dict[str, Any]]:
    return [
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D001", "decision": "S140093 é o identificador direto primário de entrega por plataforma.", "rationale": "Evita proxy silenciosa.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D002", "decision": "PNAD COVID C007C=17 é ocupação de entrega, não uso direto de plataforma.", "rationale": "Limite semântico oficial.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D003", "decision": "RAIS/PNAD COVID/PNADc não serão raw-pooled como painel causal.", "rationale": "Universos e desenhos incompatíveis.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D004", "decision": "Custos do TFD serão outcomes/cenários separados e nunca subtraídos somente do tratado antes da estimação.", "rationale": "Evita inserir mecanicamente a penalidade.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D005", "decision": "Pedidos, despacho, espera e retorno vazio sem logs serão sempre rotulados como simulados.", "rationale": "Separação observado/estimado/simulado.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D006", "decision": "STGNN usará targets urbanos observados e validação espaço-temporal; random labels são proibidos.", "rationale": "Validade preditiva.", "status": "locked"},
        {"run_id": run_id, "timestamp_utc": utc_now(), "decision_id": "D007", "decision": "Fase 4 identifica conjunto de regimes compatíveis; não recupera o algoritmo proprietário real.", "rationale": "Identificação parcial e equifinalidade.", "status": "locked"},
    ]


def generate_report(root: Path, run_id: str, mode: str, records: Sequence[ArtifactRecord],
                    audits: Sequence[AuditResult], gates: Sequence[GateResult], root_mode: str) -> str:
    status_counts = Counter(r.status for r in records)
    gate_counts = Counter(g.status for g in gates)
    required_failures = [g for g in gates if g.status in {"FAIL", "BLOCKED"} and g.severity == "critical"]
    lines = [
        "# SPINE-GPE v7 — Relatório da Fase 0",
        "",
        f"- **Run ID:** `{run_id}`",
        f"- **Versão:** `{SCRIPT_VERSION}`",
        f"- **Modo:** `{mode}`",
        f"- **Diretório:** `{root}`",
        f"- **Resolução do diretório:** `{root_mode}`",
        f"- **Gerado em:** `{utc_now()}`",
        "",
        "## Resultado executivo",
        "",
    ]
    if required_failures:
        lines.append("**LOCK: BLOQUEADO.** Há gates críticos não atendidos; nenhuma fase inferencial deve prosseguir.")
    else:
        lines.append("**LOCK: LIBERADO PARA A PRÓXIMA ETAPA**, condicionado aos limites registrados no Claim/Evidence Book.")
    lines += [
        "",
        "## Artefatos por status",
        "",
        "| Status | N |",
        "|---|---:|",
    ]
    for status, n in sorted(status_counts.items()):
        lines.append(f"| {status} | {n} |")
    lines += ["", "## Gates", "", "| Status | N |", "|---|---:|"]
    for status, n in sorted(gate_counts.items()):
        lines.append(f"| {status} | {n} |")
    lines += ["", "## Gates críticos não atendidos", ""]
    if not required_failures:
        lines.append("Nenhum.")
    else:
        for g in required_failures:
            lines.append(f"- **{g.gate_id}** — {g.message}")
    lines += [
        "",
        "## Arquivos locais auditados",
        "",
        "| Fonte | Caminho | GiB | SHA-256 | Linhas estimadas | Comprimento modal |",
        "|---|---|---:|---|---:|---:|",
    ]
    for a in audits:
        gib = a.size_bytes / 1024**3
        lines.append(f"| {a.source_id} | `{a.path}` | {gib:.3f} | `{a.sha256[:16]}…` | {a.estimated_rows or ''} | {a.line_length_mode or ''} |")
    lines += [
        "",
        "## Regras epistemológicas congeladas",
        "",
        "1. Observado, proxy, imputado e simulado permanecem separados em todos os manifests.",
        "2. Ausência de `S140093` bloqueia identificação direta; não ativa proxy automaticamente.",
        "3. PNAD COVID mede ocupação de entrega no choque pandêmico, não plataforma confirmada.",
        "4. RAIS mede formalidade registrada e não representa o universo informal.",
        "5. CTTU mede tráfego veicular geral, não demanda observada de entregadores.",
        "6. NAIN/NACH representam configuração; não são fluxo ou demanda observados.",
        "7. Outputs da Fase 4 são contrafactuais dentro do modelo e do conjunto compatível.",
        "",
        "## Próximo artefato esperado",
        "",
        "Executar o parser dos layouts oficiais PNADc e produzir a primeira `certified_table` com golden tests de prevalência, desenho survey e identificação direta.",
    ]
    return "\n".join(lines) + "\n"


# -----------------------------------------------------------------------------
# Orquestrador
# -----------------------------------------------------------------------------

def _sanitize_notebook_argv(argv: Sequence[str] | None = None) -> list[str]:
    """Remove apenas argumentos internos do kernel Jupyter/Colab.

    Ao executar um arquivo com ``%run``, ``exec`` ou ``runpy`` dentro de um
    notebook, o ipykernel pode acrescentar ``-f <kernel-....json>`` a
    ``sys.argv``. Esse argumento não pertence ao pipeline e fazia o argparse
    encerrar a execução com SystemExit(2). Argumentos desconhecidos reais
    continuam sendo rejeitados pelo argparse.
    """
    raw = list(sys.argv[1:] if argv is None else argv)
    in_notebook = (
        "ipykernel" in sys.modules
        or "google.colab" in sys.modules
        or os.environ.get("COLAB_RELEASE_TAG") is not None
    )
    if not in_notebook:
        return raw

    cleaned: list[str] = []
    i = 0
    while i < len(raw):
        token = raw[i]
        if token == "-f" and i + 1 < len(raw):
            candidate = raw[i + 1]
            if candidate.endswith(".json") and ("kernel-" in candidate or "jupyter" in candidate):
                i += 2
                continue
        if token.startswith("-f="):
            candidate = token.split("=", 1)[1]
            if candidate.endswith(".json") and ("kernel-" in candidate or "jupyter" in candidate):
                i += 1
                continue
        cleaned.append(token)
        i += 1
    return cleaned


def parse_args(argv: Sequence[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="SPINE-GPE v7 — Fase 0")
    parser.add_argument("--root", help="Diretório raiz; padrão D:\\aCidadeAlgoritmica\\SPINE-GPEv7 no Windows")
    parser.add_argument("--pnadc-2022-path", help="Caminho explícito para PNADC_042022.txt")
    parser.add_argument("--pnadc-2024-path", help="Caminho explícito para PNADC_032024.txt")
    parser.add_argument("--mode", choices=["inventory", "core", "full"], default="core")
    parser.add_argument("--bootstrap", action="store_true", help="Instala dependências mínimas")
    parser.add_argument("--allow-official-ftp", action="store_true", help="Autoriza FTP oficial MTE sem criptografia")
    parser.add_argument("--download-osm", action="store_true", help="Baixa redes OSM Recife via OSMnx")
    parser.add_argument("--core-max-resource-mb", type=int, default=600, help="Limite por recurso CKAN em mode=core")
    parser.add_argument("--overwrite", action="store_true", help="Reservado para futuras versões; downloads existentes são preservados")
    parser.add_argument("--strict", action="store_true", help="Retorna exit code 2 quando gate crítico falha")
    return parser.parse_args(_sanitize_notebook_argv(argv))


def main(argv: Sequence[str] | None = None) -> int:
    args = parse_args(argv)
    root, root_mode = resolve_root(args.root)
    tree = create_project_tree(root)
    run_id = dt.datetime.now().strftime("%Y%m%dT%H%M%S") + "Z"
    logger = setup_logger(tree["logs"] / f"phase0_{run_id}.log")
    logger.info("SPINE-GPE v7 Fase 0 | run=%s | mode=%s | root=%s", run_id, args.mode, root)
    if is_colab() and str(root).startswith("/content/") and not str(root).startswith("/content/drive/"):
        logger.warning("ROOT efêmero do Colab: %s. Os arquivos serão perdidos ao encerrar a sessão; prefira /content/drive/MyDrive/...", root)

    if args.bootstrap:
        bootstrap_packages(logger)

    # Imports externos validados após bootstrap.
    try:
        import requests  # noqa: F401
        import pandas  # noqa: F401
        import bs4  # noqa: F401
    except ImportError as exc:
        logger.error("Dependência ausente: %s. Execute com --bootstrap.", exc)
        return 1

    write_environment_lock(tree["admin"] / f"environment_lock_{run_id}.json", run_id, root, root_mode)
    local_paths = resolve_local_pnadc_paths(args.pnadc_2022_path, args.pnadc_2024_path, root)
    sources = build_source_registry(local_paths)
    write_json(tree["registry"] / "source_registry.json", [asdict(s) for s in sources])
    write_json(tree["registry"] / "semantic_dictionary.json", semantic_dictionary())
    write_json(tree["contracts"] / "data_contracts.json", data_contracts())
    write_json(tree["registry"] / "estimand_registry.json", estimand_registry())
    for name, content in dag_files().items():
        (tree["dags"] / name).write_text(content + "\n", encoding="utf-8")
    for decision in initial_decisions(run_id):
        append_jsonl(tree["decisions"] / "analysis_decisions.jsonl", decision)
    # Inicializa registros mutáveis sem inventar decisões futuras.
    for name in ["exclusions.jsonl", "data_losses.jsonl", "protocol_deviations.jsonl"]:
        (tree["decisions"] / name).touch(exist_ok=True)

    records: list[ArtifactRecord] = []
    audits: list[AuditResult] = []
    gates: list[GateResult] = global_fail_closed_gates()

    # 1) Arquivos locais fundamentais.
    for source in sources:
        if source.strategy == "local_reference":
            rec, audit = register_local_reference(source, run_id, tree["raw_local_refs"])
            records.append(rec)
            if audit:
                audits.append(audit)

    # 2) Descoberta/download público.
    http = SafeHTTP(logger)
    packages_meta: list[dict[str, Any]] = []
    for source in sources:
        try:
            logger.info("Fonte: %s | estratégia=%s", source.source_id, source.strategy)
            if source.strategy == "local_reference":
                continue
            if source.strategy in {"ibge_directory_regex", "ibge_directory_all"}:
                source_mode = args.mode
                # In hosted Colab the Windows D: files are unavailable. If either direct
                # PNADc file is missing, core mode downloads only its official archive.
                if source.source_id == "pnadc_official_2022q4_archive" and not local_paths["pnadc_2022q4_direct"].exists() and args.mode == "core":
                    source_mode = "full"
                if source.source_id == "pnadc_official_2024q3_archive" and not local_paths["pnadc_2024q3_direct"].exists() and args.mode == "core":
                    source_mode = "full"
                records.extend(download_ibge_directory(
                    http, source, tree["raw_ibge"] / source.source_id, run_id, source_mode, logger
                ))
            elif source.strategy == "pnadc_historical_quarters":
                records.extend(download_pnadc_historical(
                    http, source, tree["raw_ibge"] / "pnadc_regular_historical", run_id, args.mode, logger
                ))
            elif source.strategy == "ibge_product_sidra_links":
                recs, sidra_meta = discover_ibge_sidra_tables(
                    http, source, tree["raw_ibge"] / "official_benchmarks", run_id, args.mode, logger
                )
                records.extend(recs)
                write_json(tree["registry"] / "sidra_platform_tables.json", sidra_meta)
            elif source.strategy == "ibge_pnad_covid":
                records.extend(download_pnad_covid(
                    http, source, tree["raw_ibge"] / "pnad_covid", run_id, args.mode, logger
                ))
            elif source.strategy == "direct_https":
                if args.mode == "inventory":
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="DISCOVERED", phase="0",
                        agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                        source_url=source.discovery_url, file_name=Path(urllib.parse.urlparse(source.discovery_url).path).name,
                        security="https", discovered_at_utc=utc_now(),
                    ))
                else:
                    name = sanitize_filename(Path(urllib.parse.urlparse(source.discovery_url).path).name)
                    records.append(http.download(source.discovery_url, tree["raw_ibge"] / "censo2022" / name, source, run_id))
            elif source.strategy == "recife_ckan_bundle":
                recs, meta = download_recife_ckan(
                    http, source, tree["raw_recife"], run_id, args.mode, logger, args.core_max_resource_mb
                )
                records.extend(recs)
                packages_meta.extend(meta)
            elif source.strategy == "inmet_annual_station":
                records.extend(download_inmet(http, source, tree["raw_inmet"], run_id, args.mode, logger))
            elif source.strategy == "anp_page_links":
                records.extend(download_anp(http, source, tree["raw_anp"], run_id, args.mode, logger))
            elif source.strategy == "bcb_api":
                records.extend(download_bcb_ipca(http, source, tree["raw_bcb"], run_id, args.mode))
            elif source.strategy in {"mte_ftp_rais_nordeste", "mte_ftp_novo_caged"}:
                if not args.allow_official_ftp:
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="BLOCKED_SECURITY_OPT_IN", phase="0",
                        agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                        source_url=source.discovery_url, security=source.security, discovered_at_utc=utc_now(),
                        notes="Use --allow-official-ftp para autorizar o FTP oficial MTE sem criptografia."
                    ))
                    gates.append(GateResult(
                        gate_id=f"{source.source_id}.transport_opt_in", status="BLOCKED", severity="high",
                        source_id=source.source_id,
                        message="Download automático bloqueado porque a fonte oficial oferece FTP sem criptografia.",
                    ))
                elif args.mode == "full":
                    if source.strategy == "mte_ftp_rais_nordeste":
                        records.extend(download_mte_rais(source, tree["raw_mte"] / "RAIS", run_id, logger, source.years))
                    else:
                        records.extend(download_mte_novo_caged(source, tree["raw_mte"] / "NOVO_CAGED", run_id, logger, source.years))
                else:
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="DISCOVERED_NOT_DOWNLOADED_MODE_FULL_REQUIRED",
                        phase="0", agency=source.agency, dataset=source.dataset,
                        measurement_status=source.measurement_status, source_url=source.discovery_url,
                        security=source.security, discovered_at_utc=utc_now(),
                        notes="Use --mode full --allow-official-ftp."
                    ))
            elif source.strategy == "osmnx_place":
                if args.download_osm and args.mode == "full":
                    records.extend(download_osm(source, tree["raw_osm"], run_id, logger))
                else:
                    records.append(ArtifactRecord(
                        run_id=run_id, source_id=source.source_id, status="DISCOVERED_NOT_DOWNLOADED",
                        phase="0", agency=source.agency, dataset=source.dataset,
                        measurement_status=source.measurement_status, source_url=source.discovery_url,
                        security="https", license=source.license, discovered_at_utc=utc_now(),
                        notes="Use --mode full --download-osm."
                    ))
        except Exception as exc:
            logger.error("Erro não tratado em %s: %s", source.source_id, exc)
            logger.debug(traceback.format_exc())
            records.append(ArtifactRecord(
                run_id=run_id, source_id=source.source_id, status="ERROR", phase="0",
                agency=source.agency, dataset=source.dataset, measurement_status=source.measurement_status,
                source_url=source.discovery_url, local_path=source.local_path,
                security=source.security, discovered_at_utc=utc_now(),
                error=f"{type(exc).__name__}: {exc}",
            ))

    # Recover required direct PNADc files from official archives when hosted Colab
    # cannot see the original Windows D: drive.
    recovered_records, recovered_audits = recover_required_pnadc_from_archives(
        tree, records, local_paths, run_id, logger
    )
    records.extend(recovered_records)
    audits.extend(recovered_audits)
    # Update source paths and re-register recovered files in the provenance registry.
    key_by_source = {
        "pnadc_2022q4_direct_local": "pnadc_2022q4_direct",
        "pnadc_2024q3_direct_local": "pnadc_2024q3_direct",
    }
    for source in sources:
        key = key_by_source.get(source.source_id)
        if key:
            source.local_path = str(local_paths[key])
    write_json(tree["registry"] / "source_registry.json", [asdict(s) for s in sources])
    gates.extend(run_local_pnadc_gates(local_paths, audits))

    write_json(tree["registry"] / "recife_ckan_packages_resolved.json", packages_meta)

    # 3) Extração de documentação baixada e resolução de nomes de variáveis.
    doc_dir = tree["raw_ibge"] / "pnadc_documentation"
    for archive in doc_dir.rglob("*.zip"):
        extract_archive(archive, tree["interim"] / "pnadc_documentation_extracted" / archive.stem, logger)
    doc_paths = discover_layout_files(tree["raw_ibge"]) + discover_layout_files(tree["interim"])
    variable_names = extract_variable_names_from_docs(doc_paths)
    write_json(tree["registry"] / "pnadc_detected_variables.json", {
        "schema_version": SCHEMA_VERSION,
        "detected_at_utc": utc_now(),
        "documentation_files": [str(p) for p in doc_paths],
        "variables": sorted(variable_names),
    })
    gates.extend(run_semantic_gates(variable_names, docs_found=bool(doc_paths)))

    # 4) Auditoria leve de arquivos tabulares baixados (evita reprocessar arquivos gigantes).
    for rec in records:
        if rec.status not in {"DOWNLOADED", "EXISTS", "EXTRACTED"} or not rec.local_path:
            continue
        path = Path(rec.local_path)
        if not path.exists() or path.stat().st_size > 3 * 1024**3:
            continue
        if path.suffix.lower() in {".csv", ".tsv", ".txt"}:
            try:
                audits.append(audit_generic(path, rec.source_id))
            except Exception as exc:
                logger.warning("Auditoria tabular falhou em %s: %s", path, exc)

    # 5) Gates de disponibilidade CKAN essenciais.
    required_aliases = [k for k, v in RECIFE_CKAN_PACKAGES.items() if v.get("required")]
    for alias in required_aliases:
        matched = [r for r in records if r.source_id == f"recife_{alias}" and r.status not in {"ERROR", "NOT_FOUND"}]
        gates.append(GateResult(
            gate_id=f"recife.{alias}.available",
            status="PASS" if matched else "FAIL",
            severity="critical" if alias in {"velocidade_2022", "velocidade_2023", "velocidade_2024", "equipamentos_transito"} else "high",
            source_id=f"recife_{alias}",
            message="Pacote/recurso CKAN localizado." if matched else "Pacote/recurso CKAN obrigatório não localizado.",
            evidence={"records": len(matched)},
        ))

    # 6) Manifest, auditorias, gates e relatório.
    write_json(tree["manifests"] / f"artifacts_{run_id}.json", [asdict(r) for r in records])
    manifest_to_csv(records, tree["manifests"] / f"artifacts_{run_id}.csv")
    write_json(tree["reports"] / f"audits_{run_id}.json", [asdict(a) for a in audits])
    write_json(tree["reports"] / f"gates_{run_id}.json", [asdict(g) for g in gates])
    gates_to_csv(gates, tree["reports"] / f"gates_{run_id}.csv")

    report = generate_report(root, run_id, args.mode, records, audits, gates, root_mode)
    report_path = tree["reports"] / f"PHASE0_REPORT_{run_id}.md"
    report_path.write_text(report, encoding="utf-8")
    (tree["reports"] / "PHASE0_REPORT_LATEST.md").write_text(report, encoding="utf-8")

    # Lock final: qualquer FAIL/BLOCKED critical impede avanço.
    critical_fail = [g for g in gates if g.severity == "critical" and g.status in {"FAIL", "BLOCKED"}]
    lock = {
        "run_id": run_id,
        "script_version": SCRIPT_VERSION,
        "schema_version": SCHEMA_VERSION,
        "status": "BLOCKED" if critical_fail else "RELEASED",
        "critical_failures": [asdict(g) for g in critical_fail],
        "report": str(report_path),
        "created_at_utc": utc_now(),
    }
    write_json(tree["admin"] / "PHASE0_LOCK.json", lock)

    logger.info("Fase 0 concluída. LOCK=%s | relatório=%s", lock["status"], report_path)
    if critical_fail:
        for gate in critical_fail:
            logger.error("GATE CRÍTICO: %s — %s", gate.gate_id, gate.message)
    return 2 if args.strict and critical_fail else 0


if __name__ == "__main__":
    raise SystemExit(main())